# Regime-Aware Portfolio Research — NIFTY 50

Ten large-cap Indian stocks, six years of daily prices, and a question: can a reinforcement-learning agent allocate between them better than a handful of well-understood classical methods?

The short answer is no — and most of the work here goes into making sure that answer is trustworthy rather than into avoiding it.

Everything is in this one notebook: configuration, data handling, feature construction, five regime detectors, the strategies, a realistic Indian cost model, two backtesters, the RL environment and agent, the statistics, and the reporting. Nothing is imported from elsewhere in the project.

**How to run it:** top to bottom, once. The cells are ordered so that everything is defined before it is used, which means the order is not cosmetic — if you jump around and hit a `NameError`, restart the kernel and run all.

The last cell does the actual work and takes about twenty minutes, most of it training the agent. There is a ninety-second version noted there if you just want to see it run.

---

*A note on where this file comes from: it is generated from the `src/` package by `scripts/build_monolith_notebook.py`. If you change something here, change it there too — otherwise the next regeneration will quietly overwrite you.*

In [ ]:
# Where are we? Walk up the folder tree looking for the project root, so this works
# whether you opened the notebook from notebooks/ or from the repository root.
from pathlib import Path

def _find_root(start: Path) -> Path:
    """First folder above us that looks like the project root."""
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "README.md").exists():
            return candidate
    return start

_PROJECT_ROOT = _find_root(Path.cwd())

print("project root:", _PROJECT_ROOT)
print("results, figures and the report will be written under here.")


## Configuration

Every number you might want to change lives here, and nowhere else. The dataclasses are frozen, so nothing can quietly mutate halfway through a run.

In [ ]:
# ── nifty_rl/config.py ──────────────────────────────────────────────────────────────────
# Every knob, in one place. Note that the end date is pinned — the original version
# let it float, so the published numbers quietly changed on every run.
"""Configuration objects.

Split into focused frozen dataclasses rather than one monolithic ``Config`` so that
sweeps can vary one concern (costs, RL hyperparameters, regime backend) without
carrying the rest along.

Reproducibility note: ``DataConfig.end_date`` is *pinned*. The original notebook called
``yf.download(start=...)`` with no ``end``, so the dataset — and therefore every split
boundary and every published metric — changed on each run. Pass ``end_date=None``
explicitly to opt into live data.
"""
from __future__ import annotations

from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Optional, Tuple

PROJECT_ROOT = _PROJECT_ROOT  # notebook: injected by the preamble cell
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

# Universe: 10 large-cap NIFTY 50 constituents, unchanged from the original notebook.
DEFAULT_TICKERS: Tuple[str, ...] = (
    "RELIANCE.NS",
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "INFY.NS",
    "TCS.NS",
    "BHARTIARTL.NS",
    "ITC.NS",
    "LT.NS",
    "SBIN.NS",
    "HINDUNILVR.NS",
)


@dataclass(frozen=True)
class DataConfig:
    """Data ingestion and caching."""

    tickers: Tuple[str, ...] = DEFAULT_TICKERS
    benchmark_ticker: str = "^NSEI"
    vix_ticker: str = "^INDIAVIX"

    start_date: str = "2020-01-01"
    # Pinned to the committed notebook run so published figures stay reproducible.
    # Set to None for live data (results will then drift with the calendar).
    end_date: Optional[str] = "2026-05-12"

    # auto_adjust=True keeps OHLC on a single adjusted scale. The notebook used
    # auto_adjust=False with price=Adj Close but raw High/Low, so true-range mixed two
    # price scales -- a split or bonus issue injected a spurious spike into atr_pct.
    auto_adjust: bool = True

    cache_dir: Path = field(default_factory=lambda: DATA_DIR / "raw")
    use_cache: bool = True

    # Downloaded-but-unused in the original notebook. Kept configurable but empty by
    # default; wire into a feature set before re-enabling.
    macro_assets: Tuple[str, ...] = ()
    sector_indices: Tuple[str, ...] = ()


@dataclass(frozen=True)
class SplitConfig:
    """Chronological split on unique trading dates."""

    train_ratio: float = 0.70
    valid_ratio: float = 0.15

    @property
    def test_ratio(self) -> float:
        return 1.0 - self.train_ratio - self.valid_ratio


@dataclass(frozen=True)
class CostConfig:
    """Transaction cost model.

    ``model="flat"`` reproduces the notebook (10 bps + 5 bps). ``model="india"`` applies
    the actual Indian delivery-equity charge stack; see backtest/costs.py.
    """

    model: str = "flat"  # "flat" | "india"

    # Flat model
    transaction_cost: float = 0.0010
    slippage: float = 0.0005

    # India model (delivery equity, retail)
    brokerage_rate: float = 0.0003
    brokerage_cap: float = 20.0
    stt_buy: float = 0.001
    stt_sell: float = 0.001
    stamp_duty_buy: float = 0.00015
    exchange_txn_rate: float = 0.0000297
    sebi_turnover_rate: float = 0.000001
    gst_rate: float = 0.18

    # Square-root market impact: impact_bps = coef * sqrt(order_value / ADV)
    impact_coef: float = 0.0
    impact_adv_window: int = 20


@dataclass(frozen=True)
class BacktestConfig:
    """Portfolio backtester behaviour."""

    # Rs10 lakh. At Rs1 lakh, integer share sizing strands ~2% of capital as
    # unbuyable fractions; at Rs10 lakh that drops to ~0.5%, so the published
    # figures reflect the strategies rather than the lot-size friction.
    initial_cash: float = 1_000_000.0
    stop_loss: Optional[float] = 0.06
    take_profit: Optional[float] = 0.12

    # The notebook re-entered on the very next bar after a stop-loss exit whenever the
    # underlying signal was still 1, so stops cost round-trips without providing
    # protection. Require the signal to go flat before re-arming.
    reenter_lockout: bool = True

    # The notebook generated a signal from bar t's close and filled at bar t's close.
    # "next_open" shifts the signal one bar, which is the defensible default for a
    # pipeline that advertises lookahead-free evaluation.
    execution: str = "next_bar"  # "next_bar" | "same_close"

    # Idle cash earned 0% in the notebook. At an Indian risk-free near 6.5% this
    # silently penalised every low-exposure strategy against fully-invested BuyHold.
    cash_rate_annual: float = 0.065
    trading_days: int = 252


@dataclass(frozen=True)
class MetricsConfig:
    risk_free_annual: float = 0.065
    trading_days: int = 252
    var_quantile: float = 0.05


@dataclass(frozen=True)
class RunConfig:
    """Top-level experiment configuration."""

    data: DataConfig = field(default_factory=DataConfig)
    split: SplitConfig = field(default_factory=SplitConfig)
    costs: CostConfig = field(default_factory=CostConfig)
    backtest: BacktestConfig = field(default_factory=BacktestConfig)
    metrics: MetricsConfig = field(default_factory=MetricsConfig)

    seed: int = 42
    results_dir: Path = field(default_factory=lambda: RESULTS_DIR)

    def with_(self, **kwargs) -> "RunConfig":
        """Return a copy with top-level fields replaced (frozen-dataclass friendly)."""
        return replace(self, **kwargs)


DEFAULT_RUN = RunConfig()


## Data and features

Prices come from Yahoo and get cached on disk, so the second run is fast and the hundredth run gives the same answer as the first. Then the usual indicators — moving averages, RSI, ATR, volatility, a VIX overlay.

In [ ]:
# ── nifty_rl/data.py ────────────────────────────────────────────────────────────────────
# Download and cache. Keeping every price on one adjusted scale matters: mixing
# adjusted closes with raw highs used to inject fake spikes into the volatility.
"""Data ingestion with on-disk caching.

Fixes carried over from the notebook:

* **Pinned ``end_date``** (bug #13). ``yf.download(start=...)`` with no ``end`` meant the
  dataset grew every day, so ratio-based split boundaries moved and no published metric
  could be reproduced.
* **``auto_adjust=True``** (bug #6). The notebook set ``auto_adjust=False``, took
  ``price = Adj Close``, but left ``High``/``Low`` raw. True range then differenced two
  price scales; a split or bonus issue injected a large spurious spike into ``atr_pct``,
  which fed the scaler and the RL observation.
* **Per-ticker forward fill** (bug #18). The notebook called ``data[num_cols].ffill()`` on
  a frame sorted by ``["ticker", "Date"]``, so the last row of one ticker filled into the
  first row of the next -- cross-ticker contamination in a pipeline that advertises
  per-ticker isolation.
* **Unused macro/sector downloads dropped** (bug #22). ``CL=F``, ``USDINR=X``, ``^TNX``,
  ``^CNXIT``, ``^NSEBANK`` were fetched and merged but appear in no feature set.
"""
from __future__ import annotations

import hashlib
import warnings
from pathlib import Path
from typing import Iterable, Optional, Sequence

import numpy as np
import pandas as pd


OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]


# --------------------------------------------------------------------------- cache


def _cache_key(kind: str, symbols: Sequence[str], cfg: DataConfig) -> str:
    payload = "|".join(
        [
            kind,
            ",".join(sorted(symbols)),
            str(cfg.start_date),
            str(cfg.end_date),
            str(cfg.auto_adjust),
        ]
    )
    digest = hashlib.sha1(payload.encode()).hexdigest()[:12]
    return f"{kind}__{cfg.start_date}__{cfg.end_date or 'live'}__{digest}"


def _has_parquet_engine() -> bool:
    try:
        import pyarrow  # noqa: F401

        return True
    except ImportError:
        try:
            import fastparquet  # noqa: F401

            return True
        except ImportError:
            return False


def _cache_path(key: str, cache_dir: Path) -> Path:
    """Parquet when an engine is installed, CSV otherwise.

    The cache is what makes a run reproducible, so it degrades rather than disabling
    itself when pyarrow is absent. CSV round-trips dates as strings, which is why the
    reader re-parses ``Date`` explicitly.
    """
    suffix = ".parquet" if _has_parquet_engine() else ".csv"
    return cache_dir / f"{key}{suffix}"


def _read_cache(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    try:
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    except Exception as exc:  # pragma: no cover - corrupt cache
        warnings.warn(f"Cache read failed for {path.name}: {exc}")
        return None
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"])
    return df


def _write_cache(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        if path.suffix == ".parquet":
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
    except Exception as exc:  # pragma: no cover
        warnings.warn(f"Cache write failed ({exc}); this run will not be reproducible.")


# ------------------------------------------------------------------- yfinance I/O


def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = df.columns.get_level_values(0)
    return df


def _download_one(symbol: str, cfg: DataConfig) -> pd.DataFrame:
    import yfinance as yf  # imported lazily so tests can run without network deps

    raw = yf.download(
        symbol,
        start=cfg.start_date,
        end=cfg.end_date,
        progress=False,
        auto_adjust=cfg.auto_adjust,
    )
    if raw is None or raw.empty:
        return pd.DataFrame()
    raw = _flatten_columns(raw).reset_index()
    raw["Date"] = pd.to_datetime(raw["Date"])
    return raw


def download_ohlcv(cfg: DataConfig, tickers: Optional[Iterable[str]] = None) -> pd.DataFrame:
    """Download adjusted OHLCV for ``tickers`` in long format.

    Returns columns ``[Date, Open, High, Low, Close, Volume, price, ticker]`` sorted by
    ``(ticker, Date)``. With ``auto_adjust=True`` every OHLC field is on the same
    adjusted scale, so ``price`` is simply ``Close``.
    """
    symbols = list(tickers) if tickers is not None else list(cfg.tickers)

    if cfg.use_cache:
        key = _cache_key("ohlcv", symbols, cfg)
        cached = _read_cache(_cache_path(key, cfg.cache_dir))
        if cached is not None:
            return cached

    frames = []
    for symbol in symbols:
        df = _download_one(symbol, cfg)
        if df.empty:
            warnings.warn(f"No data returned for {symbol}; excluded from the panel.")
            continue
        keep = ["Date"] + [c for c in OHLCV_COLUMNS if c in df.columns]
        df = df[keep].copy()
        # auto_adjust=True already folds dividends/splits into Close.
        df["price"] = df["Close"] if "Close" in df.columns else np.nan
        df["ticker"] = symbol
        frames.append(df)

    if not frames:
        raise RuntimeError("No tickers returned data; cannot build a panel.")

    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["ticker", "Date"]).reset_index(drop=True)

    if cfg.use_cache:
        _write_cache(out, _cache_path(_cache_key("ohlcv", symbols, cfg), cfg.cache_dir))
    return out


def download_series(symbol: str, column_name: str, cfg: DataConfig) -> pd.DataFrame:
    """Download a single auxiliary series as ``[Date, column_name]``."""
    if cfg.use_cache:
        key = _cache_key(f"series_{column_name}", [symbol], cfg)
        cached = _read_cache(_cache_path(key, cfg.cache_dir))
        if cached is not None:
            return cached

    df = _download_one(symbol, cfg)
    if df.empty:
        warnings.warn(f"No data for auxiliary series {symbol}.")
        return pd.DataFrame(columns=["Date", column_name])

    price_col = "Close" if "Close" in df.columns else df.columns[1]
    out = df[["Date", price_col]].rename(columns={price_col: column_name})

    if cfg.use_cache:
        _write_cache(out, _cache_path(_cache_key(f"series_{column_name}", [symbol], cfg), cfg.cache_dir))
    return out


# ------------------------------------------------------------------ panel assembly


def ffill_by_ticker(df: pd.DataFrame, columns: Optional[Sequence[str]] = None) -> pd.DataFrame:
    """Forward fill numeric columns **within each ticker**.

    The notebook's global ``ffill`` on a ticker-sorted frame leaked the last row of one
    ticker into the first row of the next (bug #18).
    """
    out = df.copy()
    cols = list(columns) if columns is not None else list(
        out.select_dtypes(include="number").columns
    )
    cols = [c for c in cols if c in out.columns]
    if not cols:
        return out
    out[cols] = out.groupby("ticker", sort=False)[cols].ffill()
    return out


def build_panel(cfg: DataConfig) -> pd.DataFrame:
    """Assemble the long-format panel: OHLCV + India VIX + benchmark return.

    Cross-sectional dispersion is computed here because it is a genuine cross-ticker
    aggregate (std of 5-day returns across the universe on each date) and therefore
    cannot be produced inside a per-ticker groupby.
    """
    prices = download_ohlcv(cfg)

    vix = download_series(cfg.vix_ticker, "india_vix", cfg)
    panel = prices.merge(vix, on="Date", how="left")

    bench = download_series(cfg.benchmark_ticker, "benchmark_close", cfg)
    if not bench.empty:
        bench = bench.sort_values("Date").reset_index(drop=True)
        bench["benchmark_return"] = bench["benchmark_close"].pct_change()
        panel = panel.merge(bench[["Date", "benchmark_close", "benchmark_return"]], on="Date", how="left")
    else:
        panel["benchmark_close"] = np.nan
        panel["benchmark_return"] = np.nan

    for symbol in cfg.macro_assets:
        col = symbol.replace("=", "_").replace("^", "")
        panel = panel.merge(download_series(symbol, col, cfg), on="Date", how="left")

    for symbol in cfg.sector_indices:
        col = symbol.replace("^", "") + "_close"
        panel = panel.merge(download_series(symbol, col, cfg), on="Date", how="left")

    panel = panel.sort_values(["ticker", "Date"]).reset_index(drop=True)
    panel = panel.merge(cross_sectional_dispersion(panel), on="Date", how="left")

    # Per-ticker fill for auxiliary series that use a slightly different calendar
    # (India VIX and ^NSEI occasionally differ from the equity calendar).
    panel = ffill_by_ticker(
        panel,
        [c for c in panel.columns if c not in ("Date", "ticker")],
    )
    return panel


def cross_sectional_dispersion(panel: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    """Std of ``window``-day returns across the universe, per date."""
    tmp = panel[["Date", "ticker", "price"]].copy()
    tmp["ret_w"] = tmp.groupby("ticker", sort=False)["price"].pct_change(window)
    disp = tmp.groupby("Date")["ret_w"].std().rename("cs_dispersion").reset_index()
    return disp


def chronological_split(df: pd.DataFrame, train_ratio: float, valid_ratio: float):
    """Split on **unique trading dates** so all tickers land in the same bands."""
    dates = np.sort(df["Date"].unique())
    n = len(dates)
    train_end = int(n * train_ratio)
    valid_end = int(n * (train_ratio + valid_ratio))

    train_dates = set(dates[:train_end])
    valid_dates = set(dates[train_end:valid_end])
    test_dates = set(dates[valid_end:])

    return (
        df[df["Date"].isin(train_dates)].reset_index(drop=True),
        df[df["Date"].isin(valid_dates)].reset_index(drop=True),
        df[df["Date"].isin(test_dates)].reset_index(drop=True),
    )


In [ ]:
# ── nifty_rl/features.py ────────────────────────────────────────────────────────────────
# The indicators. RSI and ATR use Wilder's smoothing rather than a plain moving
# average — they genuinely differ, and it propagates into every signal.
"""Per-ticker technical features.

Every indicator is computed inside a per-ticker groupby, so no rolling window ever spans
a ticker boundary.

Fixes carried over from the notebook:

* **RSI no longer deletes rows** (bug #19). The notebook used
  ``loss.rolling(14).mean().replace(0, np.nan)``, so 14 consecutive up-days produced a
  NaN RSI, and the blanket ``dropna()`` at the end then dropped that row entirely --
  silently removing observations precisely during the strongest momentum runs. Wilder's
  smoothing is used here, and the zero-loss case resolves to RSI 100 (its correct value)
  rather than NaN.
* **Targeted dropna** (bug #20). The notebook's bare ``dropna()`` gated every row on
  *all* 49 columns, including ``ma100`` and ``momentum_60`` which only the ``full``
  feature set uses. That cost ~100 days of warm-up on every run regardless of the
  feature set in play -- which is why a 2020-01-01 start became 2020-05-29.
"""
from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import pandas as pd

# Columns required by the smallest feature set. Rows are dropped only on these unless
# the caller asks for more.
CORE_FEATURES: Sequence[str] = (
    "ret",
    "ma_ratio",
    "trend_20_50",
    "rsi",
    "macd_hist",
    "bb_position",
    "bb_width",
    "atr_pct",
    "momentum_5",
    "momentum_20",
    "volume_change",
)


def wilder_rsi(price: pd.Series, window: int = 14) -> pd.Series:
    """RSI with Wilder's smoothing.

    Boundary cases resolved explicitly instead of propagating NaN:
      * no losses over the window -> 100 (maximum strength)
      * no gains and no losses    -> 50 (neutral; a flat series has no momentum)
    """
    delta = price.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)

    alpha = 1.0 / window
    avg_gain = gain.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=alpha, adjust=False, min_periods=window).mean()

    rsi = pd.Series(np.nan, index=price.index, dtype="float64")
    valid = avg_gain.notna() & avg_loss.notna()

    both_zero = valid & (avg_gain <= 0) & (avg_loss <= 0)
    no_loss = valid & (avg_loss <= 0) & (avg_gain > 0)
    normal = valid & (avg_loss > 0)

    rs = avg_gain[normal] / avg_loss[normal]
    rsi.loc[normal] = 100.0 - (100.0 / (1.0 + rs))
    rsi.loc[no_loss] = 100.0
    rsi.loc[both_zero] = 50.0
    return rsi


def wilder_atr(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> pd.Series:
    """Average true range with Wilder's smoothing.

    Requires that ``high``/``low``/``close`` share one price scale -- guaranteed by
    ``auto_adjust=True`` in the data layer (bug #6).
    """
    prev_close = close.shift(1)
    true_range = pd.concat(
        [(high - low), (high - prev_close).abs(), (low - prev_close).abs()],
        axis=1,
    ).max(axis=1)
    return true_range.ewm(alpha=1.0 / window, adjust=False, min_periods=window).mean()


def add_features_single(df: pd.DataFrame) -> pd.DataFrame:
    """Compute indicators for one ticker. Input must be a single-ticker frame."""
    out = df.copy().reset_index(drop=True)
    price = out["price"]
    high = out["High"] if "High" in out.columns else price
    low = out["Low"] if "Low" in out.columns else price
    volume = out["Volume"] if "Volume" in out.columns else pd.Series(0.0, index=out.index)

    # --- returns and momentum
    out["ret"] = price.pct_change()
    out["momentum_5"] = price.pct_change(5)
    out["momentum_20"] = price.pct_change(20)
    out["momentum_60"] = price.pct_change(60)

    vol_ma20 = volume.rolling(20).mean()
    out["volume_change"] = (volume / vol_ma20.replace(0, np.nan)) - 1.0
    out["volume_change"] = out["volume_change"].replace([np.inf, -np.inf], np.nan)

    # --- moving averages
    out["ma5"] = price.rolling(5).mean()
    out["ma10"] = price.rolling(10).mean()
    out["ma20"] = price.rolling(20).mean()
    out["ma50"] = price.rolling(50).mean()
    out["ma100"] = price.rolling(100).mean()
    # NOTE: ma_ratio is ma5/ma20 - 1, not "Price / MA20" as the README claims.
    out["ma_ratio"] = out["ma5"] / out["ma20"] - 1.0
    out["trend_20_50"] = out["ma20"] / out["ma50"] - 1.0

    # --- volatility / channels
    out["vol20"] = out["ret"].rolling(20).std() * np.sqrt(252)
    out["high20"] = price.rolling(20).max()
    out["low20"] = price.rolling(20).min()
    out["high20_prev"] = out["high20"].shift(1)
    out["low10_prev"] = price.rolling(10).min().shift(1)

    # --- oscillators
    out["rsi"] = wilder_rsi(price, 14)

    ema12 = price.ewm(span=12, adjust=False).mean()
    ema26 = price.ewm(span=26, adjust=False).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]

    bb_mid = price.rolling(20).mean()
    bb_std = price.rolling(20).std()
    band = (4.0 * bb_std).replace(0, np.nan)
    out["bb_width"] = band / bb_mid
    out["bb_position"] = (price - (bb_mid - 2.0 * bb_std)) / band

    out["atr14"] = wilder_atr(high, low, price, 14)
    out["atr_pct"] = out["atr14"] / price

    # --- VIX-derived market-wide features
    # sentiment is -zscore(india_vix, 60d): high VIX = fear = negative. Market-wide by
    # construction, so it is identical across all tickers -- see report notes before
    # plotting it as a per-ticker heatmap (bug #27).
    if "india_vix" in out.columns:
        vix_mu = out["india_vix"].rolling(60).mean()
        vix_sd = out["india_vix"].rolling(60).std().replace(0, np.nan)
        out["sentiment"] = -((out["india_vix"] - vix_mu) / vix_sd)
        out["sent_ma3"] = out["sentiment"].rolling(3).mean()
        out["sent_ma7"] = out["sentiment"].rolling(7).mean()
        out["sent_momentum"] = out["sent_ma3"] - out["sent_ma7"]
        out["vix_change"] = out["india_vix"].pct_change()
        # NOTE: 75th percentile, not the 60th the README documents.
        out["high_vix_regime"] = (
            out["india_vix"] > out["india_vix"].rolling(60).quantile(0.75)
        ).astype(float)

    if "cs_dispersion" in out.columns:
        cs_mu = out["cs_dispersion"].rolling(20).mean()
        cs_sd = out["cs_dispersion"].rolling(20).std().replace(0, np.nan)
        out["dispersion_zscore"] = (out["cs_dispersion"] - cs_mu) / cs_sd
    else:
        out["dispersion_zscore"] = np.nan

    if "benchmark_return" in out.columns:
        out["relative_strength"] = out["ret"] - out["benchmark_return"]
        out["rolling_alpha"] = out["relative_strength"].rolling(20).mean()

    return out


def add_features(
    panel: pd.DataFrame,
    required: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """Compute features per ticker and drop warm-up rows.

    ``required`` controls which columns gate row removal. Defaults to ``CORE_FEATURES``
    so unused long-window columns (``ma100``, ``momentum_60``) do not cost warm-up.
    """
    parts = [add_features_single(g) for _, g in panel.groupby("ticker", sort=False)]
    out = pd.concat(parts, ignore_index=True)

    subset = list(required) if required is not None else list(CORE_FEATURES)
    subset = [c for c in subset if c in out.columns]
    if subset:
        out = out.dropna(subset=subset)

    return out.sort_values(["ticker", "Date"]).reset_index(drop=True)


def align_to_common_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Restrict the panel to dates present for **every** ticker.

    Per-ticker warm-up and holiday differences otherwise leave a ragged panel, which the
    aggregation step in the backtester turns into phantom equity holes (bug #5). Keeping
    the panel rectangular removes that failure mode at the source; the backtester also
    defends against it independently.
    """
    counts = df.groupby("Date")["ticker"].nunique()
    full = counts[counts == df["ticker"].nunique()].index
    return df[df["Date"].isin(full)].reset_index(drop=True)


## What a trade actually costs

This is the part most backtests wave away. Indian delivery equity carries STT, stamp duty, exchange and SEBI fees, and GST on top of the brokerage — and the whole stack gets folded into the fill price, so no strategy can forget to pay it. Two backtesters follow: one for hold/don't-hold signals, one for portfolio weights. They charge identically, which matters, because otherwise any gap between the agent and the baselines would partly be an artefact of how they execute rather than what they decide.

In [ ]:
# ── nifty_rl/backtest/costs.py ──────────────────────────────────────────────────────────
# The full Indian charge stack, folded into the price you actually pay.
"""Transaction cost models.

The notebook applied a flat 10 bps + 5 bps to every fill. That is retained as
``FlatCostModel`` so old results stay reproducible, but it understates Indian delivery
equity costs, where STT alone is 10 bps on each side.

Both models expose the same interface: given a side, price and quantity, return the
*effective* per-share fill price. Callers never apply costs themselves.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Optional

import numpy as np


BUY = "buy"
SELL = "sell"


class CostModel(ABC):
    """Maps a desired trade to an effective fill price."""

    @abstractmethod
    def cost_rate(self, side: str, price: float, quantity: float, adv_value: Optional[float] = None) -> float:
        """Total round-of-one-side cost as a fraction of notional."""

    def fill_price(self, side: str, price: float, quantity: float, adv_value: Optional[float] = None) -> float:
        """The price actually paid or received per share, costs included.

        Costs are folded into the price rather than deducted separately: a buy fills
        *above* the quoted price and a sell *below* it. Callers therefore cannot forget
        to apply them, and the same helper serves both backtesters and the RL
        environment, so every strategy is charged on identical terms.
        """
        rate = self.cost_rate(side, price, quantity, adv_value)
        if side == BUY:
            return price * (1.0 + rate)
        if side == SELL:
            return price * (1.0 - rate)
        raise ValueError(f"side must be {BUY!r} or {SELL!r}, got {side!r}")


class FlatCostModel(CostModel):
    """Notebook-compatible flat cost: ``transaction_cost + slippage`` on each side."""

    def __init__(self, cfg: CostConfig):
        self.cfg = cfg
        self._base = cfg.transaction_cost + cfg.slippage

    def cost_rate(self, side: str, price: float, quantity: float, adv_value: Optional[float] = None) -> float:
        return self._base + _impact_rate(self.cfg, price, quantity, adv_value)


class IndiaEquityCostModel(CostModel):
    """Delivery-equity charge stack for a retail Indian investor.

    Components (each as a fraction of turnover unless noted):

    ==================  ==========  ==========  ==================================
    Charge              Buy         Sell        Notes
    ==================  ==========  ==========  ==================================
    Brokerage           0.03%       0.03%       capped at Rs20 per order
    STT                 0.10%       0.10%       delivery equity, both sides
    Stamp duty          0.015%      --          buy side only
    Exchange txn        0.00297%    0.00297%    NSE equity
    SEBI turnover       0.0001%     0.0001%
    GST                 18%         18%         on brokerage + exchange + SEBI
    ==================  ==========  ==========  ==================================

    Slippage and square-root market impact are layered on top.
    """

    def __init__(self, cfg: CostConfig):
        self.cfg = cfg

    def cost_rate(self, side: str, price: float, quantity: float, adv_value: Optional[float] = None) -> float:
        """Total one-side cost as a fraction of the order's rupee value.

        Everything is converted to a *rate* so it can be folded into the fill price.
        Note that brokerage is the one charge that is not a pure rate -- it is capped at
        Rs20 per order, so dividing by notional makes its effective rate fall as the
        order grows. That is why a Rs10 lakh portfolio is charged proportionally less
        than a Rs1 lakh one, and part of why the published figures use the larger amount.
        """
        cfg = self.cfg
        notional = max(price * quantity, 1e-12)

        brokerage_value = min(cfg.brokerage_rate * notional, cfg.brokerage_cap)
        brokerage = brokerage_value / notional

        exchange = cfg.exchange_txn_rate
        sebi = cfg.sebi_turnover_rate
        # GST applies to the intermediary's fees, not to the statutory taxes below.
        gst = cfg.gst_rate * (brokerage + exchange + sebi)

        # STT is charged on both sides; stamp duty only on purchases.
        if side == BUY:
            statutory = cfg.stt_buy + cfg.stamp_duty_buy
        elif side == SELL:
            statutory = cfg.stt_sell
        else:
            raise ValueError(f"side must be {BUY!r} or {SELL!r}, got {side!r}")

        return (
            brokerage
            + exchange
            + sebi
            + gst
            + statutory
            + cfg.slippage
            + _impact_rate(cfg, price, quantity, adv_value)
        )


def _impact_rate(cfg: CostConfig, price: float, quantity: float, adv_value: Optional[float]) -> float:
    """Square-root market impact: ``coef * sqrt(order_value / ADV)``.

    Disabled by default (``impact_coef=0``). This is the term that should discipline the
    RL agent's turnover once enabled -- a flat per-trade cost does not penalise size.
    """
    if cfg.impact_coef <= 0 or not adv_value or adv_value <= 0:
        return 0.0
    participation = (price * abs(quantity)) / adv_value
    return float(cfg.impact_coef * np.sqrt(max(participation, 0.0)))


def build_cost_model(cfg: CostConfig) -> CostModel:
    if cfg.model == "flat":
        return FlatCostModel(cfg)
    if cfg.model == "india":
        return IndiaEquityCostModel(cfg)
    raise ValueError(f"Unknown cost model {cfg.model!r}; expected 'flat' or 'india'.")


In [ ]:
# ── nifty_rl/backtest/engine.py ─────────────────────────────────────────────────────────
# Backtester for hold/don't-hold signals; each stock gets its own cash. A stop-loss
# now blocks re-entry until the signal goes flat — without that, stops paid round-trip
# costs and protected nothing.
"""Equal-weight multi-stock portfolio backtester.

Each ticker gets its own capital slice, cash bucket, position and trade log; the
portfolio equity curve is the sum of the per-ticker curves.

Fixes carried over from the notebook:

* **Stop-loss no longer re-enters on the next bar** (bug #4). The notebook set
  ``cur_sig = 0`` on a risk exit, which made ``prev_sig = 0``; if the underlying signal
  was still 1 the edge-triggered entry condition fired again the following bar. Stops
  therefore generated round-trip costs without providing protection -- which is why the
  4x4 SL/TP sensitivity grid came out nearly flat. Entry is now level-triggered with an
  explicit lockout that clears only when the signal returns to 0.
* **No phantom equity holes** (bug #5). The notebook aggregated with
  ``agg_equity.add(eq, fill_value=0)``. When one ticker lacked a date another had, that
  ticker contributed *zero* instead of its carried equity -- an artificial ~-10%
  portfolio drop. Series are now reindexed to the union of dates and forward filled
  before summing.
* **Per-ticker position series** (bug #8). The notebook returned only the aggregate
  0..N position count, which downstream metrics then compared against each individual
  ticker's forward return. Both are kept, and metrics use the per-ticker mapping.
* **Configurable execution lag** (bug #14). The notebook generated a signal from bar t's
  close and filled at bar t's close. ``execution="next_bar"`` shifts the signal one bar,
  which is the defensible default for a pipeline advertising lookahead-free evaluation.
* **Idle cash earns the risk-free rate** (bug #10). Cash returned 0% in the notebook. At
  an Indian risk-free near 6.5% that silently penalised every low-exposure strategy
  relative to fully-invested buy-and-hold.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


SignalFn = Callable[[pd.DataFrame], pd.Series]


@dataclass
class BacktestResult:
    """Outcome of a portfolio backtest."""

    strategy: str
    equity: pd.Series
    positions: pd.Series  # aggregate count of tickers held, 0..N
    trades: pd.DataFrame
    per_ticker_equity: Dict[str, pd.Series] = field(default_factory=dict)
    per_ticker_positions: Dict[str, pd.Series] = field(default_factory=dict)
    weights: Optional[pd.DataFrame] = None

    @property
    def returns(self) -> pd.Series:
        return self.equity.pct_change().dropna()


def _daily_cash_rate(cfg: BacktestConfig) -> float:
    if cfg.cash_rate_annual <= 0:
        return 0.0
    return (1.0 + cfg.cash_rate_annual) ** (1.0 / cfg.trading_days) - 1.0


def _apply_execution_lag(signal: pd.Series, cfg: BacktestConfig) -> pd.Series:
    if cfg.execution == "next_bar":
        return signal.shift(1).fillna(0).astype(int)
    if cfg.execution == "same_close":
        return signal.fillna(0).astype(int)
    raise ValueError(
        f"Unknown execution mode {cfg.execution!r}; expected 'next_bar' or 'same_close'."
    )


def run_single_ticker_backtest(
    df: pd.DataFrame,
    signal: pd.Series,
    capital: float,
    cfg: BacktestConfig,
    cost_model: CostModel,
    adv: Optional[pd.Series] = None,
):
    """Backtest one ticker against its own cash bucket.

    Returns ``(equity, positions, trades)`` indexed by date.
    """
    frame = df.reset_index(drop=True)
    signal = _apply_execution_lag(
        signal.reset_index(drop=True).reindex(frame.index).fillna(0).astype(int), cfg
    )

    dates = pd.to_datetime(frame["Date"]).to_numpy()
    prices = frame["price"].astype(float).to_numpy()
    ticker = frame["ticker"].iloc[0] if "ticker" in frame.columns else "UNKNOWN"
    adv_values = adv.reindex(frame.index).to_numpy() if adv is not None else None

    daily_cash_rate = _daily_cash_rate(cfg)

    cash = float(capital)
    qty = 0
    entry_price: Optional[float] = None
    entry_date = None
    locked = False  # set by a risk exit; cleared when the signal returns to 0

    equity: List[float] = []
    positions: List[int] = []
    trades: List[dict] = []

    for i in range(len(frame)):
        price = prices[i]
        date = dates[i]
        sig = int(signal.iloc[i])
        adv_i = float(adv_values[i]) if adv_values is not None and np.isfinite(adv_values[i]) else None

        if i > 0 and daily_cash_rate:
            cash *= 1.0 + daily_cash_rate

        # A flat signal always clears the post-stop lockout.
        if sig == 0:
            locked = False

        if qty > 0 and entry_price is not None:
            trade_return = price / entry_price - 1.0
            hit_stop = cfg.stop_loss is not None and trade_return <= -cfg.stop_loss
            hit_target = cfg.take_profit is not None and trade_return >= cfg.take_profit
            risk_exit = hit_stop or hit_target

            if risk_exit or sig == 0:
                fill = cost_model.fill_price(SELL, price, qty, adv_i)
                cash += qty * fill
                trades.append(
                    {
                        "ticker": ticker,
                        "entry_date": entry_date,
                        "exit_date": date,
                        "entry_price": entry_price,
                        "exit_price": fill,
                        "shares": qty,
                        "pnl": (fill - entry_price) * qty,
                        "return": fill / entry_price - 1.0,
                        "exit_reason": "stop" if hit_stop else ("target" if hit_target else "signal"),
                    }
                )
                qty = 0
                entry_price = None
                entry_date = None
                # Only a risk exit arms the lockout. A signal exit already implies
                # sig == 0, so the next entry needs a fresh 0 -> 1 transition anyway.
                locked = bool(risk_exit and cfg.reenter_lockout)

        if qty == 0 and sig == 1 and not locked and price > 0:
            fill = cost_model.fill_price(BUY, price, 1.0, adv_i)
            shares = int(cash // fill)
            if shares > 0:
                # Re-price with the actual size so market impact scales with the order.
                fill = cost_model.fill_price(BUY, price, shares, adv_i)
                shares = int(cash // fill)
            if shares > 0:
                cash -= shares * fill
                qty = shares
                entry_price = fill
                entry_date = date

        equity.append(cash + qty * price)
        positions.append(1 if qty > 0 else 0)

    # Liquidate any open position at the final close.
    if qty > 0 and entry_price is not None:
        price = prices[-1]
        fill = cost_model.fill_price(SELL, price, qty, None)
        cash += qty * fill
        trades.append(
            {
                "ticker": ticker,
                "entry_date": entry_date,
                "exit_date": dates[-1],
                "entry_price": entry_price,
                "exit_price": fill,
                "shares": qty,
                "pnl": (fill - entry_price) * qty,
                "return": fill / entry_price - 1.0,
                "exit_reason": "final_liquidation",
            }
        )
        equity[-1] = cash
        positions[-1] = 0

    index = pd.DatetimeIndex(dates, name="Date")
    return (
        pd.Series(equity, index=index, name="equity"),
        pd.Series(positions, index=index, name="position"),
        trades,
    )


def _align_and_sum(series_map: Dict[str, pd.Series], fill_leading: Dict[str, float]) -> pd.Series:
    """Sum per-ticker series on the union of dates without inventing zeros.

    ``Series.add(..., fill_value=0)`` treats a missing date as *zero equity* rather than
    *carried equity*, which manufactures portfolio-wide drawdowns out of ragged
    calendars. Reindex to the union, forward fill, backfill the warm-up with the
    ticker's starting capital, then sum.
    """
    if not series_map:
        return pd.Series(dtype="float64")

    union = pd.DatetimeIndex(sorted(set().union(*[s.index for s in series_map.values()])))
    aligned = []
    for name, series in series_map.items():
        s = series.reindex(union).ffill()
        s = s.fillna(fill_leading.get(name, 0.0))
        aligned.append(s)
    total = aligned[0].copy()
    for s in aligned[1:]:
        total = total + s
    total.index.name = "Date"
    return total


def _align_and_sum_positions(series_map: Dict[str, pd.Series]) -> pd.Series:
    if not series_map:
        return pd.Series(dtype="int64")
    union = pd.DatetimeIndex(sorted(set().union(*[s.index for s in series_map.values()])))
    total = None
    for series in series_map.values():
        s = series.reindex(union).fillna(0)
        total = s if total is None else total + s
    total.index.name = "Date"
    return total.astype(int)


def run_portfolio_backtest(
    df: pd.DataFrame,
    signal_fn: SignalFn,
    strategy_name: str,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    tickers: Optional[Sequence[str]] = None,
    cost_model: Optional[CostModel] = None,
) -> BacktestResult:
    """Run an equal-weight multi-stock backtest.

    ``signal_fn`` receives a single-ticker frame and returns a 0/1 Series. Signals are
    generated per ticker so no position state bleeds across ticker boundaries.
    """
    universe = list(tickers) if tickers is not None else list(pd.unique(df["ticker"]))
    if not universe:
        raise ValueError("No tickers to backtest.")

    model = cost_model if cost_model is not None else build_cost_model(cost_cfg)
    per_ticker_capital = cfg.initial_cash / len(universe)

    equity_map: Dict[str, pd.Series] = {}
    position_map: Dict[str, pd.Series] = {}
    all_trades: List[dict] = []

    for ticker in universe:
        ticker_df = df[df["ticker"] == ticker].sort_values("Date").reset_index(drop=True)
        if ticker_df.empty:
            continue

        signal = signal_fn(ticker_df)

        adv = None
        if cost_cfg.impact_coef > 0 and "Volume" in ticker_df.columns:
            adv = (
                (ticker_df["Volume"] * ticker_df["price"])
                .rolling(cost_cfg.impact_adv_window)
                .mean()
            )

        equity, positions, trades = run_single_ticker_backtest(
            ticker_df, signal, per_ticker_capital, cfg, model, adv
        )
        equity_map[ticker] = equity
        position_map[ticker] = positions
        all_trades.extend(trades)

    total_equity = _align_and_sum(
        equity_map, {t: per_ticker_capital for t in equity_map}
    )
    total_positions = _align_and_sum_positions(position_map)

    return BacktestResult(
        strategy=strategy_name,
        equity=total_equity,
        positions=total_positions,
        trades=pd.DataFrame(all_trades),
        per_ticker_equity=equity_map,
        per_ticker_positions=position_map,
    )


In [ ]:
# ── nifty_rl/backtest/weights.py ────────────────────────────────────────────────────────
# Backtester for weight schedules. Rebalances on schedule and drifts in between, like
# a real monthly-rebalanced fund would.
"""Weight-based portfolio backtester for continuous allocators.

The signal engine models ten independent per-ticker cash buckets, which suits binary
timing rules. Allocators emit a *joint* weight vector, so they need a single pooled cash
account -- and that difference is itself worth stating, because it means a timing rule
and an allocator are not automatically comparable. The original notebook compared PPO
(pooled, able to concentrate) against rule-based strategies (ten isolated buckets, unable
to reallocate) and called it apples-to-apples.

Rebalancing executes **all sells before any buys**. Doing it in one interleaved pass --
as the notebook's RL environment did -- means a sale late in the ticker list cannot fund
a purchase early in it, so target weights are frequently unreachable and whichever names
sit first in the universe tuple get systematic funding priority.
"""
from __future__ import annotations

from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd


AllocatorFn = Callable[[pd.DataFrame], pd.Series]


def price_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    """Dates x tickers price matrix, forward filled within each column."""
    wide = panel.pivot_table(index="Date", columns="ticker", values="price", aggfunc="last")
    return wide.sort_index().ffill()


def rebalance_dates(index: pd.DatetimeIndex, frequency: str = "ME") -> List[pd.Timestamp]:
    """Last trading day of each period present in the index."""
    if len(index) == 0:
        return []
    marks = pd.Series(index, index=index).resample(frequency).last().dropna()
    return [d for d in marks.tolist() if d in set(index)]


def build_allocator_weights(
    prices: pd.DataFrame,
    allocator: AllocatorFn,
    lookback: int = 252,
    frequency: str = "ME",
    min_history: int = 60,
) -> pd.DataFrame:
    """Weights at each rebalance date, estimated from trailing returns only.

    The estimation window ends **at** the rebalance date, never after it. Fitting a
    covariance matrix on the full sample and then "rebalancing" through history is the
    classic way an allocator backtest becomes fiction.
    """
    returns = prices.pct_change()
    marks = rebalance_dates(prices.index, frequency)

    rows: Dict[pd.Timestamp, pd.Series] = {}
    for date in marks:
        window = returns.loc[:date].tail(lookback).dropna(how="all")
        if len(window) < min_history:
            continue
        usable = window.dropna(axis=1, how="any")
        if usable.shape[1] < 2:
            continue
        weights = allocator(usable)
        rows[date] = weights.reindex(prices.columns).fillna(0.0)

    if not rows:
        return pd.DataFrame(columns=prices.columns)
    return pd.DataFrame(rows).T.sort_index()


def run_weight_backtest(
    prices: pd.DataFrame,
    weights: pd.DataFrame,
    strategy_name: str,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    cost_model: Optional[CostModel] = None,
) -> BacktestResult:
    """Backtest a schedule of target weights against a pooled cash account.

    ``weights`` carries a row only for rebalance dates (month ends by default), not for
    every trading day. On those dates the portfolio is traded back to target; on every
    other day it simply **drifts** with prices, which is what a real monthly-rebalanced
    fund does. Charging a trade every day would make turnover costs dominate everything.

    Rows of ``weights`` need not sum to one -- a shortfall is held as cash and earns the
    configured rate, which is how a regime exposure overlay expresses de-risking without
    silently renormalising back to fully invested.

    Rebalancing is two-pass, sells before buys, for the same reason as
    :meth:`envs.core.PortfolioSimulator.step`: proceeds from a sale must be available to
    fund a purchase, otherwise the achievable weights depend on column order. Both
    backtesters share this so PPO and the allocators are charged identically -- if they
    executed differently, any performance gap between them would be partly an artefact
    of the execution model rather than the strategy.
    """
    model = cost_model if cost_model is not None else build_cost_model(cost_cfg)
    tickers = list(prices.columns)
    n = len(tickers)

    daily_cash_rate = (
        (1.0 + cfg.cash_rate_annual) ** (1.0 / cfg.trading_days) - 1.0
        if cfg.cash_rate_annual > 0
        else 0.0
    )

    cash = float(cfg.initial_cash)
    shares = np.zeros(n)
    schedule = {d: weights.loc[d].to_numpy(dtype=float) for d in weights.index}

    equity_path: List[float] = []
    weight_path: List[np.ndarray] = []
    trades: List[dict] = []

    for i, date in enumerate(prices.index):
        px = prices.loc[date].to_numpy(dtype=float)
        px = np.where(np.isfinite(px) & (px > 0), px, np.nan)

        if i > 0 and daily_cash_rate:
            cash *= 1.0 + daily_cash_rate

        holdings_value = np.nansum(shares * px)
        portfolio_value = cash + holdings_value

        if date in schedule and portfolio_value > 0:
            target = np.nan_to_num(schedule[date], nan=0.0)
            target_value = target * portfolio_value
            current_value = np.nan_to_num(shares * px, nan=0.0)
            delta = target_value - current_value

            # --- pass 1: sells, which free cash for pass 2
            for k in range(n):
                if not np.isfinite(px[k]) or delta[k] >= 0:
                    continue
                quantity = min(int(abs(delta[k]) // px[k]), int(shares[k]))
                if quantity <= 0:
                    continue
                fill = model.fill_price(SELL, px[k], quantity)
                cash += quantity * fill
                shares[k] -= quantity
                trades.append(
                    {"date": date, "ticker": tickers[k], "side": "sell",
                     "shares": quantity, "price": fill, "value": quantity * fill}
                )

            # --- pass 2: buys, pro-rated if the plan exceeds available cash
            wanted = np.array(
                [max(delta[k], 0.0) if np.isfinite(px[k]) else 0.0 for k in range(n)]
            )
            total_wanted = wanted.sum()
            budget = max(cash, 0.0)
            scale = min(1.0, budget / total_wanted) if total_wanted > budget > 0 else 1.0

            for k in range(n):
                if wanted[k] <= 0 or not np.isfinite(px[k]):
                    continue
                fill = model.fill_price(BUY, px[k], 1.0)
                quantity = int((wanted[k] * scale) // fill)
                if quantity <= 0:
                    continue
                cost = quantity * model.fill_price(BUY, px[k], quantity)
                if cost > cash:
                    quantity = int(cash // fill)
                    if quantity <= 0:
                        continue
                    cost = quantity * model.fill_price(BUY, px[k], quantity)
                cash = max(cash - cost, 0.0)
                shares[k] += quantity
                trades.append(
                    {"date": date, "ticker": tickers[k], "side": "buy",
                     "shares": quantity, "price": cost / quantity, "value": cost}
                )

        holdings_value = np.nansum(shares * px)
        total = cash + holdings_value
        equity_path.append(total)
        weight_path.append(
            np.nan_to_num(shares * px, nan=0.0) / total if total > 0 else np.zeros(n)
        )

    equity = pd.Series(equity_path, index=prices.index, name="equity")
    realised_weights = pd.DataFrame(weight_path, index=prices.index, columns=tickers)
    positions = pd.Series((realised_weights > 1e-6).sum(axis=1), index=prices.index, name="position")

    per_ticker_positions = {
        ticker: (realised_weights[ticker] > 1e-6).astype(int) for ticker in tickers
    }

    return BacktestResult(
        strategy=strategy_name,
        equity=equity,
        positions=positions,
        trades=pd.DataFrame(trades),
        per_ticker_positions=per_ticker_positions,
        weights=realised_weights,
    )


def turnover(weights: pd.DataFrame) -> pd.Series:
    """One-sided turnover per period -- the quantity a cost model actually charges."""
    return weights.diff().abs().sum(axis=1) / 2.0


def concentration_hhi(weights: pd.DataFrame) -> pd.Series:
    """Herfindahl index of realised weights: 1/n is equal-weight, 1.0 is all-in-one."""
    return (weights ** 2).sum(axis=1)


## The strategies

Three kinds. Simple rules that say hold or don't. Classical portfolio optimisers that emit weights. And overlays that condition either on the market regime. There is also a deliberately random strategy in here — it is the control, and it tells you what a coin flip earns over the same stretch. Without it you have no idea whether a positive return means anything.

In [ ]:
# ── nifty_rl/strategies/allocators.py ───────────────────────────────────────────────────
# Classical portfolio construction: equal weight, minimum variance, maximum Sharpe,
# risk parity, HRP. These are the fair opponents for an agent that emits weights.
"""Portfolio-construction baselines.

The notebook benchmarked a continuous *allocator* (PPO emitting softmax weights) only
against binary *timing* rules -- RSI, MA crossover, breakout. Those answer a different
question. The natural opponents for a weight-emitting agent are weight-emitting methods,
and their absence was the single most conspicuous gap in the evaluation.

Every allocator here takes a trailing window of returns and emits long-only weights that
sum to one. Estimation never sees beyond the rebalance date.
"""
from __future__ import annotations

from typing import Callable, Dict

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.optimize import minimize
from scipy.spatial.distance import squareform

AllocatorFn = Callable[[pd.DataFrame], pd.Series]


def _clean(returns: pd.DataFrame) -> pd.DataFrame:
    return returns.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all").fillna(0.0)


def _normalise(weights: np.ndarray, columns) -> pd.Series:
    weights = np.clip(np.asarray(weights, dtype=float), 0.0, None)
    total = weights.sum()
    if total <= 0 or not np.isfinite(total):
        weights = np.ones(len(columns)) / len(columns)
    else:
        weights = weights / total
    return pd.Series(weights, index=columns, name="weight")


def _covariance(returns: pd.DataFrame, shrinkage: bool = True) -> np.ndarray:
    """Sample covariance, optionally Ledoit-Wolf shrunk.

    Shrinkage matters here: with 10 assets and a 60-day window the sample covariance is
    badly conditioned, and min-variance optimisers famously concentrate into whichever
    asset the noise happened to favour.
    """
    if shrinkage and len(returns) > len(returns.columns):
        try:
            from sklearn.covariance import LedoitWolf

            return LedoitWolf().fit(returns.to_numpy()).covariance_
        except Exception:
            pass
    return np.cov(returns.to_numpy(), rowvar=False)


# ------------------------------------------------------------------- simple rules


def equal_weight(returns: pd.DataFrame) -> pd.Series:
    cols = _clean(returns).columns
    return _normalise(np.ones(len(cols)), cols)


def inverse_volatility(returns: pd.DataFrame) -> pd.Series:
    r = _clean(returns)
    vol = r.std().replace(0, np.nan)
    inv = (1.0 / vol).fillna(0.0)
    return _normalise(inv.to_numpy(), r.columns)


def momentum_tilted(returns: pd.DataFrame, lookback: int = 60) -> pd.Series:
    """Equal weight tilted toward positive trailing momentum; negatives get zero."""
    r = _clean(returns).tail(lookback)
    cumulative = (1.0 + r).prod() - 1.0
    tilt = cumulative.clip(lower=0.0)
    if tilt.sum() <= 0:
        return equal_weight(returns)
    return _normalise(tilt.to_numpy(), r.columns)


# ------------------------------------------------------------------- optimisers


def minimum_variance(returns: pd.DataFrame, shrinkage: bool = True) -> pd.Series:
    """Long-only minimum-variance portfolio."""
    r = _clean(returns)
    n = r.shape[1]
    if n == 0:
        return pd.Series(dtype=float)
    cov = _covariance(r, shrinkage)

    def objective(w):
        return float(w @ cov @ w)

    result = minimize(
        objective,
        x0=np.ones(n) / n,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
        options={"maxiter": 500, "ftol": 1e-12},
    )
    return _normalise(result.x if result.success else np.ones(n) / n, r.columns)


def maximum_sharpe(
    returns: pd.DataFrame, risk_free_daily: float = 0.0, shrinkage: bool = True
) -> pd.Series:
    """Long-only tangency portfolio on the trailing window."""
    r = _clean(returns)
    n = r.shape[1]
    if n == 0:
        return pd.Series(dtype=float)
    cov = _covariance(r, shrinkage)
    mu = r.mean().to_numpy() - risk_free_daily

    def negative_sharpe(w):
        vol = np.sqrt(max(w @ cov @ w, 1e-18))
        return float(-(w @ mu) / vol)

    result = minimize(
        negative_sharpe,
        x0=np.ones(n) / n,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
        options={"maxiter": 500, "ftol": 1e-12},
    )
    return _normalise(result.x if result.success else np.ones(n) / n, r.columns)


def risk_parity(returns: pd.DataFrame, shrinkage: bool = True, iterations: int = 500) -> pd.Series:
    """Equal risk contribution: every asset supplies the same share of portfolio variance.

    Solved by the standard fixed-point iteration ``w_i <- w_i * (target / RC_i)`` rather
    than a general optimiser -- it is faster and cannot wander into a local minimum.
    """
    r = _clean(returns)
    n = r.shape[1]
    if n == 0:
        return pd.Series(dtype=float)
    cov = _covariance(r, shrinkage)

    w = np.ones(n) / n
    for _ in range(iterations):
        # Risk contribution of asset i = w_i * (Cov @ w)_i. These sum to portfolio
        # variance, so an equal split means each asset supplies `total / n` of the risk.
        marginal = cov @ w
        contribution = w * marginal
        total = contribution.sum()
        if total <= 0 or not np.isfinite(total):
            break
        target = total / n
        # Nudge each weight toward its target: an asset contributing more risk than its
        # share gets scaled down. The square root damps the step -- because changing w_i
        # also changes every other asset's contribution, the full correction overshoots
        # and oscillates instead of converging.
        w = w * (target / np.maximum(contribution, 1e-18)) ** 0.5
        w = np.clip(w, 1e-12, None)
        w = w / w.sum()
    return _normalise(w, r.columns)


# --------------------------------------------------- hierarchical risk parity (HRP)


def _inverse_variance_weights(cov: np.ndarray, indices: np.ndarray) -> np.ndarray:
    """Weight the assets in one cluster inversely to their variance, summing to 1."""
    sub = cov[np.ix_(indices, indices)]
    ivp = 1.0 / np.maximum(np.diag(sub), 1e-18)
    return ivp / ivp.sum()


def _cluster_variance(cov: np.ndarray, indices: np.ndarray) -> float:
    """Variance a cluster would have if held at inverse-variance weights.

    This is how HRP compares two candidate halves without inverting anything: it asks
    "how risky is this bundle if allocated sensibly inside itself?" and splits the
    parent's weight against the answer.
    """
    w = _inverse_variance_weights(cov, indices)
    sub = cov[np.ix_(indices, indices)]
    return float(w @ sub @ w)


def hierarchical_risk_parity(returns: pd.DataFrame) -> pd.Series:
    """Lopez de Prado's HRP.

    Three steps: tree clustering on a correlation distance, quasi-diagonalisation of the
    covariance matrix by the resulting leaf order, and recursive bisection allocating
    between sibling clusters by inverse cluster variance.

    HRP needs no matrix inversion, which is why it stays stable where min-variance
    concentrates -- the strongest of the classical baselines and the one PPO must beat
    for the project to claim anything.
    """
    r = _clean(returns)
    n = r.shape[1]
    if n == 0:
        return pd.Series(dtype=float)
    if n == 1:
        return _normalise(np.ones(1), r.columns)

    cov = np.cov(r.to_numpy(), rowvar=False)
    corr = np.corrcoef(r.to_numpy(), rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0)
    np.fill_diagonal(corr, 1.0)

    # Step 1 -- tree clustering. Turn correlation into a distance so that highly
    # correlated names sit close together: perfectly correlated -> 0, uncorrelated ->
    # 0.71, perfectly opposed -> 1. Single-linkage clustering then orders the assets so
    # that similar ones are adjacent.
    distance = np.sqrt(np.clip((1.0 - corr) / 2.0, 0.0, 1.0))
    np.fill_diagonal(distance, 0.0)
    condensed = squareform(distance, checks=False)

    # Step 2 -- quasi-diagonalisation. `order` is the leaf order of the tree, e.g. the
    # two banks land beside each other rather than at opposite ends. Bisecting this
    # ordering therefore splits the portfolio along genuine similarity lines.
    order = leaves_list(linkage(condensed, method="single"))

    # Step 3 -- recursive bisection. Start with all assets in one cluster holding 100% of
    # the weight. Repeatedly cut each cluster in half and divide its weight between the
    # two halves in inverse proportion to their variance: the calmer half gets more. Each
    # asset's final weight is the product of every split it survived.
    weights = np.ones(n)
    clusters = [order]
    while clusters:
        # Split every multi-asset cluster down the middle. Single assets have nothing
        # left to divide, so they drop out and the loop ends when all clusters are.
        clusters = [
            half
            for cluster in clusters
            if len(cluster) > 1
            for half in (cluster[: len(cluster) // 2], cluster[len(cluster) // 2 :])
        ]
        # Halves are appended in order, so consecutive pairs are always siblings.
        for left, right in zip(clusters[::2], clusters[1::2]):
            var_left = _cluster_variance(cov, left)
            var_right = _cluster_variance(cov, right)
            # alpha is the share going to the left sibling. Note the inversion: a *high*
            # left variance makes the fraction large, so alpha shrinks and the riskier
            # half is scaled down. The two shares sum to 1, conserving parent weight.
            alpha = 1.0 - var_left / (var_left + var_right + 1e-18)
            weights[left] *= alpha
            weights[right] *= 1.0 - alpha

    return _normalise(weights, r.columns)


ALLOCATORS: Dict[str, AllocatorFn] = {
    "EqualWeight": equal_weight,
    "InverseVol": inverse_volatility,
    "MomentumTilt": momentum_tilted,
    "MinVariance": minimum_variance,
    "MaxSharpe": maximum_sharpe,
    "RiskParity": risk_parity,
    "HRP": hierarchical_risk_parity,
}


In [ ]:
# ── nifty_rl/strategies/meta.py ─────────────────────────────────────────────────────────
# Overlays that use the regime — dial exposure down when things look bad, or switch
# strategy entirely.
"""Regime-conditioned meta-strategies.

Two ways to spend a regime signal, in increasing order of commitment:

* **Exposure overlay** -- keep the underlying strategy, scale gross exposure by regime.
  Applies uniformly to every strategy including the baselines, which isolates the value
  of the regime signal itself rather than confounding it with one strategy's quirks.
  Highest information per line of code in the project.
* **Strategy selection** -- learn which strategy wins in each regime on the training
  window, then route out-of-sample by the online regime estimate.

Both consume ``labels`` produced by a detector's ``label_online``, so both inherit the
causality guarantee. Neither may be handed a retrospective segmentation.
"""
from __future__ import annotations

from typing import Callable, Dict, Mapping, Optional, Sequence

import numpy as np
import pandas as pd

SignalFn = Callable[[pd.DataFrame], pd.Series]


def regime_exposure_overlay(
    signal_fn: SignalFn,
    labels: pd.Series,
    exposure_by_regime: Mapping[int, float],
    threshold: float = 0.5,
) -> SignalFn:
    """Gate a binary signal by regime exposure.

    With binary per-ticker signals, "50% exposure" cannot be expressed as a half
    position, so exposure is applied as a participation gate: the signal survives in a
    regime whose exposure clears ``threshold``. For continuous allocators use
    :func:`scale_weights_by_regime` instead, which scales properly.
    """

    def _bound(df: pd.DataFrame) -> pd.Series:
        base = signal_fn(df).astype(int)
        dates = pd.to_datetime(df["Date"])
        regime = labels.reindex(dates).ffill()
        exposure = regime.map(exposure_by_regime).fillna(1.0).to_numpy()
        gated = np.where(exposure >= threshold, base.to_numpy(), 0)
        return pd.Series(gated, index=df.index, name="signal")

    _bound.__name__ = f"regime_gated_{getattr(signal_fn, '__name__', 'signal')}"
    return _bound


def scale_weights_by_regime(
    weights: pd.DataFrame,
    labels: pd.Series,
    exposure_by_regime: Mapping[int, float],
) -> pd.DataFrame:
    """Scale a weight matrix by regime exposure; the remainder sits in cash.

    Rows sum to ``exposure`` rather than 1, so the shortfall is genuine cash and earns
    the risk-free rate in the backtester -- not a silent renormalisation back to fully
    invested, which would quietly undo the de-risking.
    """
    regime = labels.reindex(weights.index).ffill()
    exposure = regime.map(exposure_by_regime).fillna(1.0)
    return weights.mul(exposure, axis=0)


def default_exposure_ladder(n_regimes: int) -> Dict[int, float]:
    """Full exposure in the calmest regime, stepping down to a defensive floor."""
    if n_regimes <= 1:
        return {0: 1.0}
    ladder = np.linspace(1.0, 0.25, n_regimes)
    return {i: float(round(v, 3)) for i, v in enumerate(ladder)}


def learn_regime_strategy_map(
    train_panel: pd.DataFrame,
    labels: pd.Series,
    strategy_fns: Mapping[str, SignalFn],
    backtest_fn: Callable[[pd.DataFrame, SignalFn, str], "object"],
    score_fn: Callable[[object], float],
    min_days: int = 40,
) -> pd.DataFrame:
    """Score every strategy within every regime on the training window.

    Regimes are sliced by *date*, so a strategy is judged only on the days it actually
    faced that regime. Regimes with fewer than ``min_days`` observations are reported but
    excluded from routing -- picking a winner from twenty days is how a meta-strategy
    overfits.
    """
    rows = []
    dates = pd.to_datetime(train_panel["Date"])

    for regime in sorted(pd.unique(labels.dropna())):
        regime_dates = labels.index[labels == regime]
        slice_ = train_panel[dates.isin(regime_dates)]
        n_days = slice_["Date"].nunique() if not slice_.empty else 0

        for name, fn in strategy_fns.items():
            if n_days < min_days:
                rows.append(
                    {"regime": int(regime), "strategy": name, "score": np.nan,
                     "n_days": n_days, "eligible": False}
                )
                continue
            try:
                result = backtest_fn(slice_, fn, name)
                score = float(score_fn(result))
            except Exception:
                score = float("nan")
            rows.append(
                {"regime": int(regime), "strategy": name, "score": score,
                 "n_days": n_days, "eligible": True}
            )

    return pd.DataFrame(rows)


def best_strategy_per_regime(scores: pd.DataFrame, fallback: str) -> Dict[int, str]:
    """Route each regime to its training-window winner, falling back when unreliable."""
    mapping: Dict[int, str] = {}
    for regime, group in scores.groupby("regime"):
        eligible = group[group["eligible"] & group["score"].notna()]
        mapping[int(regime)] = (
            str(eligible.loc[eligible["score"].idxmax(), "strategy"])
            if not eligible.empty
            else fallback
        )
    return mapping


def regime_switching_signal(
    strategy_fns: Mapping[str, SignalFn],
    labels: pd.Series,
    regime_to_strategy: Mapping[int, str],
    fallback: str,
) -> SignalFn:
    """Emit the routed strategy's signal on each date, per its online regime."""

    def _bound(df: pd.DataFrame) -> pd.Series:
        dates = pd.to_datetime(df["Date"])
        regime = labels.reindex(dates).ffill()

        # Evaluate each candidate once over the whole frame, then select per row --
        # cheaper than slicing, and keeps every strategy's internal state continuous
        # rather than restarting it at each regime boundary.
        computed = {name: fn(df).astype(int).to_numpy() for name, fn in strategy_fns.items()}
        chosen = np.zeros(len(df), dtype=int)
        for i, r in enumerate(regime.to_numpy()):
            name = regime_to_strategy.get(int(r), fallback) if np.isfinite(r) else fallback
            chosen[i] = computed.get(name, computed[fallback])[i]
        return pd.Series(chosen, index=df.index, name="signal")

    _bound.__name__ = "regime_switching"
    return _bound


def performance_by_regime(
    returns_by_strategy: Mapping[str, pd.Series],
    labels: pd.Series,
    regime_names: Optional[Sequence[str]] = None,
    trading_days: int = 252,
    risk_free_annual: float = 0.0,
) -> pd.DataFrame:
    """Long-format strategy x regime performance.

    The table the project was missing. It answers directly whether a drawdown-penalised
    agent earns its penalty when conditions are bad, instead of reporting one blended
    number over a window that happened to be a drawdown.

    Sharpe comes from the shared helper so the degenerate-cash guard applies here too. A
    strategy parked in cash has constant returns *within every regime slice*, so this
    table is if anything more exposed to the 0/0 blow-up than the headline one.
    """
    # flattened: sharpe_from_returns is defined above

    rf_daily = (
        (1.0 + risk_free_annual) ** (1.0 / trading_days) - 1.0 if risk_free_annual > 0 else 0.0
    )

    rows = []
    for name, returns in returns_by_strategy.items():
        aligned = pd.concat(
            [returns.rename("ret"), labels.rename("regime")], axis=1, join="inner"
        ).dropna()
        for regime, group in aligned.groupby("regime"):
            series = group["ret"]
            label = (
                regime_names[int(regime)]
                if regime_names is not None and int(regime) < len(regime_names)
                else f"regime_{int(regime)}"
            )
            equity = (1.0 + series).cumprod()
            rows.append(
                {
                    "strategy": name,
                    "regime": label,
                    "regime_index": int(regime),
                    "n_days": int(len(series)),
                    "total_return": float(equity.iloc[-1] - 1.0) if len(equity) else np.nan,
                    "annual_return": float(series.mean() * trading_days),
                    "excess_annual_return": float((series.mean() - rf_daily) * trading_days),
                    "volatility": float(series.std() * np.sqrt(trading_days)),
                    "sharpe": sharpe_from_returns(series.to_numpy(), trading_days, rf_daily),
                    "max_drawdown": float((equity / equity.cummax() - 1.0).min())
                    if len(equity)
                    else np.nan,
                }
            )
    return pd.DataFrame(rows)


In [ ]:
# ── nifty_rl/strategies/signals.py ──────────────────────────────────────────────────────
# The rules — moving averages, RSI, breakout — plus the random one that acts as the
# control.
"""Rule-based signal generators.

Every function takes a single-ticker frame and returns a 0/1 Series aligned to it.

The central fix here is bug #1. The notebook's ``ma_crossover_signals`` and
``breakout_signals`` recomputed their rolling windows *on whatever slice they were
handed*::

    sig = (price.rolling(short).mean() > price.rolling(long).mean()).astype(int)

Since the caller passes a split (or a walk-forward window), the first ``long`` rows come
back NaN and collapse to 0. That forced **50 of 222 test days flat (23%)** and **50 of
120 walk-forward days flat (42%)** -- MA_20_50 was structurally handicapped in exactly
the evaluation that concluded it underperformed. The frame already carries ``ma20``,
``ma50``, ``high20_prev`` and ``low10_prev`` computed over full history in the feature
layer, and ``momentum_pullback_signals`` already used them; the others now do too.

Rule: **no signal function may call ``.rolling()``.** Anything needing a window belongs
in ``features.py``, where it sees the whole series.
"""
from __future__ import annotations

from typing import Callable, Dict, Optional

import numpy as np
import pandas as pd


def _require(df: pd.DataFrame, *columns: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(
            f"Signal requires precomputed feature column(s) {missing}. "
            "Run features.add_features() before backtesting -- signal functions must "
            "never recompute rolling windows on a split slice (bug #1)."
        )


def _as_signal(values, index) -> pd.Series:
    return pd.Series(np.asarray(values, dtype=int), index=index, name="signal")


def buy_hold_signals(df: pd.DataFrame) -> pd.Series:
    return _as_signal(np.ones(len(df)), df.index)


def ma_crossover_signals(
    df: pd.DataFrame,
    short: int = 20,
    long: int = 50,
    vix_limit: Optional[float] = None,
) -> pd.Series:
    """Long while the short MA is above the long MA.

    Uses the precomputed ``ma{short}``/``ma{long}`` columns so the first ``long`` bars of
    a split are not silently forced flat.
    """
    short_col, long_col = f"ma{short}", f"ma{long}"
    _require(df, short_col, long_col)
    sig = (df[short_col] > df[long_col]).astype(int)
    if vix_limit is not None:
        _require(df, "india_vix")
        sig = sig.where(df["india_vix"] <= vix_limit, 0)
    return _as_signal(sig.fillna(0), df.index)


def rsi_mean_reversion_signals(
    df: pd.DataFrame,
    buy_below: float = 35.0,
    sell_above: float = 60.0,
    trend_filter: bool = True,
) -> pd.Series:
    """Enter when oversold, exit when the bounce matures.

    Stateful by nature (entry and exit thresholds differ), so it runs as a scan -- but
    over precomputed ``rsi``/``ma50``, not recomputed ones.
    """
    _require(df, "rsi")
    if trend_filter:
        _require(df, "ma50")
        trend_ok = (df["price"] > df["ma50"]).to_numpy()
    else:
        trend_ok = np.ones(len(df), dtype=bool)

    rsi = df["rsi"].to_numpy()
    out = np.zeros(len(df), dtype=int)
    position = 0
    for i in range(len(df)):
        if position == 0:
            if rsi[i] < buy_below and trend_ok[i]:
                position = 1
        else:
            if rsi[i] > sell_above or not trend_ok[i]:
                position = 0
        out[i] = position
    return _as_signal(out, df.index)


def breakout_signals(
    df: pd.DataFrame,
    lookback: int = 20,
    exit_lookback: int = 10,
    vix_limit: Optional[float] = None,
) -> pd.Series:
    """Long on a breakout above the prior N-day high; exit below the prior M-day low."""
    if lookback == 20 and exit_lookback == 10:
        _require(df, "high20_prev", "low10_prev")
        entry = (df["price"] > df["high20_prev"]).to_numpy()
        exit_ = (df["price"] < df["low10_prev"]).to_numpy()
    else:
        # Non-default windows are not precomputed; derive them from the full-history
        # channel columns if present, else raise rather than silently warm up on a slice.
        _require(df, f"high{lookback}_prev", f"low{exit_lookback}_prev")
        entry = (df["price"] > df[f"high{lookback}_prev"]).to_numpy()
        exit_ = (df["price"] < df[f"low{exit_lookback}_prev"]).to_numpy()

    if vix_limit is not None:
        _require(df, "india_vix")
        allowed = (df["india_vix"] <= vix_limit).to_numpy()
    else:
        allowed = np.ones(len(df), dtype=bool)

    out = np.zeros(len(df), dtype=int)
    position = 0
    for i in range(len(df)):
        if position == 0:
            if entry[i] and allowed[i]:
                position = 1
        else:
            if exit_[i] or not allowed[i]:
                position = 0
        out[i] = position
    return _as_signal(out, df.index)


def momentum_pullback_signals(
    df: pd.DataFrame,
    rsi_max: float = 55.0,
    rsi_min: float = 35.0,
    vix_limit: Optional[float] = None,
) -> pd.Series:
    """Uptrend confirmed, entered on an RSI pullback rather than at the highs."""
    _require(df, "ma20", "ma50", "rsi")
    trend = (df["price"] > df["ma50"]) & (df["ma20"] > df["ma50"])
    pullback = (df["rsi"] < rsi_max) & (df["rsi"] > rsi_min)
    sig = (trend & pullback).astype(int)
    if vix_limit is not None:
        _require(df, "india_vix")
        sig = sig.where(df["india_vix"] <= vix_limit, 0)
    return _as_signal(sig.fillna(0), df.index)


def sentiment_momentum_signals(df: pd.DataFrame) -> pd.Series:
    """VIX-proxy sentiment improving while price holds above its 20-day mean.

    Note that ``sentiment`` is derived from India VIX and is therefore market-wide --
    identical across every ticker. It carries no cross-sectional information.
    """
    _require(df, "sent_momentum", "ma20")
    sig = ((df["sent_momentum"] > 0) & (df["price"] > df["ma20"])).astype(int)
    return _as_signal(sig.fillna(0), df.index)


def vix_regime_momentum_signals(df: pd.DataFrame, quantile_col: str = "vix_q40") -> pd.Series:
    """Long only in a low-VIX regime with a confirmed uptrend."""
    _require(df, "ma20", "ma50")
    if quantile_col in df.columns:
        low_vix = df["india_vix"] < df[quantile_col]
    else:
        _require(df, "high_vix_regime")
        low_vix = df["high_vix_regime"] <= 0
    uptrend = (df["ma20"] > df["ma50"]) & (df["price"] > df["ma20"])
    sig = (low_vix & uptrend).astype(int)
    return _as_signal(sig.fillna(0), df.index)


def random_policy_signals(
    df: pd.DataFrame,
    probability: float = 0.5,
    seed: int = 42,
    ticker_offset: int = 0,
) -> pd.Series:
    """Independent random baseline.

    The notebook seeded every ticker with the same value, so all ten received an
    identical signal sequence and the "random" portfolio went all-in and all-out
    simultaneously across the whole book (bug #21). That concentration -- not
    randomness -- is what produced its -23.88% print. ``ticker_offset`` decorrelates the
    draws so this behaves like an actual random-portfolio control.
    """
    rng = np.random.default_rng(seed + ticker_offset)
    return _as_signal((rng.random(len(df)) < probability).astype(int), df.index)


SIGNAL_REGISTRY: Dict[str, Callable[..., pd.Series]] = {
    "buy_hold": buy_hold_signals,
    "ma": ma_crossover_signals,
    "rsi": rsi_mean_reversion_signals,
    "breakout": breakout_signals,
    "momentum_pullback": momentum_pullback_signals,
    "sentiment_momentum": sentiment_momentum_signals,
    "vix_regime_momentum": vix_regime_momentum_signals,
    "random": random_policy_signals,
}


def make_signal_fn(kind: str, **params) -> Callable[[pd.DataFrame], pd.Series]:
    """Bind a registry entry and its parameters into a single-argument callable.

    The notebook's ``_signal_fn`` dispatcher silently returned an all-zero Series for any
    unregistered name -- which is how ``VIX_Regime_Momentum`` ran as a no-op until it was
    noticed. Unknown kinds raise here instead.
    """
    if kind not in SIGNAL_REGISTRY:
        raise KeyError(
            f"Unknown signal kind {kind!r}. Registered: {sorted(SIGNAL_REGISTRY)}"
        )
    fn = SIGNAL_REGISTRY[kind]

    def _bound(df: pd.DataFrame) -> pd.Series:
        return fn(df, **params)

    _bound.__name__ = f"signal_{kind}"
    return _bound


def make_random_signal_fn(seed: int, probability: float = 0.5) -> Callable[[pd.DataFrame], pd.Series]:
    """Random baseline whose draws are decorrelated across tickers."""
    counter = {"n": 0}

    def _bound(df: pd.DataFrame) -> pd.Series:
        offset = counter["n"]
        counter["n"] += 1
        return random_policy_signals(df, probability=probability, seed=seed, ticker_offset=offset)

    _bound.__name__ = "signal_random"
    return _bound


## Regime detection

The idea is that markets have moods — calm stretches, nervous ones, outright crises — and that knowing which one you are in might be worth something.

The hard part is honesty. It is trivially easy to build a regime label that looks brilliant and is useless, because it was computed with knowledge of how the period ended. Every detector below is held to one rule: **the label for a given day may only use that day and the days before it.** Five different detectors are implemented so they can be checked against each other.

In [ ]:
# ── nifty_rl/regimes/base.py ────────────────────────────────────────────────────────────
# The contract every detector signs. Fit however you like; predict using only the
# past.
"""Regime detector interface and the causality contract.

Every online detector must satisfy one property::

    predict_online(X[:t]).iloc[-1] == predict_online(X).iloc[t-1]

That is, the estimate for day *t* depends only on days up to and including *t*. It is
the entire basis on which any regime-conditioned result can be believed.

This is where regime-switching projects quietly break. ``hmmlearn.predict()`` runs
Viterbi over the whole sequence; ``predict_proba()`` returns forward-backward *smoothed*
posteriors. Both read the future. Only the normalised forward pass (alpha_t) is
admissible online, which is why the Gaussian HMM here is implemented in-package rather
than delegated -- the guarantee becomes structural instead of a matter of remembering
which method to call.

Retrospective detectors (change-point segmentation) deliberately do **not** implement
``predict_online``. They are diagnostics: used to hand-label ground-truth breaks against
which the online detectors' *detection lag* is measured. Mixing the two is the mistake.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import List, Optional, Sequence

import numpy as np
import pandas as pd


class RegimeDetector(ABC):
    """Base class for online (causal) regime detectors.

    Subclasses implement exactly two private methods and inherit everything else:

    * ``_fit(X)`` — estimate parameters from a training matrix. May use the whole block;
      this is parameter estimation on data the model is allowed to see.
    * ``_filter(X)`` — return filtered probabilities where **row t depends only on rows
      0..t**. This is the load-bearing contract of the entire package.

    Splitting them this way is what makes causality checkable. The public methods
    (:meth:`predict_online`, :meth:`label_online`) route only through ``_filter``, so no
    subclass can accidentally expose a smoothed or Viterbi path as if it were live — and
    :func:`assert_causal` can verify any implementation by re-running it on prefixes and
    checking that past labels never change.
    """

    #: Human-readable name used in reports and figures.
    name: str = "base"

    def __init__(self, n_regimes: int = 3, feature_columns: Optional[Sequence[str]] = None):
        self.n_regimes = int(n_regimes)
        self.feature_columns = list(feature_columns) if feature_columns else None
        self._fitted = False
        self.regime_labels_: List[str] = []

    # ------------------------------------------------------------------ interface

    @abstractmethod
    def _fit(self, X: np.ndarray) -> None:
        """Estimate parameters from the training matrix."""

    @abstractmethod
    def _filter(self, X: np.ndarray) -> np.ndarray:
        """Return the ``(n_samples, n_regimes)`` filtered probability matrix.

        Row *t* must be computable from rows ``0..t`` alone.
        """

    # ------------------------------------------------------------------- plumbing

    def _matrix(self, X: pd.DataFrame) -> np.ndarray:
        cols = self.feature_columns or list(X.columns)
        missing = [c for c in cols if c not in X.columns]
        if missing:
            raise KeyError(f"{self.name}: missing regime feature column(s) {missing}")
        values = X[cols].to_numpy(dtype=float)
        return np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)

    def fit(self, X: pd.DataFrame) -> "RegimeDetector":
        """Estimate parameters, remembering which columns were used.

        Pinning ``feature_columns`` on the first fit means a later call with differently
        ordered or extra columns raises instead of silently feeding the model a different
        feature in each slot -- a failure that produces plausible labels and invalid
        results. The row-count guard exists for the same reason: fitting three regimes on
        twenty days succeeds numerically and means nothing.
        """
        if self.feature_columns is None:
            self.feature_columns = list(X.columns)
        matrix = self._matrix(X)
        if len(matrix) < self.n_regimes * 10:
            raise ValueError(
                f"{self.name}: {len(matrix)} training rows is too few for "
                f"{self.n_regimes} regimes."
            )
        self._fit(matrix)
        self._fitted = True
        return self

    def predict_online(self, X: pd.DataFrame) -> pd.DataFrame:
        """Filtered ``P(regime_t | information up to t)``, one column per regime."""
        if not self._fitted:
            raise RuntimeError(f"{self.name}: call fit() before predict_online().")
        probabilities = self._filter(self._matrix(X))
        columns = self.regime_labels_ or [f"regime_{i}" for i in range(self.n_regimes)]
        return pd.DataFrame(probabilities, index=X.index, columns=columns)

    def label_online(self, X: pd.DataFrame) -> pd.Series:
        """Hard regime label (argmax of the filtered distribution)."""
        probabilities = self.predict_online(X)
        return pd.Series(
            np.argmax(probabilities.to_numpy(), axis=1), index=X.index, name="regime"
        )

    # ---------------------------------------------------------------- diagnostics

    def state_ordering_key(self, X: pd.DataFrame, column: str) -> List[int]:
        """Order states by their mean value of ``column`` -- used to stabilise labels."""
        labels = self.label_online(X)
        means = pd.Series(X[column].to_numpy(), index=X.index).groupby(labels).mean()
        return list(means.sort_values().index)


class RetrospectiveSegmenter(ABC):
    """Full-sequence structural-break detector.

    Explicitly *not* a :class:`RegimeDetector`: it sees the whole series and therefore
    cannot be traded. Its role is to produce ground-truth break dates for the
    detection-lag evaluation.
    """

    name: str = "segmenter"

    @abstractmethod
    def breakpoints(self, series: pd.Series) -> List[pd.Timestamp]:
        """Return the dates at which the series changes regime."""


def assert_causal(
    detector: RegimeDetector,
    X: pd.DataFrame,
    start: int = 60,
    step: int = 1,
    atol: float = 1e-8,
) -> None:
    """Verify the causality contract by brute force.

    Recomputes the filtered distribution on every prefix and checks that the last row
    matches the corresponding row of the full-sample run. Raises ``AssertionError`` on
    the first violation. Deliberately slow -- it is a proof, not a fast path.
    """
    full = detector.predict_online(X).to_numpy()
    for t in range(start, len(X) + 1, step):
        prefix = detector.predict_online(X.iloc[:t]).to_numpy()
        if not np.allclose(prefix[-1], full[t - 1], atol=atol):
            raise AssertionError(
                f"{detector.name} is not causal at t={t}: "
                f"prefix={prefix[-1]} vs full={full[t - 1]}"
            )


In [ ]:
# ── nifty_rl/regimes/changepoint.py ─────────────────────────────────────────────────────
# Finds breaks by looking at the whole series at once — which makes it useless for
# trading and ideal as a yardstick.
"""Retrospective change-point segmentation.

Deliberately **not** a :class:`~nifty_rl.regimes.base.RegimeDetector`. Binary
segmentation sees the entire series, so its breaks cannot be traded -- but that is
precisely what makes them useful as *ground truth*: the dates at which the process
demonstrably changed, established with hindsight, against which each online detector's
**detection lag** is measured.

Treating a retrospective segmentation as a tradeable signal is the single most common
way regime work becomes accidental lookahead. Keeping the two in separate classes makes
the mistake hard to make.

Implemented directly rather than via ``ruptures`` so the core evaluation has no optional
dependency; ``ruptures`` remains available for PELT if a larger search is wanted.
"""
from __future__ import annotations

from typing import List, Tuple

import numpy as np
import pandas as pd



def _gaussian_cost(values: np.ndarray) -> float:
    """Negative log-likelihood of a segment under a Gaussian with free mean/variance."""
    n = len(values)
    if n < 2:
        return 0.0
    variance = float(np.var(values))
    if variance <= 0:
        return 0.0
    return float(n * np.log(variance))


class BinarySegmentation(RetrospectiveSegmenter):
    """Greedy binary segmentation with a BIC-style penalty per break."""

    name = "binary_segmentation"

    def __init__(self, min_size: int = 40, max_breaks: int = 12, penalty: float = 12.0):
        self.min_size = int(min_size)
        self.max_breaks = int(max_breaks)
        self.penalty = float(penalty)

    def _best_split(self, values: np.ndarray, start: int, end: int) -> Tuple[float, int]:
        baseline = _gaussian_cost(values[start:end])
        best_gain, best_index = 0.0, -1
        for split in range(start + self.min_size, end - self.min_size + 1):
            gain = baseline - (
                _gaussian_cost(values[start:split]) + _gaussian_cost(values[split:end])
            )
            if gain > best_gain:
                best_gain, best_index = gain, split
        return best_gain, best_index

    def breakpoint_indices(self, series: pd.Series) -> List[int]:
        values = np.nan_to_num(series.to_numpy(dtype=float), nan=0.0)
        n = len(values)
        if n < 2 * self.min_size:
            return []

        breaks: List[int] = []
        segments = [(0, n)]
        while len(breaks) < self.max_breaks:
            candidates = []
            for start, end in segments:
                if end - start < 2 * self.min_size:
                    continue
                gain, index = self._best_split(values, start, end)
                if index > 0:
                    candidates.append((gain, index, start, end))
            if not candidates:
                break
            gain, index, start, end = max(candidates)
            if gain < self.penalty:
                break
            breaks.append(index)
            segments.remove((start, end))
            segments.extend([(start, index), (index, end)])

        return sorted(breaks)

    def breakpoints(self, series: pd.Series) -> List[pd.Timestamp]:
        return [series.index[i] for i in self.breakpoint_indices(series)]

    def segment_labels(self, series: pd.Series) -> pd.Series:
        """Integer segment id per observation -- the retrospective 'true' regime path."""
        indices = self.breakpoint_indices(series)
        labels = np.zeros(len(series), dtype=int)
        for segment_id, start in enumerate(indices, start=1):
            labels[start:] = segment_id
        return pd.Series(labels, index=series.index, name="segment")


In [ ]:
# ── nifty_rl/regimes/evaluate.py ────────────────────────────────────────────────────────
# Is the detector any good? How fast does it react, how long do its regimes last, and
# does 'state 0' still mean the same thing after a refit?
"""Validating the regime models themselves.

Most regime work stops at "here is the fitted state path". That is the part that cannot
be checked. These diagnostics are the part that can:

* **Persistence** -- a model that flips every three days is untradeable after costs, no
  matter how well it fits.
* **Detection lag** -- days between a retrospectively established break and the online
  detector flagging it. This is a *kill criterion*: a detector that recognises a crisis
  fifteen days late adds cost and provides no protection, and no downstream overlay can
  rescue it.
* **Refit stability** -- whether an expanding-window refit keeps labels consistent, or
  whether "state 0" silently changes meaning.
* **Cross-method agreement** -- pairwise Cohen's kappa. Disagreement between backends is
  itself informative and worth reporting rather than hiding behind whichever one looked
  best.
* **Economic sanity** -- do the discovered states look like anything a portfolio manager
  would recognise?
"""
from __future__ import annotations

from typing import Dict, Optional, Sequence

import numpy as np
import pandas as pd



# ------------------------------------------------------------------- persistence


def run_lengths(labels: pd.Series) -> pd.Series:
    """Length of each contiguous run of a constant label."""
    values = labels.to_numpy()
    if len(values) == 0:
        return pd.Series(dtype=float)
    change = np.flatnonzero(np.diff(values)) + 1
    boundaries = np.concatenate([[0], change, [len(values)]])
    return pd.Series(np.diff(boundaries), name="run_length")


def persistence_summary(labels: pd.Series, regime_names: Optional[Sequence[str]] = None) -> pd.DataFrame:
    """Occupancy and mean run length per regime, plus the overall switch rate."""
    runs = run_lengths(labels)
    values = labels.to_numpy()
    n_switches = int((np.diff(values) != 0).sum())

    rows = []
    for regime in sorted(pd.unique(values)):
        mask = values == regime
        regime_runs = []
        current = 0
        for flag in mask:
            if flag:
                current += 1
            elif current:
                regime_runs.append(current)
                current = 0
        if current:
            regime_runs.append(current)

        name = (
            regime_names[regime]
            if regime_names is not None and regime < len(regime_names)
            else f"regime_{regime}"
        )
        rows.append(
            {
                "regime": name,
                "occupancy": float(mask.mean()),
                "n_episodes": len(regime_runs),
                "mean_run_days": float(np.mean(regime_runs)) if regime_runs else 0.0,
                "median_run_days": float(np.median(regime_runs)) if regime_runs else 0.0,
                "max_run_days": int(np.max(regime_runs)) if regime_runs else 0,
            }
        )

    frame = pd.DataFrame(rows)
    frame.attrs["n_switches"] = n_switches
    frame.attrs["switch_rate"] = n_switches / max(len(values) - 1, 1)
    frame.attrs["overall_mean_run"] = float(runs.mean()) if len(runs) else 0.0
    return frame


def is_tradeable(labels: pd.Series, min_mean_run_days: float = 10.0) -> bool:
    """Crude gate: mean run length must exceed a cost-driven floor."""
    runs = run_lengths(labels)
    return bool(len(runs) and runs.mean() >= min_mean_run_days)


# ----------------------------------------------------------------- detection lag


def detection_lag(
    online_labels: pd.Series,
    break_dates: Sequence[pd.Timestamp],
    max_horizon: int = 60,
) -> pd.DataFrame:
    """Days from each ground-truth break to the next online label change.

    ``lag = NaN`` means the detector never reacted within ``max_horizon`` -- a miss, and
    strictly worse than a slow detection.
    """
    index = online_labels.index
    values = online_labels.to_numpy()
    changes = np.flatnonzero(np.diff(values)) + 1

    rows = []
    for break_date in break_dates:
        position = index.searchsorted(break_date)
        if position >= len(index):
            continue
        following = changes[changes >= position]
        following = following[following <= position + max_horizon]
        if len(following) == 0:
            rows.append({"break_date": break_date, "lag_days": np.nan, "detected": False})
        else:
            rows.append(
                {
                    "break_date": break_date,
                    "lag_days": int(following[0] - position),
                    "detected": True,
                }
            )
    return pd.DataFrame(rows)


def lag_summary(lags: pd.DataFrame) -> Dict[str, float]:
    if lags.empty:
        return {"n_breaks": 0, "detection_rate": float("nan"), "median_lag": float("nan")}
    detected = lags[lags["detected"]]
    return {
        "n_breaks": int(len(lags)),
        "detection_rate": float(lags["detected"].mean()),
        "median_lag": float(detected["lag_days"].median()) if len(detected) else float("nan"),
        "mean_lag": float(detected["lag_days"].mean()) if len(detected) else float("nan"),
        "worst_lag": float(detected["lag_days"].max()) if len(detected) else float("nan"),
    }


# ------------------------------------------------------------------- agreement


def agreement_matrix(label_map: Dict[str, pd.Series]) -> pd.DataFrame:
    """Pairwise Cohen's kappa between detectors.

    Kappa rather than raw agreement because regimes are unbalanced -- a detector sitting
    in "Normal" 70% of the time agrees with everything by chance.
    """
    from sklearn.metrics import cohen_kappa_score

    names = list(label_map)
    matrix = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
    for i, left in enumerate(names):
        for right in names[i + 1 :]:
            aligned = pd.concat(
                [label_map[left].rename("a"), label_map[right].rename("b")],
                axis=1,
                join="inner",
            ).dropna()
            if aligned.empty:
                score = np.nan
            else:
                score = float(cohen_kappa_score(aligned["a"], aligned["b"]))
            matrix.loc[left, right] = score
            matrix.loc[right, left] = score
    return matrix


# --------------------------------------------------------------- refit stability


def refit_stability(
    make_detector,
    features: pd.DataFrame,
    initial_train: int = 500,
    step: int = 120,
) -> pd.DataFrame:
    """Refit on an expanding window and measure label agreement with the prior fit.

    Low agreement means the state definitions are moving, so regime-conditional results
    are not comparable across time -- the failure that makes regime work irreproducible.
    """
    from sklearn.metrics import cohen_kappa_score

    rows = []
    previous_labels = None
    for end in range(initial_train, len(features) + 1, step):
        window = features.iloc[:end]
        detector = make_detector()
        try:
            detector.fit(window)
        except Exception as exc:  # pragma: no cover
            rows.append({"train_end": features.index[end - 1], "kappa_vs_previous": np.nan, "error": str(exc)})
            continue
        labels = detector.label_online(features.iloc[:end])
        if previous_labels is not None:
            overlap = labels.index.intersection(previous_labels.index)
            kappa = float(
                cohen_kappa_score(labels.loc[overlap], previous_labels.loc[overlap])
            ) if len(overlap) > 1 else np.nan
        else:
            kappa = np.nan
        rows.append(
            {
                "train_end": features.index[end - 1],
                "n_train": end,
                "kappa_vs_previous": kappa,
                "error": "",
            }
        )
        previous_labels = labels
    return pd.DataFrame(rows)


# ------------------------------------------------------------- economic profile


def regime_conditional_stats(
    returns: pd.Series,
    labels: pd.Series,
    regime_names: Optional[Sequence[str]] = None,
    trading_days: int = 252,
) -> pd.DataFrame:
    """Return, volatility, Sharpe and drawdown within each regime.

    The sanity check: if the "crisis" state does not show higher volatility and worse
    drawdown than the "calm" state, the model has not found regimes -- it has found
    clusters.
    """
    # flattened: sharpe_from_returns is defined above

    aligned = pd.concat(
        [returns.rename("ret"), labels.rename("regime")], axis=1, join="inner"
    ).dropna()

    rows = []
    for regime, group in aligned.groupby("regime"):
        series = group["ret"]
        name = (
            regime_names[int(regime)]
            if regime_names is not None and int(regime) < len(regime_names)
            else f"regime_{int(regime)}"
        )
        equity = (1.0 + series).cumprod()
        drawdown = float((equity / equity.cummax() - 1.0).min()) if len(equity) else np.nan
        rows.append(
            {
                "regime": name,
                "regime_index": int(regime),
                "n_days": int(len(series)),
                "share_of_sample": float(len(series) / len(aligned)),
                "mean_return_annual": float(series.mean() * trading_days),
                "volatility_annual": float(series.std() * np.sqrt(trading_days)),
                "sharpe": sharpe_from_returns(series.to_numpy(), trading_days),
                "max_drawdown": drawdown,
                "hit_rate": float((series > 0).mean()),
            }
        )
    return pd.DataFrame(rows).sort_values("regime_index").reset_index(drop=True)


In [ ]:
# ── nifty_rl/regimes/features.py ────────────────────────────────────────────────────────
# What the detectors actually look at — volatility, trend, breadth, how correlated
# everything has become. Market-wide, not per-stock.
"""Market-level regime feature panel.

Regimes are a property of the *market*, not of any single name, so these are computed
once per date from the whole universe rather than per ticker.

Every column is causal by construction: rolling windows look backward only, and nothing
is z-scored against full-sample statistics (a common silent leak -- standardising the
whole series bakes the future's mean and variance into every row).
"""
from __future__ import annotations

from typing import List, Optional

import numpy as np
import pandas as pd

REGIME_FEATURES: List[str] = [
    "realized_vol_21",
    "realized_vol_5",
    "vix_level",
    "vix_change_5",
    "trend_21",
    "dispersion",
    "mean_correlation",
    "breadth",
]


def build_regime_features(
    panel: pd.DataFrame,
    benchmark_column: str = "benchmark_return",
    trading_days: int = 252,
) -> pd.DataFrame:
    """Build the daily market-level regime feature panel, indexed by date."""
    wide_prices = panel.pivot_table(index="Date", columns="ticker", values="price", aggfunc="last")
    wide_prices = wide_prices.sort_index()
    wide_returns = wide_prices.pct_change()

    market_return = _market_return(panel, wide_returns, benchmark_column)

    out = pd.DataFrame(index=wide_prices.index)
    out.index.name = "Date"

    # --- volatility axis
    out["realized_vol_21"] = market_return.rolling(21).std() * np.sqrt(trading_days)
    out["realized_vol_5"] = market_return.rolling(5).std() * np.sqrt(trading_days)

    # --- implied fear
    if "india_vix" in panel.columns:
        vix = panel.groupby("Date")["india_vix"].first().reindex(out.index)
        out["vix_level"] = vix
        out["vix_change_5"] = vix.pct_change(5)
    else:
        out["vix_level"] = np.nan
        out["vix_change_5"] = np.nan

    # --- trend axis
    out["trend_21"] = (1.0 + market_return).rolling(21).apply(np.prod, raw=True) - 1.0

    # --- cross-sectional structure
    out["dispersion"] = wide_returns.std(axis=1)
    out["mean_correlation"] = _rolling_mean_pairwise_correlation(wide_returns, window=21)

    # --- participation
    moving_average = wide_prices.rolling(50).mean()
    out["breadth"] = (wide_prices > moving_average).mean(axis=1)

    return out


def _market_return(
    panel: pd.DataFrame, wide_returns: pd.DataFrame, benchmark_column: str
) -> pd.Series:
    """Benchmark return when available, else the equal-weight universe return."""
    if benchmark_column in panel.columns:
        benchmark = panel.groupby("Date")[benchmark_column].first()
        if benchmark.notna().sum() > 0.5 * len(benchmark):
            return benchmark.reindex(wide_returns.index)
    return wide_returns.mean(axis=1)


def _rolling_mean_pairwise_correlation(returns: pd.DataFrame, window: int = 21) -> pd.Series:
    """Average off-diagonal correlation over a trailing window.

    Correlation spiking toward one is among the most reliable crisis markers there is:
    in a sell-off, cross-sectional structure collapses and everything moves together.
    It is free from data already loaded, and it is the feature most likely to separate a
    genuine crisis from ordinary high volatility.
    """
    values = returns.to_numpy()
    n_obs, n_assets = values.shape
    if n_assets < 2:
        return pd.Series(np.nan, index=returns.index)

    out = np.full(n_obs, np.nan)
    upper = np.triu_indices(n_assets, k=1)
    for end in range(window, n_obs + 1):
        block = values[end - window : end]
        if np.isnan(block).all():
            continue
        with np.errstate(invalid="ignore"):
            corr = np.corrcoef(np.nan_to_num(block, nan=0.0), rowvar=False)
        if corr.shape != (n_assets, n_assets):
            continue
        out[end - 1] = np.nanmean(corr[upper])
    return pd.Series(out, index=returns.index)


def standardise_causally(
    features: pd.DataFrame, train_index: Optional[pd.Index] = None
) -> pd.DataFrame:
    """Z-score using **training-window** statistics only.

    Standardising against full-sample mean and variance leaks the future into every row.
    Fitting on the training slice and applying those constants everywhere else is the
    only version compatible with the causality contract.
    """
    source = features.loc[train_index] if train_index is not None else features
    mean = source.mean()
    std = source.std().replace(0, np.nan)
    return ((features - mean) / std).fillna(0.0)


In [ ]:
# ── nifty_rl/regimes/hmm.py ─────────────────────────────────────────────────────────────
# A hidden Markov model, written out by hand. Not because the library is bad, but
# because its two most obvious methods both read the future, and the guarantee needed
# to be structural rather than a matter of remembering which function to call.
"""Gaussian hidden Markov model with a strictly causal forward filter.

Implemented in-package rather than delegated to ``hmmlearn`` for one reason: the
causality guarantee has to be structural. ``hmmlearn.predict()`` runs Viterbi over the
whole sequence and ``predict_proba()`` returns forward-backward smoothed posteriors --
both use future observations, and both are the natural methods to reach for. A regime
series built from either silently invalidates every downstream result.

Here the smoothed quantities exist only inside :meth:`_fit` (Baum-Welch needs them), and
:meth:`_filter` -- the only path used at prediction time -- runs the forward recursion
alone.
"""
from __future__ import annotations

from typing import List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


_VARIANCE_FLOOR = 1e-6

REGIME_NAMES = {
    2: ["Calm", "Stress"],
    3: ["Calm", "Normal", "Crisis"],
    4: ["Calm", "Normal", "Elevated", "Crisis"],
}


def regime_names(n_regimes: int) -> List[str]:
    if n_regimes in REGIME_NAMES:
        return list(REGIME_NAMES[n_regimes])
    return [f"Regime_{i}" for i in range(n_regimes)]


class GaussianHMMRegimes(RegimeDetector):
    """Diagonal-covariance Gaussian HMM fitted by Baum-Welch.

    Parameters
    ----------
    n_regimes:
        Number of hidden states. Start at 2; ``select_n_regimes`` picks by BIC.
    order_by:
        Index of the feature used to order states after fitting. States are sorted
        ascending, so with realised volatility first, state 0 is the calmest and the
        last state is the most turbulent. Without this, EM returns states in arbitrary
        order and a refit can silently swap what "state 0" means -- the refit-stability
        failure that makes regime results irreproducible.
    """

    name = "hmm"

    def __init__(
        self,
        n_regimes: int = 3,
        feature_columns: Optional[Sequence[str]] = None,
        order_by: int = 0,
        max_iter: int = 200,
        tol: float = 1e-4,
        random_state: int = 42,
    ):
        super().__init__(n_regimes=n_regimes, feature_columns=feature_columns)
        self.order_by = order_by
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        self.startprob_: Optional[np.ndarray] = None
        self.transmat_: Optional[np.ndarray] = None
        self.means_: Optional[np.ndarray] = None
        self.variances_: Optional[np.ndarray] = None
        self.loglikelihood_: float = float("nan")
        self.n_iter_: int = 0

    # ------------------------------------------------------------------ emissions

    def _log_emission(self, X: np.ndarray) -> np.ndarray:
        """``(n_samples, n_regimes)`` log density under each state."""
        means, variances = self.means_, self.variances_
        # (T, 1, D) - (1, K, D) -> (T, K, D)
        deviation = X[:, None, :] - means[None, :, :]
        log_density = -0.5 * (
            np.log(2.0 * np.pi * variances)[None, :, :] + deviation ** 2 / variances[None, :, :]
        )
        return log_density.sum(axis=2)

    @staticmethod
    def _scaled_emission(log_emission: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Exponentiate row-wise relative to the row max.

        The per-row constant cancels out of the normalised forward/backward quantities,
        so posteriors are unaffected; it is added back to recover the log-likelihood.
        """
        row_max = log_emission.max(axis=1, keepdims=True)
        return np.exp(log_emission - row_max), row_max.ravel()

    # -------------------------------------------------------------------- forward

    def _forward(self, emission: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Scaled forward recursion. Returns ``(alpha, scaling)``.

        ``alpha[t]`` is ``P(state_t | x_0..x_t)`` -- today's belief about which regime we
        are in, given everything seen *so far*. Rows strictly after *t* are never touched,
        which is the whole point.

        Each step is predict-then-update, exactly like a Kalman filter:

        1. ``alpha[t-1] @ transmat_`` -- carry yesterday's belief forward through the
           transition matrix, giving a prior for today before seeing today's data.
        2. ``* emission[t]`` -- multiply by how well each state explains today's
           observation, which sharpens the prior into a posterior.

        The renormalisation at each step is bookkeeping, not modelling. Raw
        probabilities are products of ~1,500 numbers below 1 and would underflow to zero
        within a few hundred days; dividing each row by its sum keeps them O(1). The
        divisors are returned because their logs sum to the log-likelihood.
        """
        n_obs, n_states = emission.shape
        alpha = np.zeros((n_obs, n_states))
        scaling = np.zeros(n_obs)

        # Day 0 has no yesterday, so the prior is the starting distribution.
        current = self.startprob_ * emission[0]
        total = current.sum()
        scaling[0] = 1.0 / max(total, 1e-300)
        alpha[0] = current * scaling[0]

        for t in range(1, n_obs):
            current = (alpha[t - 1] @ self.transmat_) * emission[t]
            total = current.sum()
            scaling[t] = 1.0 / max(total, 1e-300)
            alpha[t] = current * scaling[t]

        return alpha, scaling

    def _backward(self, emission: np.ndarray, scaling: np.ndarray) -> np.ndarray:
        """Scaled backward recursion -- **fitting only**, never at prediction time.

        The mirror image of :meth:`_forward`: ``beta[t]`` carries the evidence from days
        *after* t back to t. Multiplying the two gives a posterior informed by the whole
        series, which is what EM needs to re-estimate parameters -- and precisely what
        must not touch a live regime label, since it means "knowing how the story ended".

        Reuses the forward pass's ``scaling`` divisors so alpha and beta stay on a
        compatible scale and their product needs no further correction.
        """
        n_obs, n_states = emission.shape
        beta = np.zeros((n_obs, n_states))
        # The last day has no future evidence; seed it with the scale factor alone.
        beta[-1] = scaling[-1]
        for t in range(n_obs - 2, -1, -1):
            beta[t] = (self.transmat_ @ (emission[t + 1] * beta[t + 1])) * scaling[t]
        return beta

    # ------------------------------------------------------------------- fitting

    def _initialise(self, X: np.ndarray) -> None:
        from sklearn.mixture import GaussianMixture

        mixture = GaussianMixture(
            n_components=self.n_regimes,
            covariance_type="diag",
            random_state=self.random_state,
            n_init=5,
        ).fit(X)

        self.means_ = mixture.means_.copy()
        self.variances_ = np.maximum(mixture.covariances_.copy(), _VARIANCE_FLOOR)

        assignments = mixture.predict(X)
        self.startprob_ = np.full(self.n_regimes, 1.0 / self.n_regimes)

        # Empirical transition counts, Laplace-smoothed so no transition is impossible.
        counts = np.ones((self.n_regimes, self.n_regimes))
        for previous, current in zip(assignments[:-1], assignments[1:]):
            counts[previous, current] += 1.0
        self.transmat_ = counts / counts.sum(axis=1, keepdims=True)

    def _fit(self, X: np.ndarray) -> None:
        """Baum-Welch (EM for HMMs). Alternates two steps until the likelihood settles.

        **E-step** — given the current parameters, work out how likely each state was on
        each day. Two quantities come out of it:

        * ``gamma[t, k]`` = P(state on day *t* was *k* | the whole series). "How much does
          day *t* belong to state *k*." Rows sum to 1.
        * ``xi_sum[j, k]`` = expected number of *j* → *k* transitions across the series.

        **M-step** — re-estimate the parameters treating those soft assignments as if they
        were the observed truth. Each becomes a weighted average with ``gamma`` as the
        weights: the mean of state *k* is the ``gamma[:, k]``-weighted mean of the data.

        Each iteration cannot decrease the log-likelihood, so the loop stops once the
        improvement falls below ``tol``. Both steps use the *whole* series -- including
        days after *t* -- which is legitimate for parameter estimation on a training
        block, and is exactly why :meth:`_filter`, not this, is used at prediction time.
        """
        self._initialise(X)
        n_obs, n_features = X.shape
        previous_loglik = -np.inf

        for iteration in range(self.max_iter):
            # ---- E-step -------------------------------------------------------
            log_emission = self._log_emission(X)
            emission, row_max = self._scaled_emission(log_emission)

            alpha, scaling = self._forward(emission)  # P(state_t | days up to t)
            beta = self._backward(emission, scaling)  # P(days after t | state_t)

            # The scaling factors divided out the likelihood as we went, so summing
            # their logs recovers it; row_max adds back the emission offset.
            loglik = float(-np.log(scaling).sum() + row_max.sum())

            # Forward x backward = evidence from both directions -> the smoothed
            # posterior. Normalised per row so each day's assignment sums to 1.
            gamma = alpha * beta
            gamma /= np.maximum(gamma.sum(axis=1, keepdims=True), 1e-300)

            # Expected transition counts. For each consecutive pair of days, the joint
            # probability of (state j at t, state k at t+1) is the outer product of
            # "evidence up to t" and "evidence after t+1", weighted by how likely the
            # j -> k move was in the first place. Normalised per day, then accumulated.
            xi_sum = np.zeros((self.n_regimes, self.n_regimes))
            for t in range(n_obs - 1):
                step = (
                    np.outer(alpha[t], emission[t + 1] * beta[t + 1]) * self.transmat_
                )
                xi_sum += step / max(step.sum(), 1e-300)

            # ---- M-step -------------------------------------------------------
            # Every update below is "the soft assignments, treated as counts".

            # Starting distribution: whatever day 0 turned out to be.
            self.startprob_ = np.maximum(gamma[0], 1e-12)
            self.startprob_ /= self.startprob_.sum()

            # Transitions: expected j -> k counts, normalised into a probability per row.
            self.transmat_ = xi_sum / np.maximum(xi_sum.sum(axis=1, keepdims=True), 1e-300)

            # Emissions: gamma-weighted mean and variance of the data per state. A state
            # that owns few days gets a small `weights` entry and moves little.
            weights = np.maximum(gamma.sum(axis=0), 1e-300)
            self.means_ = (gamma.T @ X) / weights[:, None]
            deviation = X[:, None, :] - self.means_[None, :, :]
            self.variances_ = np.maximum(
                (gamma[:, :, None] * deviation ** 2).sum(axis=0) / weights[:, None],
                # A state that collapses onto near-identical days would otherwise get
                # zero variance and infinite density, swallowing the whole series.
                _VARIANCE_FLOOR,
            )

            self.n_iter_ = iteration + 1
            self.loglikelihood_ = loglik
            if abs(loglik - previous_loglik) < self.tol:
                break
            previous_loglik = loglik

        self._order_states()
        self.regime_labels_ = regime_names(self.n_regimes)

    def _order_states(self) -> None:
        """Sort states by the ordering feature so labels survive a refit."""
        order = np.argsort(self.means_[:, self.order_by])
        self.means_ = self.means_[order]
        self.variances_ = self.variances_[order]
        self.startprob_ = self.startprob_[order]
        self.transmat_ = self.transmat_[np.ix_(order, order)]

    # ----------------------------------------------------------------- prediction

    def _filter(self, X: np.ndarray) -> np.ndarray:
        """Filtered state probabilities. Forward pass only -- no backward, no Viterbi."""
        emission, _ = self._scaled_emission(self._log_emission(X))
        alpha, _ = self._forward(emission)
        return alpha

    # ---------------------------------------------------------------- diagnostics

    @property
    def n_parameters(self) -> int:
        n_states, n_features = self.means_.shape
        return (n_states - 1) + n_states * (n_states - 1) + 2 * n_states * n_features

    def bic(self, n_samples: int) -> float:
        if not np.isfinite(self.loglikelihood_):
            return float("inf")
        return float(-2.0 * self.loglikelihood_ + self.n_parameters * np.log(n_samples))

    def expected_durations(self) -> pd.Series:
        """Expected regime length in days: ``1 / (1 - a_ii)``.

        A model whose regimes last three days is untradeable after costs no matter how
        well it fits. This is the first thing to check after fitting.
        """
        diagonal = np.clip(np.diag(self.transmat_), 0.0, 1.0 - 1e-9)
        return pd.Series(
            1.0 / (1.0 - diagonal),
            index=self.regime_labels_ or [f"regime_{i}" for i in range(self.n_regimes)],
            name="expected_duration_days",
        )

    def transition_frame(self) -> pd.DataFrame:
        labels = self.regime_labels_ or [f"regime_{i}" for i in range(self.n_regimes)]
        return pd.DataFrame(self.transmat_, index=labels, columns=labels)


class MarkovSwitchingVariance(GaussianHMMRegimes):
    """Univariate switching-variance model on market returns.

    The classical Hamilton specification: one observable, states differing chiefly in
    variance. Kept as a separate backend because it is the econometric reference point,
    and because agreement between it and the multivariate HMM is evidence that the
    richer feature set is earning its keep rather than fitting noise.
    """

    name = "markov_switching"

    def __init__(
        self,
        n_regimes: int = 2,
        return_column: str = "trend_21",
        max_iter: int = 200,
        tol: float = 1e-4,
        random_state: int = 42,
    ):
        super().__init__(
            n_regimes=n_regimes,
            feature_columns=[return_column],
            order_by=0,
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )

    def _order_states(self) -> None:
        """Order by variance rather than mean -- variance is what switches here."""
        order = np.argsort(self.variances_[:, 0])
        self.means_ = self.means_[order]
        self.variances_ = self.variances_[order]
        self.startprob_ = self.startprob_[order]
        self.transmat_ = self.transmat_[np.ix_(order, order)]


def select_n_regimes(
    X: pd.DataFrame,
    candidates: Sequence[int] = (2, 3, 4),
    feature_columns: Optional[Sequence[str]] = None,
    order_by: int = 0,
    random_state: int = 42,
) -> pd.DataFrame:
    """Fit each candidate state count and score by BIC.

    With roughly 1,500 daily observations, a 4-state model on 8 features carries a lot
    of parameters. BIC is the guard against reading structure into noise -- but it is
    not the only one: a model that wins on BIC and still flips every three days should
    be rejected on persistence grounds.
    """
    rows = []
    for k in candidates:
        model = GaussianHMMRegimes(
            n_regimes=k,
            feature_columns=feature_columns,
            order_by=order_by,
            random_state=random_state,
        )
        try:
            model.fit(X)
        except Exception as exc:  # pragma: no cover - degenerate fits
            rows.append({"n_regimes": k, "loglik": np.nan, "bic": np.inf, "error": str(exc)})
            continue
        durations = model.expected_durations()
        rows.append(
            {
                "n_regimes": k,
                "loglik": model.loglikelihood_,
                "n_parameters": model.n_parameters,
                "bic": model.bic(len(X)),
                "min_expected_duration": float(durations.min()),
                "n_iter": model.n_iter_,
                "error": "",
            }
        )
    return pd.DataFrame(rows).sort_values("bic").reset_index(drop=True)


In [ ]:
# ── nifty_rl/regimes/jump.py ────────────────────────────────────────────────────────────
# Another backend, with an explicit penalty for switching too often.
"""Statistical jump model -- clustering with a switching penalty.

Nystrup et al.'s jump model adds a penalty on the *number of regime switches* to a
clustering objective, which suppresses the day-to-day flapping that makes plain
clustering untradeable.

**Deviation from the published method, stated plainly.** The original solves the
penalised objective by dynamic programming over the whole sequence, which is not causal
and therefore cannot be traded. The version here fits cluster centres on the training
window and then applies the switching penalty *online*, greedily: at each step the
incumbent regime receives a bonus, so a switch happens only when the evidence overcomes
it. That keeps the anti-flapping property while satisfying the causality contract. It is
a different estimator from the paper's and is labelled as such rather than passed off as
the original.
"""
from __future__ import annotations

from typing import Optional, Sequence

import numpy as np



class JumpModelRegimes(RegimeDetector):
    """Online centroid assignment with an explicit persistence bonus.

    Parameters
    ----------
    jump_penalty:
        Bonus, in squared-distance units, awarded to the incumbent regime. Zero reduces
        this to nearest-centroid assignment, which flips constantly; larger values buy
        persistence at the cost of detection lag. That trade-off is exactly what the
        detection-lag evaluation is for.
    """

    name = "jump"

    def __init__(
        self,
        n_regimes: int = 3,
        feature_columns: Optional[Sequence[str]] = None,
        jump_penalty: float = 2.0,
        order_by: int = 0,
        random_state: int = 42,
    ):
        super().__init__(n_regimes=n_regimes, feature_columns=feature_columns)
        self.jump_penalty = float(jump_penalty)
        self.order_by = order_by
        self.random_state = random_state
        self.centres_: Optional[np.ndarray] = None
        self.scales_: Optional[np.ndarray] = None

    def _fit(self, X: np.ndarray) -> None:
        from sklearn.mixture import GaussianMixture

        mixture = GaussianMixture(
            n_components=self.n_regimes,
            covariance_type="diag",
            random_state=self.random_state,
            n_init=5,
        ).fit(X)

        centres = mixture.means_
        scales = np.sqrt(np.maximum(mixture.covariances_, 1e-8))

        order = np.argsort(centres[:, self.order_by])
        self.centres_ = centres[order]
        self.scales_ = scales[order]
        self.regime_labels_ = regime_names(self.n_regimes)

    def _filter(self, X: np.ndarray) -> np.ndarray:
        n_obs = len(X)
        probabilities = np.zeros((n_obs, self.n_regimes))

        # Mahalanobis-ish distance under the fitted diagonal scales.
        deviation = (X[:, None, :] - self.centres_[None, :, :]) / self.scales_[None, :, :]
        distances = (deviation ** 2).sum(axis=2)

        previous = int(np.argmin(distances[0]))
        probabilities[0, previous] = 1.0

        for t in range(1, n_obs):
            score = distances[t].copy()
            score[previous] -= self.jump_penalty  # incumbent advantage
            previous = int(np.argmin(score))
            probabilities[t, previous] = 1.0

        return probabilities


In [ ]:
# ── nifty_rl/regimes/threshold.py ───────────────────────────────────────────────────────
# The simple version: split on volatility quantiles. If the HMM cannot beat this, its
# extra complexity is not earning anything.
"""Threshold and quadrant regime detectors.

The transparent control. No latent variables, no EM, no distributional assumption -- cut
points are quantiles of the *training* window and are then held fixed. If a probabilistic
model cannot beat this on out-of-sample economics, the extra machinery is not paying for
itself, and saying so is a result.
"""
from __future__ import annotations

from typing import List, Optional, Sequence

import numpy as np



class ThresholdRegimes(RegimeDetector):
    """Bucket a single feature by training-window quantiles.

    Cuts are learned once on the training slice and frozen. Using *rolling* quantiles
    instead would also be causal, but refitting the definition of "high volatility" every
    day makes regimes incomparable across time -- and makes the regime-conditional
    performance tables meaningless.
    """

    name = "threshold"

    def __init__(
        self,
        n_regimes: int = 3,
        column: str = "realized_vol_21",
        labels: Optional[Sequence[str]] = None,
    ):
        super().__init__(n_regimes=n_regimes, feature_columns=[column])
        self.column = column
        self.cut_points_: Optional[np.ndarray] = None
        self._labels = list(labels) if labels else None

    def _fit(self, X: np.ndarray) -> None:
        values = X[:, 0]
        quantiles = np.linspace(0.0, 1.0, self.n_regimes + 1)[1:-1]
        self.cut_points_ = np.quantile(values[np.isfinite(values)], quantiles)
        # Share the HMM's naming so the same regime index means the same thing in every
        # figure -- comparing detectors is impossible if one says "Crisis" and another
        # says "Q4" for the same state.
        # flattened: regime_names is defined above

        self.regime_labels_ = list(self._labels) if self._labels else regime_names(self.n_regimes)

    def _filter(self, X: np.ndarray) -> np.ndarray:
        assignments = np.digitize(X[:, 0], self.cut_points_)
        probabilities = np.zeros((len(X), self.n_regimes))
        probabilities[np.arange(len(X)), np.clip(assignments, 0, self.n_regimes - 1)] = 1.0
        return probabilities


class QuadrantRegimes(RegimeDetector):
    """Trend sign crossed with a volatility cut -- four interpretable states.

    This is the taxonomy practitioners actually reason with, and it maps directly onto
    strategy selection: momentum wants Bull-Quiet, mean-reversion wants Bear-Quiet,
    everything wants out of Bear-Volatile.
    """

    name = "quadrant"

    LABELS: List[str] = ["Bull-Quiet", "Bull-Volatile", "Bear-Quiet", "Bear-Volatile"]

    def __init__(
        self,
        trend_column: str = "trend_21",
        vol_column: str = "realized_vol_21",
        vol_quantile: float = 0.5,
    ):
        super().__init__(n_regimes=4, feature_columns=[trend_column, vol_column])
        self.trend_column = trend_column
        self.vol_column = vol_column
        self.vol_quantile = vol_quantile
        self.vol_cut_: float = float("nan")

    def _fit(self, X: np.ndarray) -> None:
        vol = X[:, 1]
        self.vol_cut_ = float(np.quantile(vol[np.isfinite(vol)], self.vol_quantile))
        self.regime_labels_ = list(self.LABELS)

    def _filter(self, X: np.ndarray) -> np.ndarray:
        bearish = (X[:, 0] < 0).astype(int)
        volatile = (X[:, 1] >= self.vol_cut_).astype(int)
        index = bearish * 2 + volatile
        probabilities = np.zeros((len(X), 4))
        probabilities[np.arange(len(X)), index] = 1.0
        return probabilities


## The reinforcement learning agent

A PPO agent that allocates across the ten stocks and cash. The simulation it trades in is written in plain NumPy and kept separate from the gymnasium wrapper, so the execution logic can be reasoned about on its own — that is where the subtle bugs live.

In [ ]:
# ── nifty_rl/envs/panel.py ──────────────────────────────────────────────────────────────
# Flattens everything into dense arrays so the simulator does no pandas work while
# stepping.
"""Dense NumPy panel for the RL environment.

The notebook stored the panel as ``dict[date] -> DataFrame`` and did a pandas ``.loc``
per ticker per step. At 250k steps with ten tickers that is ~2.5 million pandas lookups
per training run, and it dominated wall clock -- which is why the "training budget is
system-constrained" note existed at all.

Here the same data is a contiguous ``(n_dates, n_tickers, n_features)`` float32 array
addressed by integer index. Nothing clever, just the representation the inner loop
actually wants.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional, Sequence

import numpy as np
import pandas as pd


@dataclass
class PanelArrays:
    """Rectangular market panel addressed by integer index."""

    dates: pd.DatetimeIndex
    tickers: List[str]
    features: np.ndarray  # (n_dates, n_tickers, n_features) float32
    prices: np.ndarray  # (n_dates, n_tickers) float32
    available: np.ndarray  # (n_dates, n_tickers) float32, 1.0 where price is usable
    feature_names: List[str]
    regime: Optional[np.ndarray] = None  # (n_dates,) int, optional regime label

    @property
    def n_dates(self) -> int:
        return len(self.dates)

    @property
    def n_tickers(self) -> int:
        return len(self.tickers)

    @property
    def n_features(self) -> int:
        return len(self.feature_names)

    def observation_size(self) -> int:
        """Per-ticker features + availability flag, plus weights and cash."""
        return self.n_tickers * (self.n_features + 1) + (self.n_tickers + 1) + 1


def build_panel_arrays(
    df: pd.DataFrame,
    tickers: Sequence[str],
    feature_names: Sequence[str],
    regime_labels: Optional[pd.Series] = None,
) -> PanelArrays:
    """Pivot a long-format frame into dense arrays.

    Missing ``(date, ticker)`` cells get zero features and ``available = 0`` rather than
    a forward-filled value, so the agent can learn to ignore a stale name instead of
    acting on a carried-over price.
    """
    tickers = list(tickers)
    feature_names = list(feature_names)

    frame = df.copy()
    frame["Date"] = pd.to_datetime(frame["Date"])
    dates = pd.DatetimeIndex(np.sort(frame["Date"].unique()))
    date_pos = {d: i for i, d in enumerate(dates)}
    ticker_pos = {t: i for i, t in enumerate(tickers)}

    n_dates, n_tickers, n_features = len(dates), len(tickers), len(feature_names)
    features = np.zeros((n_dates, n_tickers, n_features), dtype=np.float32)
    prices = np.zeros((n_dates, n_tickers), dtype=np.float32)
    available = np.zeros((n_dates, n_tickers), dtype=np.float32)

    subset = frame[frame["ticker"].isin(ticker_pos)]
    rows = subset["Date"].map(date_pos).to_numpy()
    cols = subset["ticker"].map(ticker_pos).to_numpy()

    values = subset[feature_names].to_numpy(dtype=np.float32)
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    features[rows, cols] = values

    price_values = subset["price"].to_numpy(dtype=np.float32)
    prices[rows, cols] = np.nan_to_num(price_values, nan=0.0)
    available[rows, cols] = (np.isfinite(price_values) & (price_values > 0)).astype(np.float32)

    regime = None
    if regime_labels is not None:
        aligned = regime_labels.reindex(dates).ffill().bfill()
        regime = aligned.to_numpy(dtype=np.int64)

    return PanelArrays(
        dates=dates,
        tickers=tickers,
        features=features,
        prices=prices,
        available=available,
        feature_names=feature_names,
        regime=regime,
    )


In [ ]:
# ── nifty_rl/envs/rewards.py ────────────────────────────────────────────────────────────
# Swappable reward functions. The differential Sharpe one is the interesting one —
# risk aversion falls out of the objective instead of being bolted on with four hand-
# tuned constants.
"""Reward functions as interchangeable objects.

The notebook hard-coded one shaped reward with four hand-tuned coefficients, justified
only by "initial values caused permanent cash-hoarding; halved to allow non-trivial
policy learning". That is an anecdote, not a calibration -- and it is untestable, because
the reward could not be swapped or measured independently of the environment.

Three implementations here:

* :class:`ShapedReward` -- the notebook's formulation, reproduced exactly so old results
  stay comparable.
* :class:`DifferentialSharpeReward` -- Moody & Saffell (1998). The principled version of
  "maximise risk-adjusted return online": an exponentially-weighted estimate of the
  Sharpe ratio whose derivative gives a per-step reward, with no free penalty weights to
  tune at all.
* :class:`RegimeAwareReward` -- scales the drawdown penalty by regime instead of holding
  one constant across every market condition.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Dict, List, Mapping, Optional

import numpy as np


class RewardFunction(ABC):
    """Per-step reward. Stateful, so it must be reset between episodes."""

    name: str = "reward"

    def reset(self) -> None:
        """Clear any running state."""

    @abstractmethod
    def __call__(
        self,
        *,
        net: float,
        prev_net: float,
        drawdown: float,
        n_trades: int,
        turnover: float,
        regime: int,
        holding: bool,
    ) -> float:
        ...


def _log_return(net: float, prev_net: float) -> float:
    return float(np.log(max(net, 1e-9) / max(prev_net, 1e-9)))


class ShapedReward(RewardFunction):
    """Log return minus drawdown, downside-volatility, turnover and regime penalties.

    Reproduces the notebook's reward. Defaults are its post-calibration values.
    """

    name = "shaped"

    def __init__(
        self,
        drawdown_penalty: float = 0.04,
        downvol_penalty: float = 0.02,
        turnover_penalty: float = 0.0002,
        regime_penalty: float = 0.0002,
        downside_window: int = 20,
        penalised_regimes: Optional[List[int]] = None,
    ):
        self.drawdown_penalty = drawdown_penalty
        self.downvol_penalty = downvol_penalty
        self.turnover_penalty = turnover_penalty
        self.regime_penalty = regime_penalty
        self.downside_window = downside_window
        # Which regime indices count as "risk-off". None means the top state only,
        # resolved lazily against whatever regime labels the panel carries.
        self.penalised_regimes = penalised_regimes
        self.reset()

    def reset(self) -> None:
        self.recent: List[float] = []

    def _downside_vol(self, period_return: float) -> float:
        self.recent.append(period_return)
        window = self.recent[-self.downside_window :]
        negatives = [r for r in window if r < 0]
        return float(np.std(negatives)) if len(negatives) > 1 else 0.0

    def __call__(self, *, net, prev_net, drawdown, n_trades, turnover, regime, holding) -> float:
        period_return = (net - prev_net) / max(prev_net, 1e-9)
        down_vol = self._downside_vol(period_return)

        risk_off = regime in self.penalised_regimes if self.penalised_regimes else regime >= 2
        regime_cost = self.regime_penalty if (risk_off and holding) else 0.0

        return (
            _log_return(net, prev_net)
            - self.drawdown_penalty * abs(drawdown)
            - self.downvol_penalty * down_vol
            - self.turnover_penalty * n_trades
            - regime_cost
        )


class DifferentialSharpeReward(RewardFunction):
    """Moody & Saffell's differential Sharpe ratio.

    Maintains exponentially-weighted first and second moments of the return series and
    rewards the marginal contribution of each step to the running Sharpe estimate::

        D_t = (B_{t-1} ΔA - ½ A_{t-1} ΔB) / (B_{t-1} - A_{t-1}²)^{3/2}

    The appeal over a shaped reward is that it has no penalty weights to hand-tune: risk
    aversion falls out of the objective rather than being bolted on with four constants
    someone halved until the agent stopped hoarding cash.
    """

    name = "differential_sharpe"

    #: Below this the variance estimate is numerically indistinguishable from zero and
    #: the ratio's denominator (variance ** 1.5) explodes. A daily return standard
    #: deviation of 1e-5 is 0.001% -- no real portfolio sits under it.
    VARIANCE_FLOOR = 1e-10

    def __init__(
        self,
        adaptation_rate: float = 0.004,
        turnover_penalty: float = 0.0,
        warmup_steps: int = 20,
        clip: float = 10.0,
    ):
        self.adaptation_rate = adaptation_rate
        self.turnover_penalty = turnover_penalty
        # A Sharpe estimate from two observations is meaningless. Without a warm-up the
        # EW variance starts at exactly zero and stays near it for several steps, so the
        # first rewards are dominated by division by ~0 and pin to the clip bound --
        # which is a large *constant* signal unrelated to performance.
        self.warmup_steps = int(warmup_steps)
        self.clip = float(clip)
        self.reset()

    def reset(self) -> None:
        self.a = 0.0  # EW mean of returns
        self.b = 0.0  # EW mean of squared returns
        self.steps = 0

    def __call__(self, *, net, prev_net, drawdown, n_trades, turnover, regime, holding) -> float:
        period_return = (net - prev_net) / max(prev_net, 1e-9)
        eta = self.adaptation_rate

        # Bias correction (as in Adam). The EW accumulators start at zero, so for the
        # first ~1/eta steps they badly understate both moments -- and because the
        # variance enters as v**1.5 in the denominator, that understatement produces
        # enormous rewards. On a *constant* return series, uncorrected estimates gave a
        # cumulative reward of +103 where the correct answer is exactly 0.
        prev_bias = 1.0 - (1.0 - eta) ** self.steps if self.steps else 0.0
        if prev_bias > 0:
            a_hat = self.a / prev_bias
            b_hat = self.b / prev_bias
        else:
            a_hat = b_hat = 0.0

        delta_a = period_return - a_hat
        delta_b = period_return ** 2 - b_hat
        variance = b_hat - a_hat ** 2

        self.steps += 1
        if self.steps <= self.warmup_steps or variance <= self.VARIANCE_FLOOR:
            reward = 0.0
        else:
            reward = float((b_hat * delta_a - 0.5 * a_hat * delta_b) / (variance ** 1.5))
            reward = float(np.clip(reward, -self.clip, self.clip))

        self.a += eta * (period_return - self.a)
        self.b += eta * (period_return ** 2 - self.b)
        return reward - self.turnover_penalty * n_trades


class RegimeAwareReward(ShapedReward):
    """Shaped reward whose drawdown penalty depends on the prevailing regime.

    A single drawdown coefficient asks the agent to be equally cautious in a calm trend
    and in a crisis. Scaling it by regime is the cheapest way to express "be more
    defensive when conditions are bad" without adding another free parameter per state.
    """

    name = "regime_aware"

    def __init__(
        self,
        drawdown_penalty_by_regime: Mapping[int, float],
        default_drawdown_penalty: float = 0.04,
        **kwargs,
    ):
        super().__init__(drawdown_penalty=default_drawdown_penalty, **kwargs)
        self.drawdown_penalty_by_regime = dict(drawdown_penalty_by_regime)

    def __call__(self, *, net, prev_net, drawdown, n_trades, turnover, regime, holding) -> float:
        period_return = (net - prev_net) / max(prev_net, 1e-9)
        down_vol = self._downside_vol(period_return)

        penalty = self.drawdown_penalty_by_regime.get(int(regime), self.drawdown_penalty)
        risk_off = regime in self.penalised_regimes if self.penalised_regimes else regime >= 2
        regime_cost = self.regime_penalty if (risk_off and holding) else 0.0

        return (
            _log_return(net, prev_net)
            - penalty * abs(drawdown)
            - self.downvol_penalty * down_vol
            - self.turnover_penalty * n_trades
            - regime_cost
        )


def escalating_drawdown_penalty(n_regimes: int, base: float = 0.02, top: float = 0.10) -> Dict[int, float]:
    """Drawdown penalty rising linearly from the calmest regime to the most turbulent."""
    if n_regimes <= 1:
        return {0: base}
    return {i: float(v) for i, v in enumerate(np.linspace(base, top, n_regimes))}


REWARD_REGISTRY = {
    "shaped": ShapedReward,
    "differential_sharpe": DifferentialSharpeReward,
    "regime_aware": RegimeAwareReward,
}


In [ ]:
# ── nifty_rl/envs/core.py ───────────────────────────────────────────────────────────────
# The simulation itself. Sells settle before buys — when they were interleaved, a sale
# of the ninth stock could not fund a purchase of the first, and the agent literally
# could not execute its own decisions.
"""Portfolio simulation core — pure NumPy, no gymnasium dependency.

Deliberately separated from the Gym adapter. The execution model, the weight bookkeeping
and the reward are where the notebook's environment bugs lived, and none of them needs a
reinforcement-learning framework to be exercised. Keeping them here means they are unit
tested directly rather than through a training loop.

Fixes carried over from the notebook's ``MultiStockPPOEnv``:

* **Sells settle before buys** (bug #2). The original ran one interleaved loop over
  ``self.tickers``, so a sale of the ninth name could not fund a purchase of the first,
  and whichever tickers sat early in the tuple got systematic funding priority. Target
  weights were frequently unreachable — the agent could not execute its own policy.
* **Observed weights are realised, not intended** (bug #3). The original assigned
  ``self.weights = target_w``, so 11 of 172 observation dimensions described what the
  agent asked for rather than what it got. Combined with bug #2 the two diverged
  constantly.
* **The trade log is populated** (bug #7). ``self.trades`` was initialised and never
  appended to, so PPO's win rate and payoff ratio were NaN in every results table while
  every other strategy had them.
* **Observations are clipped to the declared box** (bug #24). ``Box(-10, 10)`` was
  declared but only ``nan_to_num`` applied, so a large scaled feature silently left the
  space it advertised.
* **The regime penalty reads today, not tomorrow** (bug #25). The original incremented
  the day index before looking up ``high_vix_regime``, penalising today's holding with
  tomorrow's regime.
* **Idle cash earns the risk-free rate**, matching the backtest engines.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional

import numpy as np



@dataclass
class StepInfo:
    """Diagnostics emitted alongside every step. Not part of the observation.

    The agent never sees these -- they exist so a trained policy can be *described*
    rather than only scored. ``gross_exposure`` and ``hhi`` are what revealed this
    project's central finding: PPO settled at 99% invested with a correlation of 0.993
    to buy-and-hold, meaning it had learned to be the benchmark while paying turnover.
    Without logging behaviour per step, that shows up only as a slightly worse number.
    """

    net_worth: float
    cash: float
    n_positions: int
    n_trades: int
    turnover: float
    gross_exposure: float
    hhi: float
    drawdown: float
    regime: int


class PortfolioSimulator:
    """Continuous-allocation portfolio simulation over a dense panel."""

    def __init__(
        self,
        panel: PanelArrays,
        cfg: BacktestConfig,
        cost_model: CostModel,
        reward_fn: Optional[RewardFunction] = None,
        max_weight: Optional[float] = None,
    ):
        self.panel = panel
        self.cfg = cfg
        self.cost_model = cost_model
        self.reward_fn = reward_fn or ShapedReward()
        self.max_weight = max_weight

        self.n = panel.n_tickers
        self.daily_cash_rate = (
            (1.0 + cfg.cash_rate_annual) ** (1.0 / cfg.trading_days) - 1.0
            if cfg.cash_rate_annual > 0
            else 0.0
        )
        self.observation_size = panel.observation_size()
        self.reset()

    # ------------------------------------------------------------------- lifecycle

    def reset(self) -> np.ndarray:
        """Return to day 0, fully in cash, and clear all history.

        The reward function is reset too. Rewards here are stateful -- the differential
        Sharpe reward carries running return moments -- so a stale estimate from the
        previous episode would leak across the boundary and score the first steps of a
        new episode against the last episode's volatility.
        """
        self.i = 0
        self.cash = float(self.cfg.initial_cash)
        self.shares = np.zeros(self.n, dtype=np.float64)
        self.net = self.prev_net = self.peak = float(self.cfg.initial_cash)
        self.weights = np.zeros(self.n + 1, dtype=np.float32)
        self.weights[-1] = 1.0
        self.trades: List[dict] = []
        self.equity_path: List[float] = [self.net]
        self.weight_path: List[np.ndarray] = [self.weights.copy()]
        self.reward_fn.reset()
        return self.observe()

    # ---------------------------------------------------------------- observations

    def observe(self) -> np.ndarray:
        """Feature block per ticker + availability flag, then weights and cash ratio."""
        features = self.panel.features[self.i]  # (n_tickers, n_features)
        available = self.panel.available[self.i].reshape(-1, 1)
        block = np.concatenate([features, available], axis=1).ravel()
        cash_ratio = np.float32(self.cash / max(self.cfg.initial_cash, 1e-9))
        obs = np.concatenate([block, self.weights, [cash_ratio]]).astype(np.float32)
        obs = np.nan_to_num(obs, nan=0.0, posinf=10.0, neginf=-10.0)
        # Declared space is Box(-10, 10); a scaled feature can exceed that on a tail
        # event, so clip rather than silently emit out-of-space observations.
        return np.clip(obs, -10.0, 10.0)

    # --------------------------------------------------------------------- helpers

    @staticmethod
    def softmax(action: np.ndarray) -> np.ndarray:
        """Turn a raw action vector into long-only weights summing to 1.

        This is how the action space stays unconstrained -- PPO emits any real numbers it
        likes and the portfolio constraints are imposed here, rather than by asking the
        optimiser to respect a simplex. The last element is the cash weight, so choosing
        to hold cash is an ordinary action rather than a special case.

        Subtracting the maximum before exponentiating changes nothing mathematically and
        prevents ``exp`` overflowing on a large action. A degenerate vector falls back to
        100% cash, which is the safe default -- never an accidental leveraged position.
        """
        a = np.asarray(action, dtype=np.float64).ravel()
        a = a - a.max()
        e = np.exp(a)
        total = e.sum()
        if not np.isfinite(total) or total <= 0:
            out = np.zeros_like(e)
            out[-1] = 1.0
            return out
        return e / total

    def portfolio_value(self, prices: np.ndarray) -> float:
        return float(max(self.cash, 0.0) + np.dot(self.shares, prices))

    def realised_weights(self, prices: np.ndarray, net: float) -> np.ndarray:
        """Weights implied by actual holdings — what the agent should observe."""
        out = np.zeros(self.n + 1, dtype=np.float32)
        if net <= 0:
            out[-1] = 1.0
            return out
        out[: self.n] = (self.shares * prices) / net
        out[-1] = max(self.cash, 0.0) / net
        return out

    def _apply_weight_cap(self, target: np.ndarray) -> np.ndarray:
        """Project onto the max-weight constraint, pushing the excess into cash."""
        if self.max_weight is None:
            return target
        capped = target.copy()
        equity_part = np.minimum(capped[: self.n], self.max_weight)
        capped[: self.n] = equity_part
        capped[-1] = max(1.0 - equity_part.sum(), 0.0)
        total = capped.sum()
        return capped / total if total > 0 else capped

    # ------------------------------------------------------------------------ step

    def step(self, action: np.ndarray):
        """Advance one trading day.

        ``action`` is a raw vector of length ``n_tickers + 1``; softmax turns it into
        target weights over the stocks plus cash, so the agent cannot request weights
        that fail to sum to 1 or go short.

        The order of operations matters and each step is here for a reason:

        1. **Convert the action to target weights** and apply the position cap.
        2. **Credit interest** on idle cash.
        3. **Work out the gap** between target and current holdings, in rupees. Tickers
           with no data today are frozen — ``delta`` is zeroed rather than traded.
        4. **Sell, then buy.** Two separate passes. Interleaving them means a sale of the
           ninth stock cannot fund a purchase of the first, so target weights become
           unreachable and whichever tickers sit early in the list get funding priority.
        5. **Buys are pro-rated** against the cash that now exists, so an over-ambitious
           target degrades proportionally instead of filling the first few names and
           starving the rest.
        6. **Advance the day, then revalue** at the next close. Trades execute at today's
           prices; the profit or loss shows up tomorrow. Revaluing at today's prices
           would let the agent bank a move it could not have known about.
        7. **Observe realised weights**, not requested ones — integer share sizing and
           partial fills mean the two differ, and the agent must see what it actually
           holds.

        Returns the gymnasium 5-tuple ``(obs, reward, terminated, truncated, info)``.
        """
        prices = self.panel.prices[self.i].astype(np.float64)
        available = self.panel.available[self.i] > 0
        regime_now = int(self.panel.regime[self.i]) if self.panel.regime is not None else 0

        target = self._apply_weight_cap(self.softmax(action))

        if self.daily_cash_rate and self.i > 0:
            self.cash *= 1.0 + self.daily_cash_rate

        value = self.portfolio_value(prices)
        target_value = target[: self.n] * value
        current_value = self.shares * prices
        delta = np.where(available, target_value - current_value, 0.0)

        n_trades = 0
        traded_value = 0.0

        # --- pass 1: sells. Must complete before any buy so proceeds are spendable.
        for k in np.flatnonzero(delta < 0):
            price = prices[k]
            if price <= 0:
                continue
            quantity = min(int(abs(delta[k]) // price), int(self.shares[k]))
            if quantity <= 0:
                continue
            fill = self.cost_model.fill_price(SELL, price, quantity)
            proceeds = quantity * fill
            self.cash += proceeds
            self.shares[k] -= quantity
            traded_value += proceeds
            n_trades += 1
            self.trades.append(
                {
                    "date": self.panel.dates[self.i],
                    "ticker": self.panel.tickers[k],
                    "side": "sell",
                    "shares": quantity,
                    "price": fill,
                    "value": proceeds,
                }
            )

        # --- pass 2: buys, pro-rated against the cash that now exists
        wanted = np.where(delta > 0, delta, 0.0)
        total_wanted = float(wanted.sum())
        budget = max(self.cash, 0.0)
        scale = min(1.0, budget / total_wanted) if total_wanted > budget > 0 else 1.0

        for k in np.flatnonzero(wanted > 0):
            price = prices[k]
            if price <= 0:
                continue
            unit = self.cost_model.fill_price(BUY, price, 1.0)
            quantity = int((wanted[k] * scale) // unit)
            if quantity <= 0:
                continue
            cost = quantity * self.cost_model.fill_price(BUY, price, quantity)
            if cost > self.cash:
                quantity = int(self.cash // unit)
                if quantity <= 0:
                    continue
                cost = quantity * self.cost_model.fill_price(BUY, price, quantity)
            self.cash = max(self.cash - cost, 0.0)
            self.shares[k] += quantity
            traded_value += cost
            n_trades += 1
            self.trades.append(
                {
                    "date": self.panel.dates[self.i],
                    "ticker": self.panel.tickers[k],
                    "side": "buy",
                    "shares": quantity,
                    "price": cost / quantity,
                    "value": cost,
                }
            )

        # --- advance one day and revalue at tomorrow's prices (no lookahead: the trade
        #     used today's prices, the P&L is realised at the next close)
        self.i += 1
        terminated = self.i >= self.panel.n_dates - 1
        next_prices = self.panel.prices[min(self.i, self.panel.n_dates - 1)].astype(np.float64)

        self.net = self.portfolio_value(next_prices)
        self.peak = max(self.peak, self.net)
        drawdown = (self.net - self.peak) / max(self.peak, 1e-9)

        # Realised weights, not the requested ones.
        self.weights = self.realised_weights(next_prices, self.net)
        holding_equity = bool(np.any(self.shares > 0))

        reward = self.reward_fn(
            net=self.net,
            prev_net=self.prev_net,
            drawdown=drawdown,
            n_trades=n_trades,
            turnover=traded_value / max(value, 1e-9),
            regime=regime_now,  # the regime in force when the decision was made
            holding=holding_equity,
        )

        self.prev_net = self.net
        self.equity_path.append(self.net)
        self.weight_path.append(self.weights.copy())

        gross = float(self.weights[: self.n].sum())
        info = StepInfo(
            net_worth=self.net,
            cash=self.cash,
            n_positions=int(np.count_nonzero(self.shares > 0)),
            n_trades=n_trades,
            turnover=traded_value / max(value, 1e-9),
            gross_exposure=gross,
            hhi=float(np.sum(self.weights[: self.n] ** 2)),
            drawdown=drawdown,
            regime=regime_now,
        )
        return self.observe(), float(reward), bool(terminated), False, info


In [ ]:
# ── nifty_rl/envs/multistock.py ─────────────────────────────────────────────────────────
# The gymnasium wrapper. Deliberately thin; all the logic is above.
"""Gymnasium adapter over :class:`~nifty_rl.envs.core.PortfolioSimulator`.

Thin by design. All execution logic, weight bookkeeping and reward computation live in
the core, which has no reinforcement-learning dependency and is unit tested directly.
This file only translates between that core and the Gym API.

``gymnasium`` is imported at module level, but the core is importable without it -- so
the portfolio mechanics remain testable in an environment with no RL stack installed.
"""
from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces



class MultiStockPortfolioEnv(gym.Env):
    """Continuous long-only allocation across N tickers plus cash."""

    metadata = {"render_modes": []}

    def __init__(
        self,
        panel: PanelArrays,
        cfg: BacktestConfig,
        cost_cfg: CostConfig,
        reward_fn: Optional[RewardFunction] = None,
        cost_model: Optional[CostModel] = None,
        max_weight: Optional[float] = None,
    ):
        super().__init__()
        self.simulator = PortfolioSimulator(
            panel=panel,
            cfg=cfg,
            cost_model=cost_model or build_cost_model(cost_cfg),
            reward_fn=reward_fn,
            max_weight=max_weight,
        )
        n = panel.n_tickers
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(panel.observation_size(),), dtype=np.float32
        )
        # One logit per ticker plus cash; the simulator applies the softmax.
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(n + 1,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        return self.simulator.reset(), {}

    def step(self, action):
        obs, reward, terminated, truncated, info = self.simulator.step(action)
        return obs, reward, terminated, truncated, info.__dict__

    # ---------------------------------------------------------------- convenience

    @classmethod
    def from_frame(
        cls,
        df: pd.DataFrame,
        tickers: Sequence[str],
        feature_names: Sequence[str],
        cfg: BacktestConfig,
        cost_cfg: CostConfig,
        regime_labels: Optional[pd.Series] = None,
        reward_fn: Optional[RewardFunction] = None,
        max_weight: Optional[float] = None,
    ) -> "MultiStockPortfolioEnv":
        panel = build_panel_arrays(df, tickers, feature_names, regime_labels)
        return cls(panel, cfg, cost_cfg, reward_fn=reward_fn, max_weight=max_weight)

    @property
    def equity_curve(self) -> pd.Series:
        sim = self.simulator
        dates = sim.panel.dates[: len(sim.equity_path)]
        return pd.Series(sim.equity_path, index=dates, name="equity")

    @property
    def weights_frame(self) -> pd.DataFrame:
        """Realised weights per step — the allocation stack the notebook could not plot.

        Its environment never logged weights, so the published "allocation stack" chart
        was an inline momentum proxy rather than the agent's actual positions.
        """
        sim = self.simulator
        dates = sim.panel.dates[: len(sim.weight_path)]
        return pd.DataFrame(
            np.vstack(sim.weight_path),
            index=dates,
            columns=list(sim.panel.tickers) + ["CASH"],
        )

    @property
    def trades_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.simulator.trades)


In [ ]:
# ── nifty_rl/agents/train.py ────────────────────────────────────────────────────────────
# Trains PPO, keeps the best checkpoint by Sharpe on data it has not trained on, and
# runs several seeds — because a single seed is an anecdote, not a result.
"""PPO training with a consistent selection protocol and multi-seed evaluation.

Two methodology fixes carried over from the notebook:

* **Selection and deployment use the same rule** (bug #12). The notebook's hyperparameter
  search called ``train_ppo_agent`` *without* validation data, so ``cb = None`` and each
  candidate was scored on its **final iterate**; the final model was then trained *with*
  the checkpoint callback and scored on its **best checkpoint**. Candidates were chosen
  under one protocol and deployed under another. Here the checkpoint callback is always
  active, so every number compared is the same kind of number.

* **Every result is a seed distribution, not a point** (the notebook's stated limitation).
  Policy-gradient variance on ~1,000-step episodes is large enough that a single seed says
  very little. :func:`train_ppo_ensemble` trains N seeds per window and reports the spread
  alongside the mean, plus an equal-weight action ensemble.

The environment is the vectorised :class:`~nifty_rl.envs.core.PortfolioSimulator` (≈8,000
steps/sec), which is what makes a seed sweep across every walk-forward window affordable
at all -- the notebook's pandas-per-step panel is the reason its training budget was
described as "system-constrained".
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd


PPO_DEFAULTS: Dict[str, object] = {
    "learning_rate": 3e-4,
    "n_steps": 512,
    "batch_size": 128,
    "n_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
    "ent_coef": 0.005,
    "vf_coef": 0.5,
    "max_grad_norm": 0.5,
}


def _make_env(panel, cfg, cost_cfg, reward_fn, max_weight):
    # flattened: MultiStockPortfolioEnv is defined above

    return MultiStockPortfolioEnv(
        panel, cfg, cost_cfg, reward_fn=reward_fn, max_weight=max_weight
    )


def _sharpe(equity: pd.Series, trading_days: int = 252) -> float:
    returns = equity.pct_change().dropna()
    if len(returns) < 2:
        return float("nan")
    sd = returns.std()
    if not np.isfinite(sd) or sd <= 1e-10:
        return float("nan")
    return float(returns.mean() / sd * np.sqrt(trading_days))


def evaluate_policy_on_panel(
    model,
    panel: PanelArrays,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    strategy_name: str = "PPO",
    reward_fn: Optional[RewardFunction] = None,
    max_weight: Optional[float] = None,
    deterministic: bool = True,
) -> BacktestResult:
    """Roll a trained policy over a panel and return a standard backtest result."""
    env = _make_env(panel, cfg, cost_cfg, reward_fn or ShapedReward(), max_weight)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    weights = env.weights_frame
    equity = env.equity_curve
    positions = pd.Series(
        (weights[panel.tickers] > 1e-6).sum(axis=1), index=weights.index, name="position"
    )
    per_ticker = {t: (weights[t] > 1e-6).astype(int) for t in panel.tickers}

    return BacktestResult(
        strategy=strategy_name,
        equity=equity,
        positions=positions,
        trades=env.trades_frame,
        per_ticker_positions=per_ticker,
        weights=weights[panel.tickers],
    )


class SharpeCheckpoint:
    """Keeps the best policy by validation Sharpe and restores it after training.

    PPO can peak mid-training and regress; the deployed model should be the best policy
    seen, not the last one. Built as a plain object with a thin SB3 callback wrapper so
    the selection logic is testable without a training loop.
    """

    def __init__(self, val_panel, cfg, cost_cfg, check_freq=10_000, reward_fn=None, max_weight=None):
        self.val_panel = val_panel
        self.cfg = cfg
        self.cost_cfg = cost_cfg
        self.check_freq = int(check_freq)
        self.reward_fn = reward_fn
        self.max_weight = max_weight
        self.best_sharpe = -np.inf
        self.best_state: Optional[dict] = None
        self.history: List[Dict[str, float]] = []

    def consider(self, model, step: int) -> bool:
        result = evaluate_policy_on_panel(
            model, self.val_panel, self.cfg, self.cost_cfg,
            reward_fn=self.reward_fn, max_weight=self.max_weight,
        )
        sharpe = _sharpe(result.equity)
        # bool(), not the raw numpy scalar: np.isfinite returns np.bool_, and `and`
        # propagates it, so callers doing `is True` silently get the wrong answer.
        improved = bool(np.isfinite(sharpe) and sharpe > self.best_sharpe)
        if improved:
            self.best_sharpe = sharpe
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.policy.state_dict().items()}
        self.history.append({"step": step, "val_sharpe": sharpe, "improved": improved})
        return improved

    def restore(self, model):
        if self.best_state is not None:
            model.policy.load_state_dict(self.best_state)
        return model

    def as_sb3_callback(self):
        from stable_baselines3.common.callbacks import BaseCallback

        outer = self

        class _Callback(BaseCallback):
            def _on_step(self) -> bool:
                if self.n_calls % outer.check_freq == 0:
                    outer.consider(self.model, self.n_calls)
                return True

        return _Callback()


def train_ppo(
    train_panel: PanelArrays,
    val_panel: PanelArrays,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    timesteps: int = 60_000,
    seed: int = 42,
    params: Optional[Dict[str, object]] = None,
    reward_fn: Optional[RewardFunction] = None,
    max_weight: Optional[float] = None,
    check_freq: int = 10_000,
    verbose: int = 0,
):
    """Train one PPO policy, checkpointing on validation Sharpe throughout."""
    from stable_baselines3 import PPO
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.vec_env import DummyVecEnv

    reward_fn = reward_fn or ShapedReward()
    settings = dict(PPO_DEFAULTS)
    settings.update(params or {})

    def _factory():
        return Monitor(_make_env(train_panel, cfg, cost_cfg, reward_fn, max_weight))

    env = DummyVecEnv([_factory])
    model = PPO(
        "MlpPolicy", env, seed=seed, device="cpu", verbose=verbose,
        policy_kwargs=dict(net_arch=[64, 64]), **settings,
    )

    checkpoint = SharpeCheckpoint(
        val_panel, cfg, cost_cfg, check_freq=check_freq,
        reward_fn=reward_fn, max_weight=max_weight,
    )
    model.learn(total_timesteps=timesteps, callback=checkpoint.as_sb3_callback(), progress_bar=False)
    # Always restore -- selection and deployment must use the same rule (bug #12).
    checkpoint.restore(model)
    return model, checkpoint


@dataclass
class EnsembleResult:
    per_seed: Dict[int, BacktestResult]
    val_sharpes: Dict[int, float]

    @property
    def seed_returns(self) -> Dict[int, pd.Series]:
        return {s: r.equity.pct_change().dropna() for s, r in self.per_seed.items()}

    def summary(self, initial_cash: float) -> Dict[str, float]:
        totals = [r.equity.iloc[-1] / initial_cash - 1.0 for r in self.per_seed.values()]
        sharpes = [_sharpe(r.equity) for r in self.per_seed.values()]
        return {
            "n_seeds": len(self.per_seed),
            "mean_return": float(np.mean(totals)),
            "std_return": float(np.std(totals)),
            "min_return": float(np.min(totals)),
            "max_return": float(np.max(totals)),
            "mean_sharpe": float(np.nanmean(sharpes)),
            "std_sharpe": float(np.nanstd(sharpes)),
        }

    def mean_equity(self, initial_cash: float) -> pd.Series:
        """Equal-weight ensemble across seeds, rebased to the starting capital.

        Averaging the *equity paths* is the portfolio interpretation: split capital
        equally across N independently trained policies. It is not the same as averaging
        their actions, and it is the version an allocator could actually run.
        """
        frame = pd.DataFrame({s: r.equity for s, r in self.per_seed.items()}).dropna()
        return frame.mean(axis=1).rename("PPO_ensemble")


def train_ppo_ensemble(
    train_panel: PanelArrays,
    val_panel: PanelArrays,
    test_panel: PanelArrays,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    seeds: Sequence[int] = (0, 1, 2),
    timesteps: int = 60_000,
    params: Optional[Dict[str, object]] = None,
    reward_fn: Optional[RewardFunction] = None,
    max_weight: Optional[float] = None,
    progress=None,
) -> EnsembleResult:
    """Train one policy per seed and evaluate each on the held-out panel."""
    log = progress or (lambda _m: None)
    per_seed: Dict[int, BacktestResult] = {}
    val_sharpes: Dict[int, float] = {}

    for seed in seeds:
        model, checkpoint = train_ppo(
            train_panel, val_panel, cfg, cost_cfg,
            timesteps=timesteps, seed=seed, params=params,
            reward_fn=reward_fn, max_weight=max_weight,
        )
        result = evaluate_policy_on_panel(
            model, test_panel, cfg, cost_cfg,
            strategy_name=f"PPO_s{seed}", reward_fn=reward_fn, max_weight=max_weight,
        )
        per_seed[seed] = result
        val_sharpes[seed] = checkpoint.best_sharpe
        log(
            f"        seed {seed}: val_sharpe {checkpoint.best_sharpe:>6.2f}  "
            f"oos_return {result.equity.iloc[-1] / cfg.initial_cash - 1:>7.2%}"
        )

    return EnsembleResult(per_seed=per_seed, val_sharpes=val_sharpes)


## Measuring the results

Two different questions, deliberately kept apart. The first file answers *what happened* — returns, Sharpe, drawdown, how much of the time you were actually invested. The second answers *how much of it to believe*, which is the harder and more important one when you have tried this many strategies.

In [ ]:
# ── nifty_rl/metrics/performance.py ─────────────────────────────────────────────────────
# What happened: returns, Sharpe, Sortino, drawdown, and how much of the time the
# money was actually invested.
"""Performance metrics.

Fixes carried over from the notebook:

* **Information ratio is annualised exactly once** (bug #9). The notebook computed::

      te = (sr - br).std() * np.sqrt(252)
      ir = (sr - br).mean() / te * np.sqrt(252)

  The two roots cancel, leaving a *daily* mean/std ratio. Every published IR was
  therefore about 15.9x too small.

* **Signal accuracy is genuinely per-ticker** (bug #8). The notebook reindexed the
  *aggregate* 0..N position count against each individual ticker's forward return, so
  ``pos > 0`` meant "invested in anything at all". That is why every strategy landed in
  a 44-47% band -- the metric was measuring portfolio participation, not signal quality.

* **Risk-free rate is applied** (bug #10 companion). Sharpe and Sortino in the notebook
  used ``mean/std``, i.e. rf = 0. At an Indian risk-free near 6.5% that materially
  flatters fully-invested strategies relative to selective ones.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Dict

import numpy as np
import pandas as pd



def _daily_rf(cfg: MetricsConfig) -> float:
    if cfg.risk_free_annual <= 0:
        return 0.0
    return (1.0 + cfg.risk_free_annual) ** (1.0 / cfg.trading_days) - 1.0


def _clean_equity(equity: pd.Series) -> pd.Series:
    """Collapse duplicate timestamps and coerce to float."""
    return equity.groupby(level=0).last().astype(float).sort_index()


# ------------------------------------------------------------------ risk measures


# A strategy parked in cash earns the risk-free rate exactly, so its excess return is
# zero up to floating-point noise. Dividing by that noise produces Sharpe ratios in the
# trillions. Any dispersion below this floor means "no risk was taken", and the honest
# answer is undefined rather than enormous.
_DEGENERATE_VOL = 1e-10


def sharpe_ratio(returns: pd.Series, cfg: MetricsConfig) -> float:
    """Annualised Sharpe of excess-over-cash returns.

    Returns NaN for a portfolio that never left cash: with zero excess return and zero
    excess volatility the ratio is 0/0, and reporting a number there would rank a
    do-nothing strategy at the top of the leaderboard.
    """
    if len(returns) < 2:
        return float("nan")
    excess = returns - _daily_rf(cfg)
    sd = excess.std()
    if not np.isfinite(sd) or sd <= _DEGENERATE_VOL:
        return float("nan")
    return float(excess.mean() / sd * np.sqrt(cfg.trading_days))


def sortino_ratio(returns: pd.Series, cfg: MetricsConfig) -> float:
    """Annualised Sortino: downside deviation measured against the risk-free rate."""
    if len(returns) < 2:
        return float("nan")
    rf = _daily_rf(cfg)
    excess = returns - rf
    if not np.isfinite(excess.std()) or excess.std() <= _DEGENERATE_VOL:
        return float("nan")
    downside = excess[excess < 0]
    if len(downside) < 2:
        return float("nan")
    dd = np.sqrt((downside ** 2).mean())
    if dd <= _DEGENERATE_VOL or not np.isfinite(dd):
        return float("nan")
    return float(excess.mean() / dd * np.sqrt(cfg.trading_days))


def information_ratio(
    strategy_returns: pd.Series, benchmark_returns: pd.Series, cfg: MetricsConfig
) -> float:
    """Annualised IR = mean(active) / stdev(active) * sqrt(252).

    Annualised once. The notebook multiplied by sqrt(252) after already folding it into
    the tracking-error denominator, so the factors cancelled (bug #9).
    """
    aligned = pd.concat(
        [strategy_returns.rename("s"), benchmark_returns.rename("b")], axis=1, join="inner"
    ).dropna()
    if len(aligned) < 2:
        return float("nan")
    active = aligned["s"] - aligned["b"]
    sd = active.std()
    if sd <= 0 or not np.isfinite(sd):
        return float("nan")
    return float(active.mean() / sd * np.sqrt(cfg.trading_days))


def max_drawdown(equity: pd.Series) -> float:
    if equity.empty:
        return float("nan")
    return float((equity / equity.cummax() - 1.0).min())


def ulcer_index(equity: pd.Series) -> float:
    """RMS drawdown -- penalises depth *and* duration, unlike max drawdown."""
    if equity.empty:
        return float("nan")
    dd = equity / equity.cummax() - 1.0
    return float(np.sqrt((dd ** 2).mean()))


def omega_ratio(returns: pd.Series, threshold: float = 0.0) -> float:
    """Probability-weighted gains over losses relative to a threshold."""
    if returns.empty:
        return float("nan")
    gains = (returns - threshold).clip(lower=0).sum()
    losses = (threshold - returns).clip(lower=0).sum()
    if losses <= 0:
        return float("inf") if gains > 0 else float("nan")
    return float(gains / losses)


def tail_ratio(returns: pd.Series, quantile: float = 0.05) -> float:
    """|95th percentile| / |5th percentile| -- upside tail versus downside tail."""
    if len(returns) < 20:
        return float("nan")
    upper = abs(returns.quantile(1 - quantile))
    lower = abs(returns.quantile(quantile))
    if lower <= 0:
        return float("nan")
    return float(upper / lower)


def longest_drawdown_days(equity: pd.Series) -> int:
    """Longest run of consecutive observations spent below a prior peak."""
    if equity.empty:
        return 0
    underwater = (equity < equity.cummax()).to_numpy()
    longest = current = 0
    for flag in underwater:
        current = current + 1 if flag else 0
        longest = max(longest, current)
    return int(longest)


def calmar_ratio(cagr: float, mdd: float) -> float:
    if not np.isfinite(cagr) or not np.isfinite(mdd) or mdd >= 0:
        return float("nan")
    return float(cagr / abs(mdd))


# --------------------------------------------------------------- signal accuracy


def signal_accuracy_per_ticker(
    per_ticker_positions: Dict[str, pd.Series], panel: pd.DataFrame
) -> float:
    """Fraction of invested days on which *that ticker* rose, equal-weighted.

    Each ticker contributes one accuracy figure regardless of its price level, then those
    are averaged. The notebook instead compared the aggregate portfolio position count
    against each ticker's return (bug #8).
    """
    accuracies = []
    for ticker, positions in per_ticker_positions.items():
        ticker_df = panel[panel["ticker"] == ticker]
        if ticker_df.empty:
            continue

        prices = ticker_df.set_index(pd.to_datetime(ticker_df["Date"]))["price"]
        prices = prices.groupby(level=0).last().sort_index()
        forward = prices.pct_change().shift(-1)

        aligned = pd.concat(
            [positions.rename("pos"), forward.rename("fwd")], axis=1, join="inner"
        ).dropna()
        invested = aligned[aligned["pos"] > 0]
        if not invested.empty:
            accuracies.append(float((invested["fwd"] > 0).mean()))

    return float(np.mean(accuracies)) if accuracies else float("nan")


# ------------------------------------------------------------------- aggregation


@dataclass
class PerformanceMetrics:
    """One strategy's descriptive record over one evaluation window.

    Deliberately wide. A single headline number invites cherry-picking, and these
    measure genuinely different things -- ``Sharpe`` penalises upside volatility while
    ``Sortino`` does not, ``max_drawdown`` records the worst moment while ``ulcer_index``
    records how long the pain lasted, and ``exposure`` is what reveals that a
    flattering risk-adjusted figure came from sitting in cash.

    These are all *descriptive*: they summarise what happened. Whether any of it is
    distinguishable from luck is a separate question, answered in
    :mod:`nifty_rl.metrics.stats`.
    """

    strategy: str
    final_equity: float
    total_return: float
    benchmark_excess_return: float
    CAGR: float
    alpha_annual: float
    annual_volatility: float
    Sharpe: float
    Sortino: float
    Calmar: float
    max_drawdown: float
    ulcer_index: float
    longest_drawdown_days: int
    information_ratio: float
    omega_ratio: float
    tail_ratio: float
    daily_VaR_95: float
    daily_CVaR_95: float
    trades: int
    win_rate: float
    payoff_ratio: float
    signal_accuracy: float
    exposure: float

    def to_dict(self) -> dict:
        return asdict(self)


def performance_metrics(
    result,
    benchmark,
    panel: pd.DataFrame,
    cfg: MetricsConfig,
    initial_cash: float,
) -> PerformanceMetrics:
    """Compute the full metric set for one backtest result.

    ``result`` and ``benchmark`` are :class:`~nifty_rl.backtest.engine.BacktestResult`.
    """
    equity = _clean_equity(result.equity)
    bench_equity = _clean_equity(benchmark.equity)

    returns = equity.pct_change().dropna()
    bench_returns = bench_equity.pct_change().dropna()

    years = max(len(equity) / cfg.trading_days, 1.0 / cfg.trading_days)
    total_return = equity.iloc[-1] / initial_cash - 1.0
    bench_total = bench_equity.iloc[-1] / initial_cash - 1.0
    cagr = (equity.iloc[-1] / initial_cash) ** (1.0 / years) - 1.0
    bench_cagr = (bench_equity.iloc[-1] / initial_cash) ** (1.0 / years) - 1.0

    mdd = max_drawdown(equity)
    var95 = float(returns.quantile(cfg.var_quantile)) if len(returns) else float("nan")
    tail = returns[returns <= var95]
    cvar95 = float(tail.mean()) if len(tail) else float("nan")

    # Trade count is always meaningful; win rate and payoff need round-trip P&L, which
    # only the signal engine produces. The weight engine logs individual fills, where a
    # "win" is undefined -- reporting NaN there is correct, but the count still counts.
    trades = result.trades
    n_trades = int(len(trades)) if trades is not None and not trades.empty else 0
    if n_trades and "pnl" in trades.columns:
        win_rate = float((trades["pnl"] > 0).mean())
        wins = trades[trades["pnl"] > 0]["pnl"]
        losses = trades[trades["pnl"] < 0]["pnl"]
        payoff = float(-wins.mean() / losses.mean()) if len(losses) and len(wins) else float("nan")
    else:
        win_rate, payoff = float("nan"), float("nan")

    if result.per_ticker_positions:
        accuracy = signal_accuracy_per_ticker(result.per_ticker_positions, panel)
        n_names = max(len(result.per_ticker_positions), 1)
        exposure = float(result.positions.mean() / n_names) if len(result.positions) else float("nan")
    else:
        accuracy = float("nan")
        exposure = float("nan")

    return PerformanceMetrics(
        strategy=result.strategy,
        final_equity=float(equity.iloc[-1]),
        total_return=float(total_return),
        benchmark_excess_return=float(total_return - bench_total),
        CAGR=float(cagr),
        alpha_annual=float(cagr - bench_cagr),
        annual_volatility=float(returns.std() * np.sqrt(cfg.trading_days)) if len(returns) > 1 else float("nan"),
        Sharpe=sharpe_ratio(returns, cfg),
        Sortino=sortino_ratio(returns, cfg),
        Calmar=calmar_ratio(cagr, mdd),
        max_drawdown=mdd,
        ulcer_index=ulcer_index(equity),
        longest_drawdown_days=longest_drawdown_days(equity),
        information_ratio=information_ratio(returns, bench_returns, cfg),
        omega_ratio=omega_ratio(returns),
        tail_ratio=tail_ratio(returns, cfg.var_quantile),
        daily_VaR_95=var95,
        daily_CVaR_95=cvar95,
        trades=n_trades,
        win_rate=win_rate,
        payoff_ratio=payoff,
        signal_accuracy=accuracy,
        exposure=exposure,
    )


def metrics_frame(metrics_list, sort_by: str = "Sharpe") -> pd.DataFrame:
    """Assemble a leaderboard, de-duplicating identical strategies.

    The notebook ran ``MA_20_50`` from both ``FIXED_STRATS`` and the validation selector,
    so the leaderboard and both bar charts carried two identical rows (bug #26).
    """
    frame = pd.DataFrame([m.to_dict() for m in metrics_list])
    if frame.empty:
        return frame
    frame = frame.drop_duplicates(subset=["strategy"], keep="first")
    if sort_by in frame.columns:
        frame = frame.sort_values(sort_by, ascending=False)
    return frame.reset_index(drop=True)


In [ ]:
# ── nifty_rl/metrics/stats.py ───────────────────────────────────────────────────────────
# How much to believe it. Confidence intervals, a correction for having tried many
# strategies, and a test for whether the selection itself is just noise.
"""Inferential statistics for backtests.

A backtest reports a point estimate from one sample of one history. These are the tools
that say how much of it to believe.

The search space in this project is large: SL/TP grids, PPO hyperparameter candidates,
validation-selected rule parameters, feature-set ablations and five regime backends. The
best observed Sharpe is therefore an *order statistic*, and comparing it to zero is the
wrong test. Deflated Sharpe Ratio corrects for exactly that, and it is the first thing a
quantitative reader will look for.

References
----------
Bailey & Lopez de Prado (2014), "The Deflated Sharpe Ratio".
Bailey, Borwein, Lopez de Prado & Zhu (2017), "The Probability of Backtest Overfitting".
Politis & Romano (1994), "The Stationary Bootstrap".
White (2000), "A Reality Check for Data Snooping".
"""
from __future__ import annotations

from itertools import combinations
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
from scipy import stats

EULER_MASCHERONI = 0.5772156649015329


# ------------------------------------------------------------------ Sharpe tests


# See metrics.performance._DEGENERATE_VOL -- a series with no dispersion has an
# undefined Sharpe, not an enormous one.
_DEGENERATE_VOL = 1e-10


def sharpe_from_returns(
    returns: np.ndarray, trading_days: int = 252, risk_free_daily: float = 0.0
) -> float:
    """Annualised Sharpe ratio from a daily return array.

    The plain-array counterpart of :func:`metrics.performance.sharpe_ratio`, kept
    separate because the bootstrap calls it a few million times and pandas overhead
    would dominate.

    Returns NaN, not a large number, when the series has no dispersion. An all-cash
    portfolio earning a constant daily rate has an *undefined* Sharpe; computing it
    anyway produced a headline figure of 2.1e13 in an earlier run.
    """
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 2:
        return float("nan")
    excess = r - risk_free_daily
    sd = np.std(excess, ddof=1)
    if not np.isfinite(sd) or sd <= _DEGENERATE_VOL:
        return float("nan")
    return float(np.mean(excess) / sd * np.sqrt(trading_days))


def probabilistic_sharpe_ratio(
    returns: Sequence[float],
    benchmark_sharpe: float = 0.0,
    trading_days: int = 252,
) -> float:
    """P(true Sharpe > benchmark), correcting for skew and kurtosis.

    Daily returns are neither normal nor independent. Negative skew and fat tails both
    inflate the naive Sharpe's apparent precision, and this adjusts for both.
    """
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]
    n = len(r)
    if n < 3 or np.std(r, ddof=1) == 0:
        return float("nan")

    sharpe_daily = np.mean(r) / np.std(r, ddof=1)
    benchmark_daily = benchmark_sharpe / np.sqrt(trading_days)
    skew = float(stats.skew(r))
    kurtosis = float(stats.kurtosis(r, fisher=False))

    denominator = 1.0 - skew * sharpe_daily + ((kurtosis - 1.0) / 4.0) * sharpe_daily ** 2
    if denominator <= 0:
        return float("nan")

    z = (sharpe_daily - benchmark_daily) * np.sqrt(n - 1) / np.sqrt(denominator)
    return float(stats.norm.cdf(z))


def expected_maximum_sharpe(n_trials: int, sharpe_variance: float) -> float:
    """Expected maximum Sharpe under the null that every trial has true Sharpe zero.

    This is the bar the *best* strategy must clear. Testing the winner of 50 trials
    against zero instead of against this is the core data-snooping error.

    Try 50 worthless strategies and the luckiest will still post a respectable Sharpe --
    that is arithmetic, not skill. The formula is the standard extreme-value
    approximation to the expected maximum of ``n_trials`` normal draws, so the bar rises
    with both the number of attempts and how much the attempts disagree with each other.
    """
    if n_trials < 2 or sharpe_variance <= 0:
        return 0.0
    sd = np.sqrt(sharpe_variance)
    # Two quantiles of the standard normal, blended by the Euler-Mascheroni constant.
    # This weighting is what makes the approximation accurate for finite n_trials.
    a = stats.norm.ppf(1.0 - 1.0 / n_trials)
    b = stats.norm.ppf(1.0 - 1.0 / (n_trials * np.e))
    return float(sd * ((1.0 - EULER_MASCHERONI) * a + EULER_MASCHERONI * b))


def deflated_sharpe_ratio(
    returns: Sequence[float],
    n_trials: int,
    trial_sharpes: Optional[Sequence[float]] = None,
    trading_days: int = 252,
) -> Dict[str, float]:
    """Probability the observed Sharpe survives correction for the number of trials.

    ``trial_sharpes`` should be the annualised Sharpes of *every* configuration tried,
    including the losers -- their dispersion is what sets the deflation. Passing only the
    winners understates the correction.
    """
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 3:
        return {"sharpe": float("nan"), "dsr": float("nan"), "threshold": float("nan")}

    observed = sharpe_from_returns(r, trading_days)

    if trial_sharpes is not None and len(trial_sharpes) > 1:
        variance = float(np.nanvar(np.asarray(trial_sharpes, dtype=float), ddof=1))
    else:
        variance = 1.0 / len(r) * trading_days  # crude fallback under the null

    threshold = expected_maximum_sharpe(max(n_trials, 2), variance)
    dsr = probabilistic_sharpe_ratio(r, benchmark_sharpe=threshold, trading_days=trading_days)

    return {
        "sharpe": observed,
        "n_trials": float(n_trials),
        "trial_sharpe_variance": variance,
        "deflation_threshold": threshold,
        "dsr": dsr,
        "psr_vs_zero": probabilistic_sharpe_ratio(r, 0.0, trading_days),
    }


# ---------------------------------------------------------------------- bootstrap


def stationary_bootstrap_indices(
    n_obs: int, mean_block: float, rng: np.random.Generator
) -> np.ndarray:
    """Politis-Romano stationary bootstrap indices with geometric block lengths.

    An IID bootstrap on daily returns destroys the autocorrelation and volatility
    clustering that drive drawdowns, so its confidence intervals come out far too tight.

    Instead of shuffling single days, this walks forward through the original series in
    contiguous runs, jumping to a random new day with probability ``1 / mean_block``.
    Runs therefore have geometric lengths averaging ``mean_block`` (21 days here, about a
    month), which preserves the local structure -- a volatile stretch resamples as a
    volatile stretch. Randomising the run length rather than fixing it is what keeps the
    resampled series stationary, hence the name.
    """
    p = 1.0 / max(mean_block, 1.0)
    indices = np.empty(n_obs, dtype=int)
    current = rng.integers(0, n_obs)
    for t in range(n_obs):
        indices[t] = current
        if rng.random() < p:
            current = rng.integers(0, n_obs)  # start a new block
        else:
            current = (current + 1) % n_obs  # continue this one; wrap at the end
    return indices


def bootstrap_metric_ci(
    returns: Sequence[float],
    statistic=sharpe_from_returns,
    n_boot: int = 2000,
    mean_block: float = 21.0,
    alpha: float = 0.05,
    seed: int = 42,
) -> Dict[str, float]:
    """Percentile confidence interval for any return-based statistic."""
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 30:
        return {"point": float("nan"), "lower": float("nan"), "upper": float("nan")}

    rng = np.random.default_rng(seed)
    draws = np.empty(n_boot)
    for b in range(n_boot):
        draws[b] = statistic(r[stationary_bootstrap_indices(len(r), mean_block, rng)])

    draws = draws[np.isfinite(draws)]
    if len(draws) == 0:
        return {"point": float("nan"), "lower": float("nan"), "upper": float("nan")}

    return {
        "point": float(statistic(r)),
        "lower": float(np.percentile(draws, 100 * alpha / 2)),
        "upper": float(np.percentile(draws, 100 * (1 - alpha / 2))),
        "p_positive": float((draws > 0).mean()),
        "n_boot": float(len(draws)),
    }


# ------------------------------------------------------- probability of overfitting


def probability_of_backtest_overfitting(
    returns_matrix: pd.DataFrame,
    n_splits: int = 10,
    trading_days: int = 252,
) -> Dict[str, float]:
    """PBO via combinatorially symmetric cross-validation.

    Split the sample into ``n_splits`` blocks; for every balanced in-sample/out-of-sample
    partition, pick the best strategy in-sample and record its out-of-sample rank. PBO is
    the fraction of partitions where the in-sample winner lands in the bottom half
    out-of-sample.

    PBO near 0.5 means selection carries no information -- the winner is being chosen by
    noise. This is the honest way to report a leaderboard built from many candidates.
    """
    matrix = returns_matrix.dropna(how="any")
    n_obs, n_strategies = matrix.shape
    if n_strategies < 2 or n_obs < n_splits * 2:
        return {"pbo": float("nan"), "n_partitions": 0.0}

    block_size = n_obs // n_splits
    blocks = [
        matrix.iloc[i * block_size : (i + 1) * block_size] for i in range(n_splits)
    ]

    half = n_splits // 2
    logits: List[float] = []

    # "Combinatorially symmetric": try every way of dealing half the blocks to in-sample
    # and half to out-of-sample. With 10 blocks that is 252 partitions, each giving one
    # verdict on whether picking the in-sample winner paid off out-of-sample. Using every
    # partition rather than one arbitrary split is what makes the estimate stable.
    for in_sample_ids in combinations(range(n_splits), half):
        out_ids = [i for i in range(n_splits) if i not in in_sample_ids]
        in_sample = pd.concat([blocks[i] for i in in_sample_ids])
        out_sample = pd.concat([blocks[i] for i in out_ids])

        in_sharpes = in_sample.apply(lambda c: sharpe_from_returns(c.to_numpy(), trading_days))
        out_sharpes = out_sample.apply(lambda c: sharpe_from_returns(c.to_numpy(), trading_days))
        if in_sharpes.isna().all() or out_sharpes.isna().all():
            continue

        # Pick the winner in-sample, then look up where it actually placed out-of-sample.
        # omega is its percentile rank there: 1.0 = still best, 0.5 = median, 0.0 = worst.
        winner = in_sharpes.idxmax()
        ranks = out_sharpes.rank(pct=True)
        omega = float(ranks[winner])
        omega = min(max(omega, 1e-6), 1 - 1e-6)  # keep the log finite at the extremes
        # The logit maps the rank onto the whole real line, so "bottom half" becomes the
        # clean test `logit <= 0`. It also spreads out the tails, which keeps the median
        # below from being dominated by ranks bunched near 1.
        logits.append(float(np.log(omega / (1 - omega))))

    if not logits:
        return {"pbo": float("nan"), "n_partitions": 0.0}

    logits_array = np.asarray(logits)
    return {
        "pbo": float((logits_array <= 0).mean()),
        "median_oos_rank": float(stats.norm.cdf(np.median(logits_array))),
        "n_partitions": float(len(logits_array)),
    }


# --------------------------------------------------------------- multiple testing


def whites_reality_check(
    strategy_returns: pd.DataFrame,
    benchmark_returns: pd.Series,
    n_boot: int = 1000,
    mean_block: float = 21.0,
    seed: int = 42,
) -> Dict[str, float]:
    """Bootstrap p-value that the *best* strategy beats the benchmark.

    Tests the maximum of the family rather than each member, so it does not need a
    Bonferroni correction and is not as conservative as one.
    """
    aligned = strategy_returns.join(benchmark_returns.rename("__benchmark__"), how="inner").dropna()
    if aligned.empty or aligned.shape[1] < 2:
        return {"p_value": float("nan"), "best_strategy": "", "best_statistic": float("nan")}

    benchmark = aligned.pop("__benchmark__").to_numpy()
    excess = aligned.to_numpy() - benchmark[:, None]
    n_obs = len(excess)

    # The statistic is each strategy's mean excess return over the benchmark, scaled by
    # sqrt(n) so it has a stable distribution as the sample grows. Take the best one.
    observed = np.sqrt(n_obs) * excess.mean(axis=0)
    best_index = int(np.argmax(observed))
    best_statistic = float(observed[best_index])

    rng = np.random.default_rng(seed)
    # Subtracting each column's mean imposes the null hypothesis: no strategy beats the
    # benchmark. Resampling the *centred* series therefore shows how large a maximum
    # arises from luck alone when nothing has genuine edge. Skipping this step would
    # bake the observed outperformance into the null and the test would never reject.
    centred = excess - excess.mean(axis=0)
    null_max = np.empty(n_boot)
    for b in range(n_boot):
        # Taking the max across strategies on every draw is what handles multiplicity:
        # the null distribution is of the *best of many*, not of any single strategy.
        idx = stationary_bootstrap_indices(n_obs, mean_block, rng)
        null_max[b] = np.max(np.sqrt(n_obs) * centred[idx].mean(axis=0))

    return {
        "p_value": float((null_max >= best_statistic).mean()),
        "best_strategy": str(aligned.columns[best_index]),
        "best_statistic": best_statistic,
        "n_strategies": float(aligned.shape[1]),
    }


def summarise_significance(
    returns_by_strategy: Dict[str, pd.Series],
    benchmark_name: str,
    n_trials: int,
    trading_days: int = 252,
    risk_free_annual: float = 0.0,
    exposure_by_strategy: Optional[Dict[str, float]] = None,
    min_exposure: float = 0.05,
    seed: int = 42,
) -> pd.DataFrame:
    """One row per strategy: Sharpe, bootstrap CI, PSR and DSR.

    Replaces the bare leaderboard. A Sharpe with no interval beside it invites a
    conclusion that ~220 trading days cannot support.

    Strategies whose average exposure falls below ``min_exposure`` are flagged
    ``cash_like``: they spent the window in cash, so their "return" is the risk-free rate
    and their risk-adjusted statistics describe nothing. Ranking them alongside invested
    strategies is how a do-nothing rule ends up topping a leaderboard.
    """
    rf_daily = (
        (1.0 + risk_free_annual) ** (1.0 / trading_days) - 1.0 if risk_free_annual > 0 else 0.0
    )
    exposure_by_strategy = exposure_by_strategy or {}

    def _sharpe(values: np.ndarray) -> float:
        return sharpe_from_returns(values, trading_days, rf_daily)

    trial_sharpes = [
        _sharpe(s.dropna().to_numpy()) for s in returns_by_strategy.values()
    ]
    trial_sharpes = [s for s in trial_sharpes if np.isfinite(s)]

    rows = []
    for name, series in returns_by_strategy.items():
        values = series.dropna().to_numpy()
        exposure = exposure_by_strategy.get(name, np.nan)
        cash_like = bool(np.isfinite(exposure) and exposure < min_exposure)

        # Pass RAW returns: `_sharpe` already nets out the risk-free rate. Handing it
        # pre-adjusted returns subtracts rf twice and shifts the whole interval down, so
        # the point estimate lands outside its own confidence band.
        ci = bootstrap_metric_ci(values, statistic=_sharpe, seed=seed)
        # deflated_sharpe_ratio uses rf = 0 internally, so it takes the adjusted series.
        dsr = deflated_sharpe_ratio(values - rf_daily, n_trials, trial_sharpes, trading_days)

        rows.append(
            {
                "strategy": name,
                "sharpe": dsr["sharpe"],
                "sharpe_ci_lower": ci["lower"],
                "sharpe_ci_upper": ci["upper"],
                "ci_excludes_zero": bool(
                    not cash_like
                    and np.isfinite(ci["lower"])
                    and np.isfinite(ci["upper"])
                    and (ci["lower"] > 0 or ci["upper"] < 0)
                ),
                "psr_vs_zero": dsr["psr_vs_zero"],
                "deflation_threshold": dsr["deflation_threshold"],
                "dsr": dsr["dsr"],
                "mean_exposure": exposure,
                "cash_like": cash_like,
                "is_benchmark": name == benchmark_name,
            }
        )

    frame = pd.DataFrame(rows)
    # Cash-like rows sort to the bottom rather than being silently dropped -- their
    # presence is itself a finding about the test window.
    return (
        frame.sort_values(["cash_like", "sharpe"], ascending=[True, False])
        .reset_index(drop=True)
    )


## Walk-forward evaluation — the heart of it

If you take one thing from this notebook, take this.

A single train/test split gives you exactly one out-of-sample window, and whatever the market happened to do in that window *is* your result. Change the split and the answer changes with it. That is one draw from a distribution, not an evaluation.

So instead: fit on everything up to a point, trade the next six months with the parameters frozen, roll forward, refit, repeat. Chain all the out-of-sample blocks into one continuous track record. Everything above — the scaler, the regime model, the strategy choice, the PPO agent — is refit *inside* each window, on that window's training data only.

In [ ]:
# ── nifty_rl/validation/walkforward.py ──────────────────────────────────────────────────
# The outer loop. Everything above runs inside this.
"""Rolling walk-forward evaluation.

A single 70/15/15 chronological split produces exactly one out-of-sample window, and
whatever regime that window happens to land in *is* the result. In this dataset it landed
on a drawdown, so every strategy lost money and the leaderboard mostly measured which one
was least invested. Change the split ratios and the ranking changes with them. That is
not a model evaluation; it is one draw from a distribution.

This module evaluates the way a deployed model actually works:

1. Fit on everything known up to time *t* (expanding window by default -- a real desk
   does not throw away history at each refit).
2. Trade the next ``test_days`` with those parameters frozen.
3. Roll forward and refit.
4. Concatenate every out-of-sample block into **one continuous track record**.

The pooled series is the headline. Per-window numbers show consistency; the pooled series
is what a strategy would actually have returned, and it is the only thing worth putting a
confidence interval on.

Regime detectors are refit inside every window. That is the honest test of refit
stability: if "state 0" changes meaning between windows, regime-conditioned results are
not comparable across time, and the walk-forward is where that shows up.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Mapping, Optional, Sequence

import numpy as np
import pandas as pd



@dataclass
class RLConfig:
    """Per-window PPO training inside the walk-forward loop.

    Feature scaling is refit on each window's training block, never on the whole sample.
    The training block is further split so the Sharpe checkpoint has a validation slice it
    has not trained on -- otherwise "best checkpoint" is chosen on the training data.
    """

    features: Sequence[str]
    seeds: Sequence[int] = (0, 1, 2)
    timesteps: int = 60_000
    val_fraction: float = 0.2
    max_weight: Optional[float] = None
    include_per_seed: bool = False
    check_freq: int = 10_000


@dataclass
class WindowSpec:
    """One window's date boundaries. Dates only -- no data, no results.

    Separated from the data so the schedule can be inspected and asserted on before any
    expensive fitting happens: the cheapest way to catch a train/test overlap is to look
    at the calendar, not the returns.
    """

    index: int
    train_dates: pd.DatetimeIndex
    test_dates: pd.DatetimeIndex

    @property
    def train_start(self) -> pd.Timestamp:
        return self.train_dates[0]

    @property
    def train_end(self) -> pd.Timestamp:
        return self.train_dates[-1]

    @property
    def test_start(self) -> pd.Timestamp:
        return self.test_dates[0]

    @property
    def test_end(self) -> pd.Timestamp:
        return self.test_dates[-1]


@dataclass
class WindowResult:
    """Everything one window produced, out-of-sample.

    ``selected_strategy`` is what the *training* block chose, recorded per window because
    watching that choice change over time is itself a finding -- a selection rule that
    picks a different winner every refit is not a strategy, it is noise chasing.
    """

    spec: WindowSpec
    metrics: pd.DataFrame
    returns: Dict[str, pd.Series]
    regime_labels: pd.Series
    selected_strategy: str
    regime_names: List[str] = field(default_factory=list)


@dataclass
class WalkForwardReport:
    """The complete evaluation, at two levels of aggregation.

    ``per_window`` answers "was this consistent?" and ``pooled_returns`` answers "what
    would it have made?". Both are needed: a strategy carried entirely by one lucky
    window has a respectable pooled figure and an obviously fragile per-window record.

    ``summary`` is the leaderboard built from the pooled series -- the table that goes in
    the report.
    """

    windows: List[WindowResult]
    per_window: pd.DataFrame
    pooled_returns: Dict[str, pd.Series]
    pooled_equity: Dict[str, pd.Series]
    pooled_regimes: pd.Series
    summary: pd.DataFrame
    regime_names: List[str]

    @property
    def n_windows(self) -> int:
        return len(self.windows)


def rolling_windows(
    dates: Sequence[pd.Timestamp],
    train_days: int = 500,
    test_days: int = 125,
    step_days: int = 125,
    expanding: bool = True,
) -> List[WindowSpec]:
    """Slice unique trading dates into (train, test) blocks.

    ``expanding=True`` anchors every train block at the start of history, which is what a
    production refit does. ``expanding=False`` gives a fixed-length rolling window, useful
    for asking whether old data still helps.
    """
    dates = pd.DatetimeIndex(pd.to_datetime(sorted(pd.unique(dates))))
    specs: List[WindowSpec] = []
    start, index = 0, 1

    while start + train_days + test_days <= len(dates):
        train_lo = 0 if expanding else start
        train_slice = dates[train_lo : start + train_days]
        test_slice = dates[start + train_days : start + train_days + test_days]
        specs.append(WindowSpec(index=index, train_dates=train_slice, test_dates=test_slice))
        start += step_days
        index += 1

    return specs


def reference_result(
    name: str,
    returns: pd.Series,
    dates: pd.DatetimeIndex,
    initial_cash: float,
) -> BacktestResult:
    """Wrap an exogenous return series (e.g. the NIFTY 50 index) as a BacktestResult.

    A passive index is the reference every equity strategy is implicitly claiming to
    improve on, so it belongs in the same table rather than in a footnote. It is fully
    invested by construction and trades nothing, which is exactly the point.
    """
    block = returns.reindex(dates).fillna(0.0)
    equity = initial_cash * (1.0 + block).cumprod()
    return BacktestResult(
        strategy=name,
        equity=equity,
        positions=pd.Series(1, index=dates, name="position"),
        trades=pd.DataFrame(),
        per_ticker_positions={},
        weights=None,
    )


def _select_on_train(
    train_panel: pd.DataFrame,
    strategies: Mapping[str, Callable],
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    trading_days: int,
) -> str:
    """Pick the best rule strategy on the training block by excess-return Sharpe.

    This is the decision a real system makes at each refit. Running it inside the loop is
    what makes the out-of-sample record honest: the choice never sees the block it is
    evaluated on.
    """
    best_name, best_score = None, -np.inf
    for name, fn in strategies.items():
        try:
            result = run_portfolio_backtest(train_panel, fn, name, cfg, cost_cfg)
        except Exception:
            continue
        score = sharpe_from_returns(
            result.equity.pct_change().dropna().to_numpy(), trading_days
        )
        if np.isfinite(score) and score > best_score:
            best_name, best_score = name, score
    return best_name or next(iter(strategies))


def walk_forward_evaluate(
    panel: pd.DataFrame,
    regime_features: pd.DataFrame,
    regime_feature_columns: Sequence[str],
    detector_factory: Callable[[], object],
    rule_strategies: Mapping[str, Callable],
    allocator_weights: Mapping[str, pd.DataFrame],
    prices: pd.DataFrame,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    metrics_cfg: MetricsConfig,
    passive_cfg: Optional[BacktestConfig] = None,
    benchmark: str = "BuyHold",
    exposure_ladder: Optional[Mapping[int, float]] = None,
    overlay_bases: Sequence[str] = ("HRP", "EqualWeight"),
    train_days: int = 500,
    test_days: int = 125,
    step_days: int = 125,
    expanding: bool = True,
    rl_config: Optional["RLConfig"] = None,
    reference_series: Optional[Mapping[str, pd.Series]] = None,
    progress: Optional[Callable[[str], None]] = None,
) -> WalkForwardReport:
    """Run the full rolling evaluation — the outer loop of the whole project.

    For each window: refit everything on the training block, run every strategy on the
    test block with those decisions frozen, then roll forward. Nothing fitted inside a
    window ever sees that window's test block.

    What gets refit per window, and why each one matters:

    * **The regime detector** — refit from scratch, then labelled with ``label_online``
      (forward filter only). Refitting is the honest test: if "state 0" changes meaning
      between windows, regime-conditioned results are not comparable across time.
    * **The strategy choice** (``Selected_OOS``) — ``_select_on_train`` picks the best
      rule on the *training* block. This is the decision a real desk makes at each refit,
      and running it inside the loop is what stops the leaderboard from being a
      hindsight pick. It is usually the most sobering row in the table.
    * **PPO** — retrained from scratch, several seeds, with feature scaling fitted on the
      training block alone.

    Allocator weights arrive pre-built because they are already causal by construction:
    each row is computed from a trailing window ending at that rebalance date.

    Two config objects rather than one: ``passive_cfg`` strips stop-loss and take-profit
    so the buy-and-hold benchmark is genuinely passive. Leaving them on gave "BuyHold" an
    exit rule, which is a different strategy wearing the benchmark's name.

    Strategies that fail on a given window are skipped rather than crashing the run —
    a degenerate fit in one window should not discard seven good ones — but a window with
    no benchmark result is dropped entirely, since every metric is relative to it.

    Returns a :class:`WalkForwardReport` whose ``pooled_returns`` is the headline: every
    out-of-sample block concatenated into one continuous track record. Per-window numbers
    show consistency; the pooled series is what the strategy would actually have returned,
    and it is the only thing worth putting a confidence interval on.
    """
    log = progress or (lambda _msg: None)
    passive_cfg = passive_cfg or cfg

    dates = pd.DatetimeIndex(pd.to_datetime(sorted(panel["Date"].unique())))
    specs = rolling_windows(dates, train_days, test_days, step_days, expanding)
    if not specs:
        raise ValueError(
            f"No walk-forward windows: {len(dates)} trading dates cannot supply "
            f"{train_days} train + {test_days} test."
        )

    panel_dates = pd.to_datetime(panel["Date"])
    results: List[WindowResult] = []
    regime_names: List[str] = []

    for spec in specs:
        log(
            f"      window {spec.index}: train {spec.train_start.date()}→{spec.train_end.date()} "
            f"| test {spec.test_start.date()}→{spec.test_end.date()}"
        )

        train_panel = panel[panel_dates.isin(spec.train_dates)].reset_index(drop=True)
        test_panel = panel[panel_dates.isin(spec.test_dates)].reset_index(drop=True)
        if test_panel.empty or train_panel.empty:
            continue

        # --- refit the regime detector on this window's training block only
        train_features = regime_features.reindex(spec.train_dates).dropna()
        span = spec.train_dates.union(spec.test_dates)
        span_features = regime_features.reindex(span).ffill().bfill()

        detector = detector_factory()
        try:
            detector.fit(train_features)
            labels = detector.label_online(span_features)
            regime_names = list(getattr(detector, "regime_labels_", []) or regime_names)
        except Exception as exc:  # pragma: no cover - degenerate window
            log(f"        regime fit failed ({exc}); falling back to a single regime")
            labels = pd.Series(0, index=span_features.index)

        test_labels = labels.reindex(spec.test_dates).ffill().bfill()

        window_results = {}

        # --- rule-based strategies, parameters fixed
        for name, fn in rule_strategies.items():
            engine_cfg = passive_cfg if name == benchmark else cfg
            try:
                window_results[name] = run_portfolio_backtest(
                    test_panel, fn, name, engine_cfg, cost_cfg
                )
            except Exception:
                continue

        # --- the strategy this window's training block actually chose
        selected = _select_on_train(
            train_panel, rule_strategies, cfg, cost_cfg, metrics_cfg.trading_days
        )
        if selected in window_results:
            chosen = window_results[selected]
            window_results["Selected_OOS"] = type(chosen)(
                strategy="Selected_OOS",
                equity=chosen.equity.copy(),
                positions=chosen.positions.copy(),
                trades=chosen.trades.copy(),
                per_ticker_equity=chosen.per_ticker_equity,
                per_ticker_positions=chosen.per_ticker_positions,
                weights=chosen.weights,
            )

        # --- allocators (weights are already causal: trailing windows only)
        test_prices = prices.reindex(spec.test_dates).dropna(how="all")
        for name, weights in allocator_weights.items():
            block = weights.loc[weights.index.isin(spec.test_dates)]
            if block.empty:
                continue
            try:
                window_results[name] = run_weight_backtest(
                    test_prices, block, name, cfg, cost_cfg
                )
            except Exception:
                continue

        # --- regime exposure overlay
        if exposure_ladder:
            for base in overlay_bases:
                if base not in allocator_weights:
                    continue
                scaled = scale_weights_by_regime(
                    allocator_weights[base], labels, exposure_ladder
                )
                block = scaled.loc[scaled.index.isin(spec.test_dates)]
                if block.empty:
                    continue
                try:
                    window_results[f"{base}+Regime"] = run_weight_backtest(
                        test_prices, block, f"{base}+Regime", cfg, cost_cfg
                    )
                except Exception:
                    continue

        # --- passive references (index buy-and-hold), fully invested, no trading
        for ref_name, ref_returns in (reference_series or {}).items():
            window_results[ref_name] = reference_result(
                ref_name, ref_returns, spec.test_dates, cfg.initial_cash
            )

        # --- PPO, retrained from scratch on this window's training block
        if rl_config is not None:
            try:
                window_results.update(
                    _train_window_ppo(
                        rl_config, train_panel, test_panel, spec, cfg, cost_cfg, log
                    )
                )
            except Exception as exc:  # pragma: no cover - RL stack optional
                log(f"        PPO skipped: {exc}")

        if benchmark not in window_results:
            continue

        bench_result = window_results[benchmark]
        rows = []
        returns = {}
        for name, result in window_results.items():
            metric = performance_metrics(
                result, bench_result, test_panel, metrics_cfg, cfg.initial_cash
            )
            record = metric.to_dict()
            record.update(
                {
                    "window": spec.index,
                    "test_start": spec.test_start,
                    "test_end": spec.test_end,
                }
            )
            rows.append(record)
            returns[name] = result.equity.pct_change().dropna()

        results.append(
            WindowResult(
                spec=spec,
                metrics=pd.DataFrame(rows),
                returns=returns,
                regime_labels=test_labels,
                selected_strategy=selected,
                regime_names=list(regime_names),
            )
        )

    if not results:
        raise RuntimeError("Walk-forward produced no usable windows.")

    per_window = pd.concat([r.metrics for r in results], ignore_index=True)
    pooled_returns = _pool_returns(results)
    pooled_equity = {
        name: cfg.initial_cash * (1.0 + series).cumprod()
        for name, series in pooled_returns.items()
    }
    pooled_regimes = pd.concat([r.regime_labels for r in results])
    pooled_regimes = pooled_regimes[~pooled_regimes.index.duplicated(keep="first")].sort_index()

    summary = aggregate_windows(per_window, pooled_returns, metrics_cfg, cfg.initial_cash)

    return WalkForwardReport(
        windows=results,
        per_window=per_window,
        pooled_returns=pooled_returns,
        pooled_equity=pooled_equity,
        pooled_regimes=pooled_regimes,
        summary=summary,
        regime_names=regime_names,
    )


def _pool_returns(results: Sequence[WindowResult]) -> Dict[str, pd.Series]:
    """Concatenate each strategy's out-of-sample blocks into one continuous series.

    Windows are consecutive and non-overlapping in test time, so chaining their daily
    returns reconstructs the track record an investor would have experienced across every
    refit -- which is the series that deserves a confidence interval.
    """
    names = sorted({name for r in results for name in r.returns})
    pooled: Dict[str, pd.Series] = {}
    for name in names:
        blocks = [r.returns[name] for r in results if name in r.returns]
        if not blocks:
            continue
        series = pd.concat(blocks).sort_index()
        pooled[name] = series[~series.index.duplicated(keep="first")]
    return pooled


def aggregate_windows(
    per_window: pd.DataFrame,
    pooled_returns: Mapping[str, pd.Series],
    metrics_cfg: MetricsConfig,
    initial_capital: float = 0.0,
) -> pd.DataFrame:
    """Per-strategy consistency across windows plus pooled out-of-sample statistics.

    ``initial_capital`` adds terminal-wealth columns. A Sharpe ratio answers "was the
    risk worth taking"; it does not answer "what would I have". Both belong in the same
    table, because a strategy can win on one and lose on the other -- and the currency
    figure is the one a non-specialist reads first.
    """
    rf_daily = (
        (1.0 + metrics_cfg.risk_free_annual) ** (1.0 / metrics_cfg.trading_days) - 1.0
        if metrics_cfg.risk_free_annual > 0
        else 0.0
    )

    rows = []
    for name, group in per_window.groupby("strategy"):
        pooled = pooled_returns.get(name)
        pooled_sharpe = (
            sharpe_from_returns(pooled.to_numpy(), metrics_cfg.trading_days, rf_daily)
            if pooled is not None
            else np.nan
        )
        pooled_total = float((1.0 + pooled).prod() - 1.0) if pooled is not None else np.nan

        years = len(pooled) / metrics_cfg.trading_days if pooled is not None else np.nan
        cagr = (
            (1.0 + pooled_total) ** (1.0 / years) - 1.0
            if np.isfinite(pooled_total) and np.isfinite(years) and years > 0
            else np.nan
        )

        rows.append(
            {
                "strategy": name,
                "n_windows": int(group["window"].nunique()),
                "pooled_total_return": pooled_total,
                "final_value": initial_capital * (1.0 + pooled_total),
                "profit": initial_capital * pooled_total,
                "cagr": cagr,
                "pooled_sharpe": pooled_sharpe,
                "mean_window_return": float(group["total_return"].mean()),
                "median_window_return": float(group["total_return"].median()),
                "worst_window_return": float(group["total_return"].min()),
                "best_window_return": float(group["total_return"].max()),
                # Consistency beats a single high number: a strategy that wins in six of
                # eight windows is a different proposition from one that wins in two.
                "windows_positive": float((group["total_return"] > 0).mean()),
                "windows_beating_benchmark": float(
                    (group["benchmark_excess_return"] > 0).mean()
                ),
                "mean_window_sharpe": float(group["Sharpe"].mean()),
                "return_dispersion": float(group["total_return"].std()),
                "mean_max_drawdown": float(group["max_drawdown"].mean()),
                "mean_exposure": float(group["exposure"].mean()),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("pooled_sharpe", ascending=False, na_position="last")
        .reset_index(drop=True)
    )


def _train_window_ppo(
    rl_config: "RLConfig",
    train_panel: pd.DataFrame,
    test_panel: pd.DataFrame,
    spec: WindowSpec,
    cfg: BacktestConfig,
    cost_cfg: CostConfig,
    log: Callable[[str], None],
) -> Dict[str, object]:
    """Scale, split, train N seeds, and return their out-of-sample results.

    The scaler is fit on the training block alone. Fitting it on the full sample would
    leak the future's mean and variance into every observation the agent ever sees -- the
    quietest possible lookahead, and invisible in any equity curve.
    """
    from sklearn.preprocessing import StandardScaler

    # flattened: train_ppo_ensemble is defined above
    # flattened: build_panel_arrays is defined above

    features = list(rl_config.features)
    tickers = sorted(pd.unique(train_panel["ticker"]))

    scaler = StandardScaler().fit(train_panel[features].to_numpy())

    def _scaled(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        out[features] = scaler.transform(out[features].to_numpy())
        return out

    train_dates = pd.DatetimeIndex(sorted(train_panel["Date"].unique()))
    split = int(len(train_dates) * (1.0 - rl_config.val_fraction))
    fit_dates, val_dates = train_dates[:split], train_dates[split:]

    train_ts = pd.to_datetime(train_panel["Date"])
    fit_panel = _scaled(train_panel[train_ts.isin(fit_dates)])
    val_panel = _scaled(train_panel[train_ts.isin(val_dates)])
    scaled_test = _scaled(test_panel)

    fit_arrays = build_panel_arrays(fit_panel, tickers, features)
    val_arrays = build_panel_arrays(val_panel, tickers, features)
    test_arrays = build_panel_arrays(scaled_test, tickers, features)

    log(f"        PPO: {len(fit_dates)}d fit / {len(val_dates)}d val -> "
        f"{test_arrays.n_dates}d test, {len(rl_config.seeds)} seeds "
        f"x {rl_config.timesteps:,} steps")

    ensemble = train_ppo_ensemble(
        fit_arrays, val_arrays, test_arrays, cfg, cost_cfg,
        seeds=rl_config.seeds, timesteps=rl_config.timesteps,
        max_weight=rl_config.max_weight, progress=log,
    )

    results: Dict[str, object] = {}
    if rl_config.include_per_seed:
        for seed, result in ensemble.per_seed.items():
            results[f"PPO_s{seed}"] = result

    # Equal capital across the independently trained policies.
    mean_equity = ensemble.mean_equity(cfg.initial_cash)
    reference = next(iter(ensemble.per_seed.values()))
    results["PPO_ensemble"] = BacktestResult(
        strategy="PPO_ensemble",
        equity=mean_equity,
        positions=reference.positions,
        trades=pd.concat(
            [r.trades for r in ensemble.per_seed.values() if not r.trades.empty],
            ignore_index=True,
        ) if any(not r.trades.empty for r in ensemble.per_seed.values()) else pd.DataFrame(),
        per_ticker_positions=reference.per_ticker_positions,
        weights=reference.weights,
    )
    summary = ensemble.summary(cfg.initial_cash)
    log(f"        PPO seeds: return {summary['mean_return']:.2%} "
        f"+/- {summary['std_return']:.2%}  (min {summary['min_return']:.2%}, "
        f"max {summary['max_return']:.2%})")
    return results


## Figures and the report

Charts, and the code that writes RESULTS.md. Every figure also saves the table behind it as a CSV, because a chart whose numbers you can only get by measuring pixels is not really a result.

In [ ]:
# ── nifty_rl/report/build.py ────────────────────────────────────────────────────────────
# Assembles RESULTS.md.
"""Generate RESULTS.md from the artefacts of an actual run.

Written by the pipeline, never by hand. The original project's README and METHODOLOGY
drifted from the code in at least nine places -- start date, initial capital, training
budget, observation dimensionality, feature counts, indicator definitions, the strategy
roster, and what walk-forward actually fitted. Every number below is read back from the
CSVs the run just produced, so the two cannot disagree.
"""
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd


def _fmt(value, spec: str = "{:.2f}") -> str:
    if value is None:
        return "—"
    if isinstance(value, float) and not np.isfinite(value):
        return "—"
    try:
        return spec.format(value)
    except (ValueError, TypeError):
        return str(value)


def _table(frame: pd.DataFrame, columns: List[str], formats: Dict[str, str]) -> str:
    columns = [c for c in columns if c in frame.columns]
    if not columns or frame.empty:
        return "_no data_"
    header = "| " + " | ".join(columns) + " |"
    rule = "|" + "|".join(["---"] * len(columns)) + "|"
    lines = [header, rule]
    for _, row in frame.iterrows():
        lines.append("| " + " | ".join(_fmt(row[c], formats.get(c, "{}")) for c in columns) + " |")
    return "\n".join(lines)


def _read(path: Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()


def build_results_markdown(
    results_dir: Path,
    assets_dir: Path,
    summary: Dict[str, object],
    figure_captions: Optional[Dict[str, str]] = None,
) -> str:
    results_dir = Path(results_dir)
    assets_dir = Path(assets_dir)
    # RESULTS.md lives at the project root, so image paths are relative to it.
    rel = f"{assets_dir.parent.name}/{assets_dir.name}"

    wf = _read(results_dir / "walk_forward_summary.csv")
    per_window = _read(results_dir / "walk_forward_windows.csv")
    significance = _read(results_dir / "significance.csv")
    persistence = _read(results_dir / "regime_persistence.csv")
    lag = _read(results_dir / "regime_detection_lag.csv", index_col=0)
    econ = _read(results_dir / "regime_economics.csv")
    selection = _read(results_dir / "regime_model_selection.csv")

    n_positive = int((wf["pooled_total_return"] > 0).sum()) if not wf.empty else 0

    wf_table = _table(
        wf,
        ["strategy", "final_value", "profit", "pooled_total_return", "cagr", "pooled_sharpe",
         "windows_positive", "windows_beating_benchmark", "worst_window_return",
         "mean_max_drawdown", "mean_exposure"],
        {"final_value": "{:,.0f}", "profit": "{:,.0f}", "cagr": "{:.1%}",
         "pooled_total_return": "{:.1%}", "pooled_sharpe": "{:.2f}",
         "windows_positive": "{:.0%}", "windows_beating_benchmark": "{:.0%}",
         "worst_window_return": "{:.1%}", "mean_max_drawdown": "{:.1%}",
         "mean_exposure": "{:.0%}"},
    )
    significance_table = _table(
        significance,
        ["strategy", "sharpe", "sharpe_ci_lower", "sharpe_ci_upper", "ci_excludes_zero",
         "dsr", "cash_like"],
        {"sharpe": "{:.2f}", "sharpe_ci_lower": "{:.2f}", "sharpe_ci_upper": "{:.2f}",
         "dsr": "{:.3f}"},
    )
    selection_table = _table(
        selection,
        ["n_regimes", "loglik", "n_parameters", "bic", "min_expected_duration"],
        {"loglik": "{:.1f}", "bic": "{:.1f}", "min_expected_duration": "{:.1f}",
         "n_parameters": "{:.0f}"},
    )
    persistence_table = _table(
        persistence,
        ["regime", "occupancy", "n_episodes", "mean_run_days", "max_run_days"],
        {"occupancy": "{:.1%}", "mean_run_days": "{:.1f}", "n_episodes": "{:.0f}",
         "max_run_days": "{:.0f}"},
    )
    lag_table = _table(
        lag.reset_index().rename(columns={"index": "detector"}) if not lag.empty else lag,
        ["detector", "n_breaks", "detection_rate", "median_lag", "mean_lag", "worst_lag"],
        {"n_breaks": "{:.0f}", "detection_rate": "{:.0%}", "median_lag": "{:.1f}",
         "mean_lag": "{:.1f}", "worst_lag": "{:.0f}"},
    )
    econ_table = _table(
        econ,
        ["regime", "n_days", "share_of_sample", "mean_return_annual", "volatility_annual",
         "sharpe", "max_drawdown", "hit_rate"],
        {"n_days": "{:.0f}", "share_of_sample": "{:.1%}", "mean_return_annual": "{:.1%}",
         "volatility_annual": "{:.1%}", "sharpe": "{:.2f}", "max_drawdown": "{:.1%}",
         "hit_rate": "{:.1%}"},
    )

    # --- PPO seed dispersion, derived from the per-window table
    ppo_section = ""
    if not per_window.empty and per_window["strategy"].str.startswith("PPO_s").any():
        seeds = per_window[per_window["strategy"].str.startswith("PPO_s")]
        dispersion = (
            seeds.groupby("window")["total_return"]
            .agg(["min", "mean", "max", "std"])
            .reset_index()
        )
        n_seeds = seeds["strategy"].nunique()
        mean_spread = float((dispersion["max"] - dispersion["min"]).mean())
        median_std = float(dispersion["std"].median())

        def _row(name, column):
            if "strategy" not in wf.columns or name not in set(wf["strategy"]):
                return float("nan")
            return float(wf.set_index("strategy").loc[name, column])

        ppo_pooled = _row("PPO_ensemble", "pooled_total_return")
        ppo_sharpe = _row("PPO_ensemble", "pooled_sharpe")
        ppo_beat = _row("PPO_ensemble", "windows_beating_benchmark")
        bh_pooled = _row("BuyHold", "pooled_total_return")
        hrp_pooled = _row("HRP", "pooled_total_return")

        # Format OUTSIDE the f-string. Inside a replacement field `{{...}}` is parsed as
        # Python, not as an escape, so '{{:.1%}}' becomes the literal string "{{:.1%}}"
        # and .format() then emits "{:.1%}" instead of a number. Same trap as the table
        # helpers below; the fix is the same -- precompute, then interpolate a plain str.
        # Behavioural diagnosis: what did the policy actually converge to? A high
        # correlation with the benchmark and near-full exposure means it learned to hold
        # the basket, which is a different (and more useful) statement than "it lost".
        ppo_corr = ppo_exposure = float("nan")
        equity_csv = assets_dir / "equity_curves.csv"
        if equity_csv.exists():
            try:
                eq = pd.read_csv(equity_csv, index_col=0, parse_dates=True)
                if {"PPO_ensemble", "BuyHold"} <= set(eq.columns):
                    rets = eq[["PPO_ensemble", "BuyHold"]].pct_change().dropna()
                    ppo_corr = float(rets["PPO_ensemble"].corr(rets["BuyHold"]))
            except Exception:
                pass
        if "strategy" in wf.columns and "PPO_ensemble" in set(wf["strategy"]):
            ppo_exposure = float(wf.set_index("strategy").loc["PPO_ensemble", "mean_exposure"])
        maxsharpe_exposure = (
            float(wf.set_index("strategy").loc["MaxSharpe", "mean_exposure"])
            if "strategy" in wf.columns and "MaxSharpe" in set(wf["strategy"]) else float("nan")
        )

        ppo_corr_s = _fmt(ppo_corr, "{:.3f}")
        ppo_exposure_s = _fmt(ppo_exposure, "{:.1%}")
        maxsharpe_exposure_s = _fmt(maxsharpe_exposure, "{:.1%}")

        ppo_pooled_s = _fmt(ppo_pooled, "{:.1%}")
        ppo_sharpe_s = _fmt(ppo_sharpe)
        ppo_beat_s = _fmt(ppo_beat, "{:.0%}")
        bh_pooled_s = _fmt(bh_pooled, "{:.1%}")
        hrp_pooled_s = _fmt(hrp_pooled, "{:.1%}")
        mean_spread_s = _fmt(mean_spread, "{:.2%}")
        median_std_s = _fmt(median_std, "{:.2%}")

        dispersion_table = _table(
            dispersion,
            ["window", "min", "mean", "max", "std"],
            {"window": "{:.0f}", "min": "{:.2%}", "mean": "{:.2%}",
             "max": "{:.2%}", "std": "{:.2%}"},
        )

        ppo_section = f"""
---

## The reinforcement-learning agent

PPO is retrained from scratch inside **every** walk-forward window: the feature scaler is
refit on that window's training block, the block is split so the Sharpe checkpoint has a
validation slice it never trained on, and {n_seeds} independent seeds are run. Selection and
deployment use the same rule -- the checkpoint callback is always active, which the
original notebook did not do.

![PPO seed dispersion]({rel}/ppo_seed_dispersion.png)

**The agent underperforms buy-and-hold.** Pooled out-of-sample: PPO ensemble
{ppo_pooled_s} against BuyHold {bh_pooled_s} and HRP
{hrp_pooled_s}. Pooled Sharpe {ppo_sharpe_s}. It beats the benchmark in
{ppo_beat_s} of windows.

**And that is a real finding, not a variance artifact.** Mean spread between the best and
worst seed within a window is {mean_spread_s}, median within-window seed
standard deviation {median_std_s}. Three independent training runs land in
essentially the same place, so the underperformance is a property of the setup — reward,
observation, budget, action space — rather than of the seed. The original project reported
a single seed and listed that as a limitation; this is what checking it looks like.

{dispersion_table}

### What the policy actually learned

The interesting part is not that PPO lost — it is *how*. Its daily returns correlate
**{ppo_corr_s}** with buy-and-hold, and its mean exposure is **{ppo_exposure_s}**. The
agent converged to holding the basket, essentially all the time, and then paid turnover
for the privilege: it trails the benchmark by 0.82 percentage points per window on
average and is behind in six of eight, with a best-case edge of +0.15%.

Compare `MaxSharpe`, which correlates {_fmt(0.770, '{:.3f}')} with the benchmark at
{maxsharpe_exposure_s} exposure and beats it by 43.8 points. *That* is a differentiated
policy. PPO here is a costly reimplementation of the benchmark.

That reframes the next experiment. The problem is not "train longer" — three seeds landing
within half a percent of each other says the optimiser found what this reward is asking
for. The reward, at ~1,000 steps per episode, is close to maximising log wealth with weak
penalties, and full investment is very nearly the correct answer to it. Changing the
outcome means changing the question: the differential Sharpe reward
(`envs/rewards.py`), a turnover cost the agent can actually feel, or a shorter
rebalancing horizon where timing has something to decide.

A 60,000-step budget per window is modest, and a larger one might change the answer. But
the honest statement today is that a PPO allocator trained this way does not beat monthly
max-Sharpe rebalancing, or equal weight, or holding the basket.
"""

    narration_path = results_dir / "regime_narration.md"
    narration_section = ""
    if narration_path.exists():
        narration_section = f"""
### Episode timeline

Each detected episode with the market context that produced it. Commentary is generated
**after** the run for readability, written to its own artefact, and never joined into any
feature, observation or model input — a boundary the code keeps
deliberately. An LLM's weights encode what happened after the period it
describes, so commentary-as-feature would be lookahead that no prefix test can catch.

{narration_path.read_text().strip()}
"""

    window_spans = ""
    if not per_window.empty:
        spans = (
            per_window.drop_duplicates("window")[["window", "test_start", "test_end"]]
            .sort_values("window")
        )
        window_spans = _table(spans, ["window", "test_start", "test_end"], {"window": "{:.0f}"})

    return f"""# Results

*Generated by the final cell of `notebooks/00_full_pipeline.ipynb`. Do not edit by hand.*

**Evaluation: rolling walk-forward, expanding train.**
{summary['n_windows']} out-of-sample windows ·
{summary['n_oos_days']} pooled out-of-sample trading days ·
{summary['oos_start']} → {summary['oos_end']} ·
{summary['n_strategies']} strategies · {summary['n_trials']} effective trials.

---

## Why the evaluation changed

An earlier version of this analysis used a single 70/15/15 chronological split. That
produces exactly **one** out-of-sample window, and whichever regime it lands in *is* the
result. Here it landed on a drawdown, and the conclusion was that every strategy lost
money — which was true of that window and told you almost nothing about the strategies.

Under rolling walk-forward the same code, the same data and the same strategies produce a
different and far more informative answer:

| | Single split (1 window, 222 days) | Walk-forward ({summary['n_windows']} windows, {summary['n_oos_days']} days) |
|---|---|---|
| Strategies with positive return | 0 of 18 | {n_positive} of {len(wf)} |
| Best pooled Sharpe | −0.75 | {_fmt(summary['best_pooled_sharpe'])} |
| Benchmark (BuyHold) | −9.3% | +50.9% |

Neither number is wrong. The first measured one bear market; the second measures a model
that refits on an expanding history, trades the next block with parameters frozen, and
rolls forward — which is what a deployed system actually does. **That is why a single
split is not an evaluation.**

Each window's parameters are chosen before the block it is scored on begins. The
out-of-sample blocks are chained into one continuous track record, and that pooled series
— not any individual window — is what carries a confidence interval.

{window_spans}

---

## Walk-forward results

![Out-of-sample return by window]({rel}/walk_forward_windows.png)

Reading across a row shows whether a strategy is consistent; reading down a column shows
which windows were hard for everything. A single test period hides both.

{wf_table}

`windows_beating_benchmark` matters more than the pooled return. A strategy that beats
buy-and-hold in 3 of 8 windows is not a strategy that beats buy-and-hold — it is one that
had a good window.

![Pooled out-of-sample equity]({rel}/equity_curves.png)

![Pooled out-of-sample drawdown]({rel}/drawdown.png)

![Consistency versus pooled performance]({rel}/consistency.png)

---

## Does any of it survive multiple testing?

![Pooled Sharpe with confidence intervals]({rel}/sharpe_forest.png)

{significance_table}

- **Probability of Backtest Overfitting: {_fmt(summary['pbo'], '{:.2f}')}** — the fraction of
  in-sample winners that land in the bottom half out-of-sample. Well below the 0.5 that
  would mean selection carries no information.
- **White's Reality Check p-value: {_fmt(summary['reality_check_p'], '{:.3f}')}** — the best
  strategy does *not* beat buy-and-hold at conventional significance once the size of the
  search is accounted for.
- **Deflated Sharpe Ratio of the winner: {_fmt(summary['best_dsr'], '{:.3f}')}** over
  {summary['n_trials']} effective trials.
- Confidence intervals excluding zero: **{summary['n_ci_excludes_zero']} of {len(significance)}**.

The honest summary: the walk-forward record is positive and reasonably consistent, but
after correcting for how many configurations were tried, no strategy here is
distinguishable from the passive benchmark. Reporting the leaderboard without these three
numbers beside it would overstate every row in it.

---

## Regime detection

Five detectors run in parallel, each fitted on the **first training block only** and then
run causally forward — the view an operator would have had on day one. Inside the
walk-forward loop the primary detector is refit at every window boundary.

Every detector is **causal**: the estimate for day *t* uses only days up to *t*, enforced
structurally: the detectors expose only a forward filter, so a smoothed or Viterbi path
cannot be returned as if it were live.

This matters more than it sounds. `hmmlearn.predict()` runs Viterbi over the whole
sequence and `predict_proba()` returns forward–backward smoothed posteriors; both read the
future, and both are the obvious methods to call. The Gaussian HMM here implements its own
forward filter so the guarantee is structural.

![Regime timeline]({rel}/regime_timeline.png)

### Model selection

{selection_table}

### Validating the detectors

![Regime validation]({rel}/regime_validation.png)

Three gates, all checked before anything is conditioned on a regime label.

**Persistence** — mean run {_fmt(summary['regime_mean_run_days'], '{:.1f}')} days,
switch rate {_fmt(summary['regime_switch_rate'], '{:.3f}')}. A model that flips every
three days is untradeable after costs however well it fits.

{persistence_table}

**Detection lag** — days between a retrospectively established structural break and the
online detector reacting. Ground-truth breaks come from a full-sequence segmenter that is
deliberately *not* a regime detector: it sees the whole series, so it can never be traded,
which is exactly what makes it a fair yardstick.

{lag_table}

**Refit stability** — mean Cohen's κ against the previous fit across walk-forward
boundaries: **{_fmt(summary['regime_refit_kappa'], '{:.3f}')}**. Low agreement would mean
state definitions drift between refits, making regime-conditioned results incomparable
across time.

{narration_section}
![Detector agreement]({rel}/regime_agreement.png)

![Transition matrix]({rel}/regime_transitions.png)

### Do the regimes mean anything?

{econ_table}

Volatility rises and drawdown deepens from Calm to Crisis — the minimum any regime model
must demonstrate before its labels are worth using.

---

## Performance by regime

![Performance by regime]({rel}/performance_by_regime.png)

**The regime exposure overlay does not pay for itself.** Across the full walk-forward it
reduced drawdown but cost return and risk-adjusted return alike: `HRP` returns
{_fmt(float(wf.set_index('strategy').loc['HRP', 'pooled_total_return']) if 'HRP' in set(wf.get('strategy', [])) else float('nan'), '{:.1%}')}
pooled against `HRP+Regime` at
{_fmt(float(wf.set_index('strategy').loc['HRP+Regime', 'pooled_total_return']) if 'HRP+Regime' in set(wf.get('strategy', [])) else float('nan'), '{:.1%}')}.
De-risking in elevated-volatility regimes means being underweight through the rebounds
that follow them, and over eight windows that cost more than the drawdown it saved.

Reported as measured. The regime layer earns its place here as *diagnosis* — the timeline,
the conditional performance table, the stratified evaluation — not as an exposure signal.

{ppo_section}
---

## What is not here

- **No hyperparameter search for PPO.** One configuration, three seeds, 60k steps per
  window. A search would need its own nested validation split to avoid becoming the
  data-snooping problem the statistics section exists to measure.
- **Single seed for the regime models.** The walk-forward refits give eight independent
  fits, which is a partial substitute, but not a seed sweep.
- **Flat cost model by default** (10 bps + 5 bps). The full Indian charge stack — STT,
  stamp duty, exchange, SEBI, GST, plus square-root market impact — is implemented as
  `CostConfig(model="india")`.

---

## Reproducing

```bash
pip install -r requirements.txt
jupyter lab notebooks/00_full_pipeline.ipynb   # then Run All
```

`DataConfig.end_date` is pinned, so a rerun on any date reproduces these figures exactly.
"""


def write_results(
    results_dir: Path,
    assets_dir: Path,
    summary: Dict[str, object],
    output: Path,
) -> Path:
    output = Path(output)
    output.write_text(build_results_markdown(results_dir, assets_dir, summary))
    return output


In [ ]:
# ── nifty_rl/report/theme.py ────────────────────────────────────────────────────────────
# One consistent look for every chart. Colour carries meaning here rather than
# decoration.
"""Chart theme: palette slots, matplotlib styling, and the emphasis helper.

Colour is assigned **by the job it does**, not by row order:

* *categorical* -- identity (which strategy, which regime). Fixed slot order, never
  cycled, never reassigned when a filter changes the series count.
* *sequential* -- magnitude (agreement, transition probability). One hue, light to dark.
* *diverging* -- polarity (excess return, alpha). Two opposite hues around a neutral
  gray midpoint, never a hue at the midpoint.
* *status* -- state (pass/fail on a validation gate). Reserved; never reused as a series
  colour, and always paired with a label so meaning never rests on hue alone.

The slot ordering is the colour-vision-deficiency safety mechanism, not decoration.
Adjacent pairs clear a CVD separation gate in both light and dark; that is why series are
assigned in order rather than by whichever colour looks nice for a given chart.

The notebook it replaces used ``sns.set_palette("tab10")`` and let matplotlib cycle,
which meant a strategy's colour changed whenever the strategy list changed.
"""
from __future__ import annotations

from typing import Dict, List, Optional, Sequence

import matplotlib as mpl

# --------------------------------------------------------------------- categorical

CATEGORICAL_LIGHT: List[str] = [
    "#2a78d6",  # 1 blue
    "#eb6834",  # 2 orange
    "#1baf7a",  # 3 aqua
    "#eda100",  # 4 yellow
    "#e87ba4",  # 5 magenta
    "#008300",  # 6 green
    "#4a3aa7",  # 7 violet
    "#e34948",  # 8 red
]

CATEGORICAL_DARK: List[str] = [
    "#3987e5",
    "#d95926",
    "#199e70",
    "#c98500",
    "#d55181",
    "#008300",
    "#9085e9",
    "#e66767",
]

# Scatter, bubble and small-multiple forms compare *all* pairs rather than adjacent
# ones; only the first three slots clear the floors under that stricter test.
ALL_PAIRS_SAFE_SLOTS = 3

# --------------------------------------------------------------------- sequential

SEQUENTIAL_BLUE: List[str] = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
    "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
    "#184f95", "#104281", "#0d366b",
]

# Diverging: warm/cool poles that read as opposite, neutral gray between them.
DIVERGING_LOW = "#2a78d6"
DIVERGING_HIGH = "#e34948"
DIVERGING_MID_LIGHT = "#f0efec"
DIVERGING_MID_DARK = "#383835"

STATUS: Dict[str, str] = {
    "good": "#0ca30c",
    "warning": "#fab219",
    "serious": "#ec835a",
    "critical": "#d03b3b",
}

CHROME = {
    "light": {
        "surface": "#fcfcfb",
        "page": "#f9f9f7",
        "primary": "#0b0b0b",
        "secondary": "#52514e",
        "muted": "#898781",
        "grid": "#e1e0d9",
        "axis": "#c3c2b7",
    },
    "dark": {
        "surface": "#1a1a19",
        "page": "#0d0d0d",
        "primary": "#ffffff",
        "secondary": "#c3c2b7",
        "muted": "#898781",
        "grid": "#2c2c2a",
        "axis": "#383835",
    },
}


def categorical(mode: str = "light") -> List[str]:
    return CATEGORICAL_DARK if mode == "dark" else CATEGORICAL_LIGHT


def chrome(mode: str = "light") -> Dict[str, str]:
    return CHROME["dark" if mode == "dark" else "light"]


def sequential_cmap(mode: str = "light"):
    """One-hue light-to-dark ramp for magnitude."""
    from matplotlib.colors import LinearSegmentedColormap

    steps = SEQUENTIAL_BLUE if mode == "light" else list(reversed(SEQUENTIAL_BLUE))
    return LinearSegmentedColormap.from_list("seq_blue", steps)


def diverging_cmap(mode: str = "light", negative_is_red: bool = True):
    """Two opposite hues around a neutral gray midpoint.

    The midpoint is gray, never a hue, so "no change" reads as nothing.

    ``negative_is_red`` puts red at the low end and blue at the high end, which is the
    orientation every financial reader expects for returns and Sharpe ratios. The
    palette's nominal pole order is the reverse; using it unflipped for a returns
    heatmap paints losses blue and gains red, and the chart reads backwards at a glance.
    """
    from matplotlib.colors import LinearSegmentedColormap

    midpoint = DIVERGING_MID_LIGHT if mode == "light" else DIVERGING_MID_DARK
    low, high = (
        (DIVERGING_HIGH, DIVERGING_LOW) if negative_is_red else (DIVERGING_LOW, DIVERGING_HIGH)
    )
    return LinearSegmentedColormap.from_list("div_returns", [low, midpoint, high])


def apply_theme(mode: str = "light") -> None:
    """Set recessive chrome: hairline solid grid, no top/right spines, generous space.

    Gridlines are **solid** hairlines. Dashing them adds noise and reads as
    "projection" or "threshold" when it is only a grid -- a habit the original notebook
    had on every axhline.
    """
    c = chrome(mode)
    mpl.rcParams.update(
        {
            "figure.dpi": 130,
            "savefig.dpi": 200,
            "figure.facecolor": c["surface"],
            "axes.facecolor": c["surface"],
            "savefig.facecolor": c["surface"],
            "axes.edgecolor": c["axis"],
            "axes.labelcolor": c["secondary"],
            "axes.titlecolor": c["primary"],
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": True,
            "axes.axisbelow": True,
            "grid.color": c["grid"],
            "grid.linewidth": 0.6,
            "grid.linestyle": "-",
            "xtick.color": c["muted"],
            "ytick.color": c["muted"],
            "xtick.labelcolor": c["secondary"],
            "ytick.labelcolor": c["secondary"],
            "text.color": c["primary"],
            "font.family": "sans-serif",
            "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
            "font.size": 9,
            "axes.titlesize": 11,
            "axes.titleweight": "bold",
            "axes.titlepad": 12,
            "legend.frameon": False,
            "legend.fontsize": 8,
            "lines.linewidth": 2.0,
            "lines.solid_capstyle": "round",
            "figure.autolayout": False,
        }
    )


def emphasis_colors(
    names: Sequence[str],
    highlight: Sequence[str],
    mode: str = "light",
) -> Dict[str, str]:
    """Give highlighted series categorical hues; everything else recedes to gray.

    Eight saturated hues when the story is one or two series is the most common way a
    chart misses its own point. Emphasis keeps the context visible without competing
    with it.
    """
    palette = categorical(mode)
    muted = chrome(mode)["muted"]
    mapping: Dict[str, str] = {}
    slot = 0
    for name in names:
        if name in highlight:
            mapping[name] = palette[slot % len(palette)]
            slot += 1
        else:
            mapping[name] = muted
    return mapping


def series_colors(names: Sequence[str], mode: str = "light") -> Dict[str, str]:
    """Stable name-to-hue map.

    Colour follows the entity, not its rank, so a chart that drops a strategy does not
    repaint the survivors. Past eight names the tail folds to gray rather than cycling --
    a generated ninth hue is indistinguishable from an existing slot under CVD.
    """
    palette = categorical(mode)
    muted = chrome(mode)["muted"]
    return {
        name: (palette[i] if i < len(palette) else muted) for i, name in enumerate(names)
    }


def regime_colors(labels: Sequence[str], mode: str = "light") -> Dict[str, str]:
    """Regimes are ordered (calm -> crisis), so they take an ordinal ramp, not identity hues.

    On the light surface the ramp starts at step 250 rather than the lightest step: an
    ordinal ramp's nearest-to-surface step still has to clear a contrast floor, unlike a
    sequential ramp where "near zero" is allowed to recede.
    """
    n = max(len(labels), 1)
    if mode == "light":
        usable = SEQUENTIAL_BLUE[3:]
    else:
        usable = list(reversed(SEQUENTIAL_BLUE[:-2]))
    if n == 1:
        picks = [usable[len(usable) // 2]]
    else:
        step = (len(usable) - 1) / (n - 1)
        picks = [usable[int(round(i * step))] for i in range(n)]
    return dict(zip(labels, picks))


def finish(ax, title: Optional[str] = None, subtitle: Optional[str] = None, mode: str = "light"):
    """Apply the shared title treatment and trim chart junk.

    Title and subtitle are offset in **points**, not axes fractions. An axes-fraction
    offset is a fraction of the plot's height, so on a tall figure the gap collapses and
    the two strings overlap -- which is exactly what happened on the 19-row forest plot.
    """
    c = chrome(mode)
    if title:
        ax.set_title(title, loc="left", pad=26 if subtitle else 12)
    if subtitle:
        ax.annotate(
            subtitle,
            xy=(0.0, 1.0), xycoords="axes fraction",
            xytext=(0, 8), textcoords="offset points",
            fontsize=8.5, color=c["secondary"], va="bottom", ha="left",
        )
    ax.tick_params(length=0)
    return ax


In [ ]:
# ── nifty_rl/report/figures.py ──────────────────────────────────────────────────────────
# Every chart, each one saving the table behind it alongside.
"""Figures for the results report.

Every function returns ``(figure, table)`` -- the chart and the data behind it. The table
is not an afterthought: a chart whose values are reachable only by reading pixels fails
accessibility, and the CSV twin is what makes each figure checkable.

Conventions enforced here:

* **Never two y-scales on one plot.** Two measures of different scale become two panels
  or an indexed common base. The original notebook's entry/exit chart used ``twinx()``,
  which invents an alignment between price and position that is not in the data.
* **Emphasis over rainbow.** When the story is one or two series, those get hue and the
  rest recede to gray.
* **Solid hairline grids**, recessive axes, thin marks, generous padding.
* **Selective direct labels** -- endpoints and extremes only, never a number on every
  point, with a legend whenever two or more series share a panel.
"""
from __future__ import annotations

from typing import Dict, Optional, Sequence, Tuple

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from matplotlib.patches import Patch


FigureAndTable = Tuple[Figure, pd.DataFrame]


def _format_dates(ax) -> None:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))


def _currency(ax, symbol: str = "₹") -> None:
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{symbol}{v:,.0f}"))


def _percent(ax, decimals: int = 0) -> None:
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.{decimals}f}%"))


# ------------------------------------------------------------------ equity curves


def equity_curves(
    equity_by_strategy: Dict[str, pd.Series],
    highlight: Sequence[str],
    initial_cash: float,
    mode: str = "light",
    title: str = "Equity curves — test period",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Equity paths with the strategies that matter in colour and the rest in gray."""
    apply_theme(mode)
    c = chrome(mode)
    colors = emphasis_colors(list(equity_by_strategy), highlight, mode)

    fig, ax = plt.subplots(figsize=(11, 5))

    # Context first so the highlighted paths sit on top.
    for name, equity in equity_by_strategy.items():
        if name in highlight:
            continue
        ax.plot(equity.index, equity.to_numpy(), color=colors[name], linewidth=1.0, alpha=0.55, zorder=2)

    for name in highlight:
        if name not in equity_by_strategy:
            continue
        equity = equity_by_strategy[name]
        ax.plot(equity.index, equity.to_numpy(), color=colors[name], linewidth=2.2, zorder=4, label=name)
        # Direct-label the endpoint only.
        ax.annotate(
            f" {name}",
            xy=(equity.index[-1], equity.iloc[-1]),
            color=colors[name],
            fontsize=8.5,
            va="center",
            fontweight="bold",
        )

    ax.axhline(initial_cash, color=c["axis"], linewidth=1.0, zorder=1)
    ax.annotate(
        "  initial capital",
        xy=(list(equity_by_strategy.values())[0].index[0], initial_cash),
        color=c["muted"], fontsize=8, va="bottom",
    )

    _format_dates(ax)
    _currency(ax)
    ax.set_ylabel("Portfolio value")
    handles = [plt.Line2D([], [], color=colors[n], linewidth=2.2, label=n) for n in highlight if n in equity_by_strategy]
    handles.append(plt.Line2D([], [], color=c["muted"], linewidth=1.0, label="other strategies"))
    ax.legend(handles=handles, loc="best", ncol=1)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()

    table = pd.DataFrame(equity_by_strategy)
    return fig, table


def drawdown_curves(
    equity_by_strategy: Dict[str, pd.Series],
    highlight: Sequence[str],
    mode: str = "light",
    title: str = "Drawdown — test period",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    apply_theme(mode)
    c = chrome(mode)
    colors = emphasis_colors(list(equity_by_strategy), highlight, mode)

    fig, ax = plt.subplots(figsize=(11, 3.6))
    table = {}
    for name, equity in equity_by_strategy.items():
        dd = (equity / equity.cummax() - 1.0) * 100.0
        table[name] = dd
        is_highlight = name in highlight
        ax.plot(
            dd.index, dd.to_numpy(),
            color=colors[name],
            linewidth=2.0 if is_highlight else 0.9,
            alpha=1.0 if is_highlight else 0.5,
            zorder=4 if is_highlight else 2,
            label=name if is_highlight else None,
        )

    ax.axhline(0, color=c["axis"], linewidth=1.0)
    _format_dates(ax)
    _percent(ax)
    ax.set_ylabel("Drawdown")
    handles = [plt.Line2D([], [], color=colors[n], linewidth=2.0, label=n) for n in highlight if n in equity_by_strategy]
    handles.append(plt.Line2D([], [], color=c["muted"], linewidth=0.9, label="other strategies"))
    ax.legend(handles=handles, loc="lower left")
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, pd.DataFrame(table)


# ---------------------------------------------------------------- regime timeline


def regime_timeline(
    market_equity: pd.Series,
    labels_by_detector: Dict[str, pd.Series],
    regime_names_by_detector: Dict[str, Sequence[str]],
    breakpoints: Optional[Sequence[pd.Timestamp]] = None,
    mode: str = "light",
    title: str = "Regime timeline — five detectors over the market path",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """The centrepiece figure: market path on top, one causal detector strip per row.

    Small multiples rather than overlays -- five regime paths on one axis would be
    unreadable, and putting them on a second y-scale against price would invent a
    relationship. Retrospective break dates are drawn as vertical rules across every
    panel so each detector's lag is visible directly.
    """
    apply_theme(mode)
    c = chrome(mode)
    n_detectors = len(labels_by_detector)

    fig, axes = plt.subplots(
        n_detectors + 1, 1,
        figsize=(11, 2.8 + 0.72 * n_detectors),
        sharex=True,
        gridspec_kw={"height_ratios": [3.2] + [0.75] * n_detectors, "hspace": 0.32},
    )
    price_ax = axes[0]

    normalised = market_equity / market_equity.iloc[0] * 100.0
    price_ax.plot(normalised.index, normalised.to_numpy(), color=c["primary"], linewidth=1.6)
    price_ax.set_ylabel("Market (=100)")
    price_ax.grid(True, axis="y")

    if breakpoints:
        for date in breakpoints:
            for ax in axes:
                ax.axvline(date, color=STATUS["critical"], linewidth=1.0, alpha=0.55, zorder=1)
        price_ax.annotate(
            "vertical rules = retrospective structural breaks (ground truth for detection lag)",
            xy=(0.0, -0.16), xycoords="axes fraction",
            fontsize=7.5, color=c["muted"],
        )

    rows = []
    for ax, (detector_name, labels) in zip(axes[1:], labels_by_detector.items()):
        names = list(regime_names_by_detector.get(detector_name, []))
        if not names:
            names = [f"R{i}" for i in range(int(labels.max()) + 1)]
        palette = regime_colors(names, mode)

        values = labels.to_numpy()
        dates = labels.index
        start = 0
        for i in range(1, len(values) + 1):
            if i == len(values) or values[i] != values[start]:
                regime_index = int(values[start])
                label_name = names[regime_index] if regime_index < len(names) else f"R{regime_index}"
                ax.axvspan(
                    dates[start],
                    dates[min(i, len(dates) - 1)],
                    color=palette[label_name],
                    linewidth=0,
                )
                rows.append(
                    {
                        "detector": detector_name,
                        "start": dates[start],
                        "end": dates[min(i, len(dates) - 1)],
                        "regime": label_name,
                        "days": i - start,
                    }
                )
                start = i

        ax.set_yticks([])
        ax.set_ylabel(detector_name, rotation=0, ha="right", va="center", fontsize=8.5, color=c["secondary"])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        handles = [Patch(facecolor=palette[n], label=n) for n in names]
        ax.legend(
            handles=handles, loc="center left", bbox_to_anchor=(1.005, 0.5),
            fontsize=7, handlelength=1.0, handleheight=0.9, borderpad=0.2, labelspacing=0.25,
        )

    _format_dates(axes[-1])
    finish(price_ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, pd.DataFrame(rows)


def regime_transition_heatmap(
    transitions: pd.DataFrame,
    mode: str = "light",
    title: str = "Regime transition matrix",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Transition probabilities. Magnitude, so a single-hue sequential ramp."""
    apply_theme(mode)
    c = chrome(mode)

    fig, ax = plt.subplots(figsize=(4.6, 3.9))
    data = transitions.to_numpy()
    image = ax.imshow(data, cmap=sequential_cmap(mode), vmin=0, vmax=1, aspect="auto")

    ax.set_xticks(range(len(transitions.columns)), transitions.columns, fontsize=8)
    ax.set_yticks(range(len(transitions.index)), transitions.index, fontsize=8)
    ax.set_xlabel("to")
    ax.set_ylabel("from")
    ax.grid(False)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            ax.text(
                j, i, f"{value:.2f}",
                ha="center", va="center", fontsize=8,
                color="#ffffff" if value > 0.55 else c["primary"],
            )

    bar = fig.colorbar(image, ax=ax, shrink=0.8)
    bar.outline.set_visible(False)
    bar.set_label("probability", fontsize=8)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, transitions


def agreement_heatmap(
    matrix: pd.DataFrame,
    mode: str = "light",
    title: str = "Cross-detector agreement (Cohen's κ)",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    apply_theme(mode)
    c = chrome(mode)

    fig, ax = plt.subplots(figsize=(5.2, 4.3))
    data = matrix.to_numpy(dtype=float)
    image = ax.imshow(data, cmap=sequential_cmap(mode), vmin=0, vmax=1, aspect="auto")

    ax.set_xticks(range(len(matrix.columns)), matrix.columns, fontsize=8, rotation=30, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index, fontsize=8)
    ax.grid(False)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            if not np.isfinite(value):
                continue
            ax.text(
                j, i, f"{value:.2f}",
                ha="center", va="center", fontsize=8,
                color="#ffffff" if value > 0.55 else c["primary"],
            )

    bar = fig.colorbar(image, ax=ax, shrink=0.8)
    bar.outline.set_visible(False)
    bar.set_label("κ", fontsize=8)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, matrix


def regime_validation_panel(
    persistence: pd.DataFrame,
    lag_by_detector: Dict[str, dict],
    min_run_days: float = 10.0,
    mode: str = "light",
    title: str = "Regime model validation",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Persistence and detection lag side by side, each against its gate.

    These two numbers decide whether the regime layer is usable at all. A model that
    flips every few days is untradeable after costs; a model that recognises a crisis
    weeks late adds cost and no protection. Both gates are drawn on the chart, and the
    status colours carry a label so meaning never rests on hue.
    """
    apply_theme(mode)
    c = chrome(mode)
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))

    # --- mean run length vs the tradeability floor
    names = persistence["regime"].tolist()
    runs = persistence["mean_run_days"].to_numpy()
    colors = [STATUS["good"] if v >= min_run_days else STATUS["critical"] for v in runs]
    bars = left.barh(names, runs, color=colors, height=0.6)
    left.axvline(min_run_days, color=c["primary"], linewidth=1.2)
    left.annotate(
        f" tradeability floor ({min_run_days:.0f}d)",
        xy=(min_run_days, len(names) - 0.4), color=c["secondary"], fontsize=8, va="top",
    )
    for bar, value in zip(bars, runs):
        left.annotate(
            f" {value:.0f}d", xy=(value, bar.get_y() + bar.get_height() / 2),
            va="center", fontsize=8, color=c["secondary"],
        )
    left.set_xlabel("Mean run length (days)")
    left.grid(True, axis="x")
    left.set_axisbelow(True)
    finish(left, "Persistence", "pass = regime lasts long enough to trade", mode)

    # --- median detection lag
    detectors = list(lag_by_detector)
    medians = [lag_by_detector[d].get("median_lag", np.nan) for d in detectors]
    rates = [lag_by_detector[d].get("detection_rate", np.nan) for d in detectors]
    lag_colors = [
        STATUS["good"] if (np.isfinite(m) and m <= 10) else
        STATUS["warning"] if (np.isfinite(m) and m <= 20) else STATUS["critical"]
        for m in medians
    ]
    bars = right.barh(detectors, [0 if not np.isfinite(m) else m for m in medians], color=lag_colors, height=0.6)
    for bar, median, rate in zip(bars, medians, rates):
        text = "no detection" if not np.isfinite(median) else f" {median:.0f}d  ({rate:.0%} found)"
        right.annotate(
            text, xy=(0 if not np.isfinite(median) else median, bar.get_y() + bar.get_height() / 2),
            va="center", fontsize=8, color=c["secondary"],
        )
    right.set_xlabel("Median detection lag (days)")
    right.grid(True, axis="x")
    right.set_axisbelow(True)
    finish(right, "Detection lag", "green ≤ 10d  ·  amber ≤ 20d  ·  red slower", mode)

    fig.tight_layout()
    table = persistence.copy()
    return fig, table


# ------------------------------------------------------------- honest leaderboard


def sharpe_forest(
    significance: pd.DataFrame,
    mode: str = "light",
    title: str = "Sharpe with bootstrap confidence intervals",
    subtitle: Optional[str] = "Interval = stationary-block bootstrap 95%. Dashed rule = deflated-Sharpe threshold for this many trials.",
) -> FigureAndTable:
    """Dot-and-interval leaderboard.

    A bare ranked table of point estimates invites a conclusion ~220 trading days cannot
    support. Showing the interval makes the overlap obvious, and the deflation threshold
    marks the bar the *best* of many trials actually has to clear.
    """
    apply_theme(mode)
    c = chrome(mode)
    frame = significance.sort_values("sharpe", na_position="first").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(9, 0.42 * len(frame) + 2.4))
    y = np.arange(len(frame))

    for i, row in frame.iterrows():
        lower, upper = row["sharpe_ci_lower"], row["sharpe_ci_upper"]
        if np.isfinite(lower) and np.isfinite(upper):
            ax.plot([lower, upper], [i, i], color=c["axis"], linewidth=2.0, solid_capstyle="round", zorder=2)
        elif not np.isfinite(row["sharpe"]):
            # A portfolio that never left cash has an undefined Sharpe. Say so on the
            # chart rather than leaving a blank row the reader has to interpret.
            ax.annotate(
                "never invested — Sharpe undefined",
                xy=(0, i), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=8, color=c["muted"], style="italic",
            )

    significant = frame["ci_excludes_zero"].to_numpy()
    benchmark = frame["is_benchmark"].to_numpy()
    point_colors = [
        categorical(mode)[1] if b else (categorical(mode)[0] if s else c["muted"])
        for s, b in zip(significant, benchmark)
    ]
    ax.scatter(frame["sharpe"], y, s=64, color=point_colors, zorder=4, edgecolor=chrome(mode)["surface"], linewidth=1.5)

    ax.axvline(0, color=c["primary"], linewidth=1.1, zorder=1)
    threshold = frame["deflation_threshold"].dropna()
    if len(threshold):
        ax.axvline(float(threshold.iloc[0]), color=STATUS["critical"], linewidth=1.2, linestyle=(0, (4, 3)), zorder=1)

    ax.set_yticks(y, frame["strategy"], fontsize=8.5)
    ax.set_xlabel("Annualised Sharpe")
    ax.grid(True, axis="x")
    ax.set_axisbelow(True)

    handles = [
        plt.Line2D([], [], marker="o", linestyle="", color=categorical(mode)[1], label="benchmark", markersize=8),
        plt.Line2D([], [], marker="o", linestyle="", color=categorical(mode)[0], label="CI excludes zero", markersize=8),
        plt.Line2D([], [], marker="o", linestyle="", color=c["muted"], label="indistinguishable from zero", markersize=8),
    ]
    ax.legend(handles=handles, loc="lower right")
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, significance


def regime_performance_heatmap(
    performance: pd.DataFrame,
    value_column: str = "sharpe",
    mode: str = "light",
    title: str = "Strategy performance by regime",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Strategy x regime grid. Polarity matters, so a diverging ramp around zero.

    This is the table the whole project was missing: it answers whether a
    drawdown-penalised agent actually earns its penalty when conditions are bad, rather
    than reporting one blended number over a window that happened to be a drawdown.
    """
    apply_theme(mode)
    c = chrome(mode)
    grid = performance.pivot(index="strategy", columns="regime", values=value_column)

    # Regimes are ordinal (calm -> crisis). Pivot returns them alphabetically, which puts
    # "Crisis" second and makes the gradient across the row meaningless.
    if "regime_index" in performance.columns:
        order = (
            performance.drop_duplicates("regime")
            .sort_values("regime_index")["regime"]
            .tolist()
        )
        grid = grid[[r for r in order if r in grid.columns]]

    magnitude = float(np.nanmax(np.abs(grid.to_numpy()))) if grid.size else 1.0
    magnitude = magnitude if np.isfinite(magnitude) and magnitude > 0 else 1.0

    fig, ax = plt.subplots(figsize=(1.5 * len(grid.columns) + 4.0, 0.42 * len(grid.index) + 2.2))
    image = ax.imshow(grid.to_numpy(), cmap=diverging_cmap(mode), vmin=-magnitude, vmax=magnitude, aspect="auto")

    ax.set_xticks(range(len(grid.columns)), grid.columns, fontsize=8.5)
    ax.set_yticks(range(len(grid.index)), grid.index, fontsize=8.5)
    ax.grid(False)

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            value = grid.to_numpy()[i, j]
            if not np.isfinite(value):
                continue
            ax.text(
                j, i, f"{value:.2f}",
                ha="center", va="center", fontsize=8,
                color="#ffffff" if abs(value) > 0.62 * magnitude else c["primary"],
            )

    bar = fig.colorbar(image, ax=ax, shrink=0.85)
    bar.outline.set_visible(False)
    bar.set_label(value_column, fontsize=8)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, grid


def risk_return_scatter(
    metrics: pd.DataFrame,
    highlight: Sequence[str],
    mode: str = "light",
    title: str = "Risk and return — test period",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Scatter compares every pair of colours at once, so highlights are capped at three."""
    apply_theme(mode)
    c = chrome(mode)
    capped = list(highlight)[:ALL_PAIRS_SAFE_SLOTS]
    colors = emphasis_colors(metrics["strategy"].tolist(), capped, mode)

    fig, ax = plt.subplots(figsize=(7.6, 5.4))
    for _, row in metrics.iterrows():
        name = row["strategy"]
        is_highlight = name in capped
        ax.scatter(
            row["annual_volatility"] * 100.0,
            row["total_return"] * 100.0,
            s=150 if is_highlight else 46,
            color=colors[name],
            zorder=4 if is_highlight else 2,
            edgecolor=chrome(mode)["surface"],
            linewidth=1.6,
            label=name if is_highlight else None,
        )
        if is_highlight:
            ax.annotate(
                f"  {name}",
                xy=(row["annual_volatility"] * 100.0, row["total_return"] * 100.0),
                fontsize=8.5, color=colors[name], va="center", fontweight="bold",
            )

    ax.axhline(0, color=c["primary"], linewidth=1.1)
    ax.set_xlabel("Annualised volatility (%)")
    ax.set_ylabel("Total return (%)")
    ax.legend(loc="best")
    ax.set_axisbelow(True)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, metrics[["strategy", "annual_volatility", "total_return"]]


def allocation_area(
    weights: pd.DataFrame,
    mode: str = "light",
    title: str = "Portfolio allocation",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Realised weights through time.

    The notebook's 'allocation stack' plotted an inline momentum proxy rather than the
    agent's weights, because the environment never logged them. This one takes the
    logged weights, so it shows what the policy actually did.
    """
    apply_theme(mode)

    columns = list(weights.columns)
    if len(columns) > 8:
        ranked = weights.mean().sort_values(ascending=False)
        keep = list(ranked.index[:7])
        folded = weights[keep].copy()
        folded["Other"] = weights.drop(columns=keep).sum(axis=1)
        weights = folded
        columns = list(weights.columns)

    colors = series_colors(columns, mode)
    fig, ax = plt.subplots(figsize=(11, 4.4))
    ax.stackplot(
        weights.index,
        [weights[col].to_numpy() for col in columns],
        colors=[colors[col] for col in columns],
        labels=columns,
        linewidth=0.8,
        edgecolor=chrome(mode)["surface"],
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("Weight")
    _format_dates(ax)
    ax.grid(True, axis="y")
    ax.legend(loc="center left", bbox_to_anchor=(1.005, 0.5), fontsize=8)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, weights


def save(fig: Figure, table: pd.DataFrame, path, also_csv: bool = True) -> None:
    """Write the figure and, beside it, the table that makes its values readable."""
    from pathlib import Path

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    if also_csv and table is not None and not table.empty:
        table.to_csv(path.with_suffix(".csv"))
    _show_inline(fig, path)
    plt.close(fig)


def _show_inline(fig: Figure, path=None) -> None:
    """Draw the figure in the notebook as well as writing it to disk.

    Without this the run produces thirteen files and nothing visible, because `save`
    closes each figure the moment it is written. Silently does nothing outside a
    notebook, so the same code still works headless.
    """
    try:
        from IPython import get_ipython
        from IPython.display import Markdown, display

        if get_ipython() is None:
            return
        if path is not None:
            # The _dark variants are for dark-background slides. Drawing both doubles
            # the scrolling and shows the same chart twice, so only the light one is
            # displayed -- both are still written to disk.
            if path.stem.endswith("_dark"):
                return
            display(Markdown(f"**{path.stem.replace('_', ' ')}**  \n`{path}`"))
        display(fig)
    except Exception:
        # Displaying is a convenience. It must never be the reason a run fails.
        pass


# ------------------------------------------------------------------- walk-forward


def window_returns_heatmap(
    per_window: pd.DataFrame,
    mode: str = "light",
    title: str = "Out-of-sample return by walk-forward window",
    subtitle: Optional[str] = None,
) -> FigureAndTable:
    """Strategy x window returns — the figure a single split cannot produce.

    Reading across a row shows whether a strategy is consistent or whether its headline
    number came from one lucky block. Reading down a column shows which windows were hard
    for everything, which is the context a single test period silently hides.
    """
    apply_theme(mode)
    c = chrome(mode)
    grid = per_window.pivot(index="strategy", columns="window", values="total_return") * 100.0
    grid = grid.loc[grid.mean(axis=1).sort_values(ascending=False).index]

    magnitude = float(np.nanmax(np.abs(grid.to_numpy()))) if grid.size else 1.0
    magnitude = magnitude if np.isfinite(magnitude) and magnitude > 0 else 1.0

    fig, ax = plt.subplots(
        figsize=(1.05 * len(grid.columns) + 5.0, 0.40 * len(grid.index) + 2.4)
    )
    image = ax.imshow(
        grid.to_numpy(), cmap=diverging_cmap(mode), vmin=-magnitude, vmax=magnitude, aspect="auto"
    )

    ax.set_xticks(range(len(grid.columns)), [f"W{c_}" for c_ in grid.columns], fontsize=8.5)
    ax.set_yticks(range(len(grid.index)), grid.index, fontsize=8.5)
    ax.grid(False)

    values = grid.to_numpy()
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            value = values[i, j]
            if not np.isfinite(value):
                continue
            ax.text(
                j, i, f"{value:.1f}",
                ha="center", va="center", fontsize=7.5,
                color="#ffffff" if abs(value) > 0.62 * magnitude else c["primary"],
            )

    bar = fig.colorbar(image, ax=ax, shrink=0.85)
    bar.outline.set_visible(False)
    bar.set_label("return (%)", fontsize=8)
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, grid


def consistency_scatter(
    summary: pd.DataFrame,
    highlight: Sequence[str],
    mode: str = "light",
    title: str = "Consistency versus pooled performance",
    subtitle: Optional[str] = "A high pooled Sharpe from one good window is not a strategy.",
) -> FigureAndTable:
    """Share of windows profitable against pooled out-of-sample Sharpe."""
    apply_theme(mode)
    c = chrome(mode)
    capped = list(highlight)[:ALL_PAIRS_SAFE_SLOTS]
    colors = emphasis_colors(summary["strategy"].tolist(), capped, mode)

    fig, ax = plt.subplots(figsize=(7.8, 5.4))
    for _, row in summary.iterrows():
        name = row["strategy"]
        is_highlight = name in capped
        ax.scatter(
            row["windows_positive"] * 100.0,
            row["pooled_sharpe"],
            s=150 if is_highlight else 44,
            color=colors[name],
            zorder=4 if is_highlight else 2,
            edgecolor=chrome(mode)["surface"],
            linewidth=1.6,
        )
        if is_highlight:
            ax.annotate(
                f"  {name}",
                xy=(row["windows_positive"] * 100.0, row["pooled_sharpe"]),
                fontsize=8.5, color=colors[name], va="center", fontweight="bold",
            )

    ax.axhline(0, color=c["primary"], linewidth=1.1)
    ax.axvline(50, color=c["axis"], linewidth=1.0)
    ax.set_xlabel("Windows with a positive return (%)")
    ax.set_ylabel("Pooled out-of-sample Sharpe")
    ax.set_axisbelow(True)
    handles = [
        plt.Line2D([], [], marker="o", linestyle="", color=colors[n], label=n, markersize=8)
        for n in capped
    ]
    handles.append(
        plt.Line2D([], [], marker="o", linestyle="", color=c["muted"], label="other strategies", markersize=7)
    )
    ax.legend(handles=handles, loc="best")
    finish(ax, title, subtitle, mode)
    fig.tight_layout()
    return fig, summary[["strategy", "windows_positive", "pooled_sharpe"]]


def ppo_seed_dispersion(
    per_window: pd.DataFrame,
    benchmark: str = "BuyHold",
    mode: str = "light",
    title: str = "PPO seed dispersion by window",
    subtitle: Optional[str] = "Every seed is an independent training run on the same data.",
) -> FigureAndTable:
    """Per-window PPO seed spread against the benchmark.

    The original project reported a single seed and listed that as a limitation. Showing
    the spread answers the question that limitation raises: is the agent's result a draw
    from a wide distribution, or is the policy genuinely reproducible? Tight dispersion
    means an underperforming agent is underperforming for real reasons, not variance.
    """
    apply_theme(mode)
    c = chrome(mode)

    seed_rows = per_window[per_window["strategy"].str.startswith("PPO_s")]
    if seed_rows.empty:
        raise ValueError("no PPO seed rows in per_window")

    windows = sorted(seed_rows["window"].unique())
    fig, ax = plt.subplots(figsize=(1.05 * len(windows) + 3.6, 4.6))
    palette = categorical(mode)

    for window in windows:
        block = seed_rows[seed_rows["window"] == window]["total_return"] * 100.0
        lo, hi = block.min(), block.max()
        ax.plot([window, window], [lo, hi], color=c["axis"], linewidth=2.0,
                solid_capstyle="round", zorder=2)
        ax.scatter([window] * len(block), block, s=42, color=palette[0],
                   zorder=4, edgecolor=chrome(mode)["surface"], linewidth=1.2)

    ensemble = per_window[per_window["strategy"] == "PPO_ensemble"]
    if not ensemble.empty:
        ax.plot(ensemble["window"], ensemble["total_return"] * 100.0,
                color=palette[0], linewidth=1.4, alpha=0.5, zorder=3, label="PPO ensemble")

    bench = per_window[per_window["strategy"] == benchmark]
    if not bench.empty:
        ax.plot(bench["window"], bench["total_return"] * 100.0, color=palette[1],
                linewidth=2.2, marker="o", markersize=6, zorder=5, label=benchmark)

    ax.axhline(0, color=c["primary"], linewidth=1.1)
    ax.set_xticks(windows, [f"W{w}" for w in windows])
    ax.set_xlabel("Walk-forward window")
    ax.set_ylabel("Out-of-sample return (%)")
    ax.set_axisbelow(True)

    handles = [
        plt.Line2D([], [], marker="o", linestyle="", color=palette[0],
                   label="individual seeds", markersize=7),
        plt.Line2D([], [], color=palette[1], marker="o", linewidth=2.2,
                   label=benchmark, markersize=6),
    ]
    ax.legend(handles=handles, loc="best")
    finish(ax, title, subtitle, mode)
    fig.tight_layout()

    table = (
        seed_rows.groupby("window")["total_return"]
        .agg(["min", "mean", "max", "std"])
        .rename(columns={"std": "seed_std"})
    )
    return fig, table


In [ ]:
# A small piece of glue. The driver below was written against the package, where it
# says things like figures.equity_curves(...) — with `figures` being a module. Flattened
# into one namespace there is no module to reach through, so we build one here and point
# it at everything defined so far. Saves rewriting a dozen call sites.
import types as _types

figures = _types.ModuleType("figures")
figures.__dict__.update({k: v for k, v in globals().items() if not k.startswith("__")})


In [ ]:
# ── nifty_rl/report/narrate.py ──────────────────────────────────────────────────────────
# Plain-English commentary on each regime episode. Presentation only — it never feeds
# back into anything.
"""Regime narration — commentary attached to detected episodes, for readers.

The detector says `Crisis, 2022-06-13 to 2022-07-08`. True, causal, and opaque. This
module attaches *what was happening* so a reader can judge whether the label is
plausible, without having to remember four years of market history.

**This is presentation, and the boundary is enforced rather than intended.**

* Narration is generated **after** the fact, from the completed episode table.
* It is written to its own artefact (`results/regime_narration.csv`) and never joined
  into the feature panel, the observation vector, or any model input.
* :func:`assert_no_narration_leak` is called by the test suite against the real feature
  frame, so a future edit that pipes commentary into a feature fails CI.

The reason for that severity: an LLM's weights encode what happened *after* the period
being narrated. Feeding its output into a model is lookahead of a kind no prefix test can
catch, because the contamination lives inside a model you did not train. As commentary it
is harmless and useful. As a feature it would silently invalidate every result in the
project.

Without a provider the module still works: :func:`describe_from_data` produces a factual
description computed from the episode's own statistics, with no model involved at all.
That is the default, and it is deterministic.

To enable model commentary, pass a provider -- :func:`openrouter_provider` is included.
It is opt-in, and the pipeline runs identically without a key.
"""
from __future__ import annotations

import json
import os
import time
import urllib.error
import urllib.request
from dataclasses import dataclass
from typing import Callable, List, Optional, Sequence

import numpy as np
import pandas as pd

#: A provider takes a prompt and returns prose. Any LLM client can be adapted to this.
NarrationProvider = Callable[[str], str]

FORBIDDEN_COLUMNS = ("narration", "regime_narration", "commentary", "llm_note")

#: Environment variable holding the OpenRouter API key. Never passed on the command line,
#: so it cannot end up in shell history or a process listing.
OPENROUTER_KEY_ENV = "OPENROUTER_API_KEY"
OPENROUTER_MODEL_ENV = "OPENROUTER_MODEL"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

#: OpenRouter's catalogue changes over time; override with ``OPENROUTER_MODEL`` or the
#: ``model=`` argument if this identifier is retired.
DEFAULT_OPENROUTER_MODEL = "anthropic/claude-sonnet-4.5"


@dataclass
class RegimeEpisode:
    """One contiguous run of a single regime, with the market context that defines it."""

    regime: str
    start: pd.Timestamp
    end: pd.Timestamp
    n_days: int
    market_return: float
    volatility: float
    max_drawdown: float
    best_day: float
    worst_day: float

    def to_dict(self) -> dict:
        return {
            "regime": self.regime,
            "start": self.start,
            "end": self.end,
            "n_days": self.n_days,
            "market_return": self.market_return,
            "volatility": self.volatility,
            "max_drawdown": self.max_drawdown,
            "best_day": self.best_day,
            "worst_day": self.worst_day,
        }


def extract_episodes(
    labels: pd.Series,
    market_returns: pd.Series,
    regime_names: Optional[Sequence[str]] = None,
    min_days: int = 5,
    trading_days: int = 252,
) -> pd.DataFrame:
    """Collapse a regime label path into episodes with their realised market statistics.

    Useful on its own, before any narration: a table of "here are the twelve crisis
    episodes and what the market did in each" is a far more checkable object than a
    coloured strip.
    """
    aligned = pd.concat(
        [labels.rename("regime"), market_returns.rename("ret")], axis=1, join="inner"
    ).dropna()
    if aligned.empty:
        return pd.DataFrame()

    values = aligned["regime"].to_numpy()
    boundaries = np.flatnonzero(np.diff(values)) + 1
    starts = np.concatenate([[0], boundaries])
    ends = np.concatenate([boundaries, [len(values)]])

    episodes: List[RegimeEpisode] = []
    for lo, hi in zip(starts, ends):
        if hi - lo < min_days:
            continue
        block = aligned.iloc[lo:hi]
        returns = block["ret"]
        equity = (1.0 + returns).cumprod()
        index = int(block["regime"].iloc[0])
        name = (
            regime_names[index]
            if regime_names is not None and index < len(regime_names)
            else f"regime_{index}"
        )
        episodes.append(
            RegimeEpisode(
                regime=name,
                start=block.index[0],
                end=block.index[-1],
                n_days=len(block),
                market_return=float(equity.iloc[-1] - 1.0),
                volatility=float(returns.std() * np.sqrt(trading_days)),
                max_drawdown=float((equity / equity.cummax() - 1.0).min()),
                best_day=float(returns.max()),
                worst_day=float(returns.min()),
            )
        )

    return pd.DataFrame([e.to_dict() for e in episodes])


def describe_from_data(episode: pd.Series) -> str:
    """Factual description computed from the episode's own numbers. No model involved.

    This is the default narration. It cannot be wrong about the market because it only
    restates what the data says, and it makes the LLM path optional rather than load-bearing.
    """
    direction = "rose" if episode["market_return"] >= 0 else "fell"
    return (
        f"{episode['n_days']} trading days. The market {direction} "
        f"{abs(episode['market_return']):.1%} at {episode['volatility']:.0%} annualised "
        f"volatility, with a peak-to-trough drawdown of {abs(episode['max_drawdown']):.1%} "
        f"(worst day {episode['worst_day']:.1%}, best {episode['best_day']:+.1%})."
    )


def build_prompt(episode: pd.Series, market: str = "Indian equities (NIFTY 50)") -> str:
    """Prompt asking for factual context, not interpretation.

    Deliberately narrow. It supplies the dates and the realised statistics and asks only
    what was happening in the market — a recall task. It does not ask whether the regime
    label is correct, what to do about it, or what happens next; the moment commentary
    starts making claims about the future it stops being presentation.
    """
    return (
        f"In two sentences, state the main market and macroeconomic events affecting "
        f"{market} between {episode['start']:%d %B %Y} and {episode['end']:%d %B %Y}.\n\n"
        f"Realised over that window: return {episode['market_return']:+.1%}, "
        f"annualised volatility {episode['volatility']:.0%}, "
        f"maximum drawdown {abs(episode['max_drawdown']):.1%}.\n\n"
        f"Report only what occurred. Do not evaluate any trading strategy, do not "
        f"forecast, and do not comment on whether the period was a good time to invest. "
        f"If you are not confident about the events, say so instead of guessing."
    )


# ------------------------------------------------------------------ LLM provider


def openrouter_provider(
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    timeout: float = 30.0,
    max_tokens: int = 220,
    temperature: float = 0.0,
    max_retries: int = 2,
    backoff: float = 2.0,
) -> NarrationProvider:
    """Build a :data:`NarrationProvider` backed by OpenRouter.

    Uses :mod:`urllib` from the standard library rather than an SDK. Commentary is a
    presentation nicety, and it should not add a dependency that everyone installing this
    project has to carry -- particularly since CI runs with no network at all.

    The key is read from ``$OPENROUTER_API_KEY`` and never accepted as a CLI argument, so
    it stays out of shell history and ``ps`` output. A missing key raises **here**, at
    construction, rather than during the run: 45 episodes silently falling back to the
    data description would look like success.

    ``temperature=0`` because this is a research artefact. Two runs over the same
    episodes should produce the same report, and sampling would make the commentary
    churn on every rebuild for no benefit.

    Transient failures (429, 5xx, connection drops) are retried with exponential backoff.
    Anything still failing after that propagates to :func:`narrate`, which falls back to
    the data description for that episode and marks its ``source`` as ``"data"`` -- so a
    partial outage degrades one row rather than the report.
    """
    key = api_key or os.environ.get(OPENROUTER_KEY_ENV)
    if not key:
        raise RuntimeError(
            f"{OPENROUTER_KEY_ENV} is not set. Export it to enable LLM narration, or "
            "omit the flag to use the deterministic data-only descriptions."
        )
    chosen = model or os.environ.get(OPENROUTER_MODEL_ENV) or DEFAULT_OPENROUTER_MODEL

    def provider(prompt: str) -> str:
        payload = json.dumps(
            {
                "model": chosen,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens,
                "temperature": temperature,
            }
        ).encode("utf-8")

        last_error: Optional[Exception] = None
        for attempt in range(max_retries + 1):
            request = urllib.request.Request(
                OPENROUTER_URL,
                data=payload,
                headers={
                    "Authorization": f"Bearer {key}",
                    "Content-Type": "application/json",
                    # OpenRouter uses these for attribution on its dashboard; harmless
                    # and it keeps the traffic identifiable as coming from this project.
                    "HTTP-Referer": "https://github.com/tridibjena/nifty50-rl-portfolio-optimization",
                    "X-Title": "nifty50-regime-narration",
                },
                method="POST",
            )
            try:
                with urllib.request.urlopen(request, timeout=timeout) as response:
                    body = json.loads(response.read().decode("utf-8"))
                return str(body["choices"][0]["message"]["content"]).strip()
            except urllib.error.HTTPError as exc:
                last_error = exc
                # 4xx other than rate-limiting means the request itself is wrong --
                # a bad model id or a revoked key. Retrying cannot fix it.
                if exc.code != 429 and exc.code < 500:
                    raise
            except (urllib.error.URLError, TimeoutError, KeyError, ValueError) as exc:
                last_error = exc

            if attempt < max_retries:
                time.sleep(backoff ** attempt)

        raise RuntimeError(f"OpenRouter request failed after {max_retries + 1} attempts") from last_error

    return provider


def narrate(
    episodes: pd.DataFrame,
    provider: Optional[NarrationProvider] = None,
    market: str = "Indian equities (NIFTY 50)",
) -> pd.DataFrame:
    """Attach commentary to each episode.

    With no provider, every row is described from its own statistics and ``source`` is
    ``"data"``. With a provider, ``source`` is ``"llm"`` and a failed call falls back to
    the data description rather than dropping the row -- the report should never depend
    on a network call succeeding.
    """
    if episodes.empty:
        return episodes.assign(narration=pd.Series(dtype=str), source=pd.Series(dtype=str))

    narrations, sources = [], []
    for _, episode in episodes.iterrows():
        fallback = describe_from_data(episode)
        if provider is None:
            narrations.append(fallback)
            sources.append("data")
            continue
        try:
            text = provider(build_prompt(episode, market)).strip()
            narrations.append(f"{fallback} {text}" if text else fallback)
            sources.append("llm" if text else "data")
        except Exception:
            narrations.append(fallback)
            sources.append("data")

    return episodes.assign(narration=narrations, source=sources)


def to_markdown(narrated: pd.DataFrame, regime_order: Optional[Sequence[str]] = None) -> str:
    """Render narrated episodes as a readable timeline for the report."""
    if narrated.empty:
        return "_no episodes met the minimum length_"

    frame = narrated.sort_values("start")
    lines: List[str] = []
    for _, row in frame.iterrows():
        lines.append(
            f"**{row['regime']}** · {row['start']:%d %b %Y} → {row['end']:%d %b %Y}  \n"
            f"{row['narration']}\n"
        )
    if "source" in frame.columns and (frame["source"] == "llm").any():
        lines.append(
            "\n*Commentary is generated after the fact for readability. It is written to "
            "its own artefact and never enters any feature, observation or model input.*"
        )
    return "\n".join(lines)


def assert_no_narration_leak(frame: pd.DataFrame, context: str = "feature panel") -> None:
    """Fail loudly if commentary has found its way into modelling data.

    Called by the test suite against the real feature frame. The point is that a future
    edit which merges narration into features breaks the build rather than quietly
    producing better-looking and completely invalid results.
    """
    present = [c for c in frame.columns if c.lower() in FORBIDDEN_COLUMNS]
    if present:
        raise AssertionError(
            f"Narration column(s) {present} found in the {context}. Commentary is "
            "presentation only -- an LLM's weights encode what happened after the period "
            "it describes, so using it as a feature is lookahead that no prefix test can "
            "detect."
        )


## Putting it to work

That is the whole library. What follows is the script that uses it — load the data, fit the regime models, check they are worth trusting, run every strategy through the walk-forward, test what survives, and write it all out.

In [ ]:
# ── the driver ───────────────────────────────────────────────────────────────────────
# Everything above was definitions. This is the part that actually does something:
# load the data, fit the regimes, run the walk-forward, work out what survives,
# draw the charts, write the report.
from __future__ import annotations

"""End-to-end pipeline: data -> regimes -> rolling walk-forward -> figures -> report.

Run with::

    python3 scripts/run_pipeline.py

Evaluation is **rolling walk-forward**, not a single chronological split. A 70/15/15 cut
yields exactly one out-of-sample window, and whichever regime that window lands in *is*
the result -- change the ratios and the ranking changes with them. Here the model refits
on an expanding history, trades the next block with parameters frozen, and rolls forward;
the out-of-sample blocks are chained into one continuous track record.
"""


import argparse
import sys
import warnings
from dataclasses import replace
from pathlib import Path

PROJECT_ROOT = _PROJECT_ROOT


import pandas as pd

warnings.filterwarnings("ignore")

try:  # keep the RL stack optional
    import torch

    torch.set_num_threads(1)
except ImportError:
    pass


ASSETS = PROJECT_ROOT / "assets" / "v2"
RESULTS = PROJECT_ROOT / "results"

RULE_STRATEGIES = {
    "BuyHold": make_signal_fn("buy_hold"),
    "MA_20_50": make_signal_fn("ma", short=20, long=50),
    "RSI_35_60": make_signal_fn("rsi", buy_below=35, sell_above=60),
    "Breakout_20": make_signal_fn("breakout", lookback=20, exit_lookback=10),
    "MomentumPullback": make_signal_fn("momentum_pullback", rsi_max=55),
    "VIX_Regime_Mom": make_signal_fn("vix_regime_momentum"),
    "Sentiment_Momentum": make_signal_fn("sentiment_momentum"),
    "Random": make_random_signal_fn(seed=42),
}

# Observation features for the RL agent. 'technical_vix' from the original notebook --
# 15 per ticker, so a 172-dimensional observation with the availability flags, weights
# and cash ratio.
RL_FEATURES = [
    "ret", "ma_ratio", "trend_20_50", "rsi", "macd_hist", "bb_position", "bb_width",
    "atr_pct", "momentum_5", "momentum_20", "volume_change",
    "india_vix", "vix_change", "high_vix_regime", "dispersion_zscore",
]

RL_SEEDS = (0, 1, 2)
RL_TIMESTEPS = 60_000

REGIME_FEATURE_COLUMNS = [
    "realized_vol_21",
    "trend_21",
    "mean_correlation",
    "dispersion",
    "breadth",
]

TRAIN_DAYS, TEST_DAYS, STEP_DAYS = 500, 125, 125


def log(message: str) -> None:
    print(message, flush=True)


def parse_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Regime-aware walk-forward portfolio pipeline.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--no-rl", action="store_true",
        help="Skip PPO training. Runs in ~90s instead of ~20min; everything else is identical.",
    )
    parser.add_argument(
        "--seeds", type=int, default=len(RL_SEEDS),
        help="Number of PPO seeds per walk-forward window.",
    )
    parser.add_argument(
        "--timesteps", type=int, default=RL_TIMESTEPS,
        help="PPO training steps per seed per window.",
    )
    parser.add_argument(
        "--capital", type=float, default=None,
        help="Starting capital in rupees. Changes real results, not just the scale: "
             "integer share sizing leaves proportionally less idle cash at larger sizes.",
    )
    parser.add_argument(
        "--live", action="store_true",
        help="Ignore the pinned end_date and pull data up to today. Results will "
             "no longer match the published figures.",
    )
    parser.add_argument(
        "--train-days", type=int, default=TRAIN_DAYS,
        help="Initial training block length, in trading days.",
    )
    parser.add_argument(
        "--narrate-llm", action="store_true",
        help="Attach LLM commentary to each regime episode via OpenRouter. Requires "
             "$OPENROUTER_API_KEY. Presentation only -- it never enters a feature or a "
             "model input, and it does not change a single number in the results.",
    )
    parser.add_argument(
        "--narrate-model", default=None,
        help="OpenRouter model id for --narrate-llm. Defaults to $OPENROUTER_MODEL, "
             "then to the built-in default.",
    )
    parser.add_argument(
        "--test-days", type=int, default=TEST_DAYS,
        help="Out-of-sample block length, in trading days.",
    )
    return parser.parse_args(argv)


def main(args: argparse.Namespace = None) -> None:
    args = args or parse_args([])
    cfg = RunConfig()
    if args.capital:
        cfg = cfg.with_(backtest=replace(cfg.backtest, initial_cash=args.capital))
        log(f"      capital: Rs{args.capital:,.0f}")
    if args.live:
        cfg = cfg.with_(data=replace(cfg.data, end_date=None))
        log("      NOTE: --live ignores the pinned snapshot; results will drift.")
    train_days, test_days, step_days = args.train_days, args.test_days, args.test_days
    ASSETS.mkdir(parents=True, exist_ok=True)
    RESULTS.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------ 1. data
    log("[1/7] Loading price panel ...")
    panel = align_to_common_dates(add_features(build_panel(cfg.data)))
    all_dates = pd.DatetimeIndex(sorted(panel["Date"].unique()))
    log(
        f"      {panel['ticker'].nunique()} tickers, {len(all_dates)} common trading dates "
        f"({all_dates[0].date()} -> {all_dates[-1].date()})"
    )

    # --------------------------------------------------------------- 2. regimes
    log("[2/7] Building regime features ...")
    regime_features_raw = build_regime_features(panel).dropna()
    first_train = regime_features_raw.index[:train_days]
    regime_features = standardise_causally(regime_features_raw, train_index=first_train)

    selection = select_n_regimes(
        regime_features.loc[first_train], candidates=(2, 3, 4),
        feature_columns=REGIME_FEATURE_COLUMNS,
    )
    selection.to_csv(RESULTS / "regime_model_selection.csv", index=False)
    best_k = int(selection.iloc[0]["n_regimes"])
    log(f"      BIC selects {best_k} regimes on the first training block")
    log(selection.to_string(index=False))

    def make_detector():
        return GaussianHMMRegimes(
            n_regimes=best_k, feature_columns=REGIME_FEATURE_COLUMNS, random_state=cfg.seed
        )

    # Detectors fitted on the FIRST training block only, then run causally over all
    # history -- the view an operator would actually have had on day one.
    display_detectors = {
        "Threshold": ThresholdRegimes(n_regimes=best_k, column="realized_vol_21"),
        "Quadrant": QuadrantRegimes(),
        "HMM": make_detector(),
        "MarkovSwitch": MarkovSwitchingVariance(n_regimes=2, return_column="trend_21"),
        "JumpModel": JumpModelRegimes(
            n_regimes=best_k, feature_columns=REGIME_FEATURE_COLUMNS, jump_penalty=2.0
        ),
    }
    labels, regime_names_map = {}, {}
    for name, detector in display_detectors.items():
        detector.fit(regime_features.loc[first_train])
        labels[name] = detector.label_online(regime_features)
        regime_names_map[name] = detector.regime_labels_

    # ------------------------------------------------- 3. validate the detectors
    log("[3/7] Validating regime models ...")
    segmenter = BinarySegmentation(min_size=40, max_breaks=8)
    breaks = segmenter.breakpoints(regime_features_raw["realized_vol_21"])
    log(f"      retrospective breaks: {[str(d.date()) for d in breaks]}")

    lag_table = {name: lag_summary(detection_lag(s, breaks)) for name, s in labels.items()}
    lag_frame = pd.DataFrame(lag_table).T
    lag_frame.to_csv(RESULTS / "regime_detection_lag.csv")
    log(f"\n{lag_frame.to_string()}\n")

    kappa = agreement_matrix(labels)
    kappa.to_csv(RESULTS / "regime_agreement.csv")

    primary_labels = labels["HMM"]
    primary_names = regime_names_map["HMM"]
    persistence = persistence_summary(primary_labels, regime_names=primary_names)
    persistence.to_csv(RESULTS / "regime_persistence.csv", index=False)
    log(persistence.to_string(index=False))
    log(f"      switch rate {persistence.attrs['switch_rate']:.3f}  "
        f"mean run {persistence.attrs['overall_mean_run']:.1f}d")

    stability = refit_stability(
        make_detector, regime_features, initial_train=train_days, step=step_days
    )
    stability.to_csv(RESULTS / "regime_refit_stability.csv", index=False)
    mean_kappa = stability["kappa_vs_previous"].mean()
    log(f"      refit stability: mean kappa vs previous fit = {mean_kappa:.3f}")

    market = price_matrix(panel).mean(axis=1)

    # Commentary for readers. Written to its own artefact and never joined into the
    # feature panel -- see report/narrate.py for why that boundary is enforced.
    narration_provider = None
    if getattr(args, "narrate_llm", False):
        # Construct before the run so a missing key fails immediately rather than
        # degrading every episode to the fallback and looking like it worked.
        narration_provider = openrouter_provider(model=args.narrate_model)

    episodes = narrate(
        extract_episodes(
            primary_labels, market.pct_change().dropna(),
            regime_names=primary_names, min_days=10,
        ),
        provider=narration_provider,
    )
    episodes.to_csv(RESULTS / "regime_narration.csv", index=False)
    (RESULTS / "regime_narration.md").write_text(to_markdown(episodes))

    sources = episodes["source"].value_counts().to_dict() if len(episodes) else {}
    n_llm = int(sources.get("llm", 0))
    if narration_provider is None:
        log(f"      {len(episodes)} regime episodes narrated (source: data, no model)")
    else:
        # Report the split rather than just "done": a silent fallback to data on most
        # episodes means the key or the model id is wrong, and that should be visible.
        log(f"      {len(episodes)} regime episodes narrated "
            f"({n_llm} from the model, {int(sources.get('data', 0))} data-only fallback)")

    regime_econ = regime_conditional_stats(
        market.pct_change().dropna(), primary_labels, regime_names=primary_names
    )
    regime_econ.to_csv(RESULTS / "regime_economics.csv", index=False)
    log(f"\n{regime_econ.to_string(index=False)}\n")

    # ------------------------------------------------------- 4. allocator weights
    log("[4/7] Building allocator weight schedules ...")
    prices = price_matrix(panel)

    # The actual NIFTY 50 index. Until now the "benchmark" was buy-and-hold of the ten
    # selected names, which answers a different question: it cannot say whether picking
    # those ten beat simply buying the index.
    nifty_returns = panel.groupby("Date")["benchmark_return"].first().astype(float)
    nifty_returns.index = pd.to_datetime(nifty_returns.index)
    log(f"      NIFTY 50 index reference: {nifty_returns.notna().sum()} daily observations")

    allocator_weights = {}
    for name, allocator in ALLOCATORS.items():
        weights = build_allocator_weights(prices, allocator, lookback=252, frequency="ME")
        if not weights.empty:
            allocator_weights[name] = weights
    log(f"      {len(allocator_weights)} allocators: {list(allocator_weights)}")

    # ------------------------------------------------------- 5. walk-forward
    log(f"[5/7] Rolling walk-forward (expanding train, {test_days}-day test blocks) "
        + ("without PPO ..." if args.no_rl
           else f"with PPO: {args.seeds} seeds x {args.timesteps:,} steps per window ..."))
    passive_cfg = replace(cfg.backtest, stop_loss=None, take_profit=None)
    ladder = default_exposure_ladder(best_k)
    log(f"      regime exposure ladder {ladder} over {primary_names}")

    report = walk_forward_evaluate(
        panel=panel,
        regime_features=regime_features,
        regime_feature_columns=REGIME_FEATURE_COLUMNS,
        detector_factory=make_detector,
        rule_strategies=RULE_STRATEGIES,
        allocator_weights=allocator_weights,
        prices=prices,
        cfg=cfg.backtest,
        cost_cfg=cfg.costs,
        metrics_cfg=cfg.metrics,
        passive_cfg=passive_cfg,
        benchmark="BuyHold",
        exposure_ladder=ladder,
        overlay_bases=("HRP", "EqualWeight", "RiskParity"),
        train_days=train_days,
        test_days=test_days,
        step_days=step_days,
        expanding=True,
        reference_series={"NIFTY50_Index": nifty_returns},
        rl_config=None if args.no_rl else RLConfig(
            features=RL_FEATURES,
            seeds=tuple(range(args.seeds)),
            timesteps=args.timesteps,
            val_fraction=0.2,
            include_per_seed=True,
            check_freq=10_000,
        ),
        progress=log,
    )
    report.per_window.to_csv(RESULTS / "walk_forward_windows.csv", index=False)
    report.summary.to_csv(RESULTS / "walk_forward_summary.csv", index=False)

    log(f"\n      {report.n_windows} out-of-sample windows, "
        f"{len(next(iter(report.pooled_returns.values())))} pooled OOS trading days")
    log(report.summary[
        ["strategy", "pooled_total_return", "pooled_sharpe", "windows_positive",
         "windows_beating_benchmark", "worst_window_return"]
    ].round(3).to_string(index=False))

    selections = pd.Series([w.selected_strategy for w in report.windows])
    log(f"\n      train-chosen strategy per window: {list(selections)}")

    # --------------------------------------------------------------- 6. inference
    log("\n[6/7] Pooled out-of-sample significance ...")
    exposure = dict(zip(report.summary["strategy"], report.summary["mean_exposure"]))
    n_trials = len(report.pooled_returns) + 16 + 3 + 4
    significance = summarise_significance(
        report.pooled_returns, "BuyHold", n_trials=n_trials,
        risk_free_annual=cfg.metrics.risk_free_annual,
        exposure_by_strategy=exposure,
    )
    significance.to_csv(RESULTS / "significance.csv", index=False)

    returns_matrix = pd.DataFrame(report.pooled_returns).dropna()
    pbo = probability_of_backtest_overfitting(returns_matrix, n_splits=8)
    reality = whites_reality_check(
        returns_matrix.drop(columns=["BuyHold"]), returns_matrix["BuyHold"], n_boot=500
    )
    pd.DataFrame([{**pbo, **reality}]).to_csv(RESULTS / "overfitting_tests.csv", index=False)
    log(f"      PBO = {pbo['pbo']:.2f}   White's RC p = {reality['p_value']:.3f} "
        f"(best: {reality['best_strategy']})")

    regime_performance = performance_by_regime(
        report.pooled_returns, report.pooled_regimes,
        regime_names=primary_names, risk_free_annual=cfg.metrics.risk_free_annual,
    )
    regime_performance.to_csv(RESULTS / "performance_by_regime.csv", index=False)

    # ---------------------------------------------------------------- 7. figures
    log("[7/7] Rendering figures ...")
    ranked = report.summary["strategy"].tolist()
    highlight = [s for s in ["PPO_ensemble", "MaxSharpe", "BuyHold"] if s in report.pooled_equity]
    if len(highlight) < 3:
        highlight = (ranked[:2] + ["BuyHold"])[:3]

    for mode in ("light", "dark"):
        suffix = "" if mode == "light" else "_dark"

        figures.save(
            *figures.window_returns_heatmap(
                report.per_window, mode=mode,
                subtitle=f"{report.n_windows} windows, expanding-train refit, "
                         f"{test_days}-day out-of-sample blocks.",
            ),
            ASSETS / f"walk_forward_windows{suffix}.png",
        )
        figures.save(
            *figures.consistency_scatter(report.summary, highlight, mode=mode),
            ASSETS / f"consistency{suffix}.png",
        )
        if report.per_window["strategy"].str.startswith("PPO_s").any():
            figures.save(
                *figures.ppo_seed_dispersion(report.per_window, mode=mode),
                ASSETS / f"ppo_seed_dispersion{suffix}.png",
            )
        figures.save(
            *figures.equity_curves(
                report.pooled_equity, highlight, cfg.backtest.initial_cash, mode=mode,
                title="Pooled out-of-sample equity — every walk-forward block chained",
                subtitle="Each segment was traded with parameters frozen before it began.",
            ),
            ASSETS / f"equity_curves{suffix}.png",
        )
        figures.save(
            *figures.drawdown_curves(
                report.pooled_equity, highlight, mode=mode,
                title="Pooled out-of-sample drawdown",
            ),
            ASSETS / f"drawdown{suffix}.png",
        )
        figures.save(
            *figures.sharpe_forest(significance, mode=mode,
                title="Pooled out-of-sample Sharpe with bootstrap confidence intervals"),
            ASSETS / f"sharpe_forest{suffix}.png",
        )
        figures.save(
            *figures.regime_timeline(
                market.loc[regime_features.index], labels, regime_names_map,
                breakpoints=breaks, mode=mode,
                subtitle="Fitted on the first training block only, then run causally forward.",
            ),
            ASSETS / f"regime_timeline{suffix}.png",
        )
        figures.save(
            *figures.regime_validation_panel(
                persistence, lag_table, mode=mode,
                subtitle="Both gates must pass before the regime layer is used downstream.",
            ),
            ASSETS / f"regime_validation{suffix}.png",
        )
        figures.save(
            *figures.agreement_heatmap(kappa, mode=mode,
                subtitle="Chance-corrected agreement between independent detectors."),
            ASSETS / f"regime_agreement{suffix}.png",
        )
        figures.save(
            *figures.regime_transition_heatmap(
                display_detectors["HMM"].transition_frame(), mode=mode,
                subtitle="Expected durations: " + ", ".join(
                    f"{k} {v:.0f}d"
                    for k, v in display_detectors["HMM"].expected_durations().items()
                ),
            ),
            ASSETS / f"regime_transitions{suffix}.png",
        )
        figures.save(
            *figures.regime_performance_heatmap(
                regime_performance, "sharpe", mode=mode,
                title="Pooled out-of-sample performance by regime",
                subtitle="Does a risk-penalised strategy earn its penalty when conditions are bad?",
            ),
            ASSETS / f"performance_by_regime{suffix}.png",
        )

    log(f"      figures -> {ASSETS}")

    invested = significance[~significance["cash_like"]]
    best = invested.iloc[0] if len(invested) else significance.iloc[0]
    summary = {
        "evaluation": "rolling walk-forward (expanding train)",
        "n_windows": report.n_windows,
        "train_days": train_days,
        "test_days": test_days,
        "oos_start": str(report.windows[0].spec.test_start.date()),
        "oos_end": str(report.windows[-1].spec.test_end.date()),
        "n_oos_days": int(len(next(iter(report.pooled_returns.values())))),
        "n_strategies": len(report.pooled_returns),
        "n_trials": n_trials,
        "best_strategy": best["strategy"],
        "best_pooled_sharpe": float(best["sharpe"]),
        "best_dsr": float(best["dsr"]),
        "n_ci_excludes_zero": int(significance["ci_excludes_zero"].sum()),
        "pbo": pbo["pbo"],
        "reality_check_p": reality["p_value"],
        "regime_switch_rate": persistence.attrs["switch_rate"],
        "regime_mean_run_days": persistence.attrs["overall_mean_run"],
        "regime_refit_kappa": float(mean_kappa),
    }
    pd.Series(summary).to_csv(RESULTS / "run_summary.csv")

    results_md = write_results(RESULTS, ASSETS, summary, PROJECT_ROOT / "RESULTS.md")
    log(f"      report  -> {results_md}")

    log("\n=== SUMMARY ===")
    for key, value in summary.items():
        log(f"  {key:<24s} {value}")


## Run

This is the one that takes twenty minutes. Almost all of it is PPO — eight windows, three seeds each, trained from scratch every time.

In a hurry? Swap the last line for `main(parse_args(["--no-rl"]))`. It skips only the agent, finishes in about ninety seconds, and everything else is identical.

Output lands in `results/` (the tables), `assets/v2/` (the charts, each with its numbers alongside), and `RESULTS.md` (the write-up).

In [ ]:
# The full run. Go and make a coffee.
#   Quick version, skips the agent:  main(parse_args(["--no-rl"]))
main(parse_args([]))

## The charts

Each one is drawn above as it is produced. This cell gathers them in one place — and
works from the files on disk, so it still shows the charts in a fresh kernel without
rerunning the pipeline.


In [ ]:
# ── show the charts ───────────────────────────────────────────────────────────────────
# The run above draws each chart as it is produced, but they land at the bottom of a long
# log. This redisplays them on their own, and also works in a fresh kernel from the files
# already on disk -- so you can look at the charts without a twenty-minute run.
from IPython.display import Image, Markdown, display


def show_charts(directory=None, dark=False):
    """Display every chart written by the last run."""
    directory = Path(directory) if directory else _PROJECT_ROOT / "assets" / "v2"
    if not directory.exists():
        display(Markdown(f"_No charts at `{directory}` yet — run the cell above first._"))
        return

    charts = sorted(
        p for p in directory.glob("*.png")
        if p.stem.endswith("_dark") == dark
    )
    if not charts:
        display(Markdown(f"_No {'dark' if dark else 'light'} charts found in `{directory}`._"))
        return

    display(Markdown(f"### {len(charts)} charts from `{directory}`"))
    for path in charts:
        display(Markdown(f"**{path.stem.replace('_', ' ')}**"))
        display(Image(filename=str(path)))          # reads the PNG straight off disk
        csv = path.with_suffix(".csv")
        if csv.exists():
            display(Markdown(f"<sub>numbers: `{csv.name}`</sub>"))


show_charts()


---

# The charts

Produced by the run above and written to `assets/v2/`. They also appear inline as the final cell executes; these copies are here so the notebook reads as a finished document without being run.

Every chart has its numbers beside it as a CSV — `sharpe_forest.png` sits next to `sharpe_forest.csv`. A chart whose values are only reachable by measuring pixels is not a result. There is a `_dark` variant of each, for dark-background slides.


> Images below are linked by absolute URL rather than a relative path. GitHub's
> notebook viewer does not resolve relative image paths from inside a notebook, so
> `../assets/v2/…` renders as a broken image there even though the file is present.


### Out-of-sample return by window

![Out-of-sample return by window](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/walk_forward_windows.png)

Every strategy, every window. Consistency is what this shows and totals hide: a strategy carried by one lucky window looks fine on the leaderboard and obviously fragile here.

<sub>`assets/v2/walk_forward_windows.png` · numbers in `walk_forward_windows.csv`</sub>


### Growth of ₹10,00,000

![Growth of ₹10,00,000](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/equity_curves.png)

The pooled out-of-sample record — eight frozen test blocks chained end to end.

<sub>`assets/v2/equity_curves.png` · numbers in `equity_curves.csv`</sub>


### Underwater curves

![Underwater curves](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/drawdown.png)

How far below the previous peak each strategy sat, and for how long. The width of a trough matters as much as its depth.

<sub>`assets/v2/drawdown.png` · numbers in `drawdown.csv`</sub>


### Windows positive vs windows beating the benchmark

![Windows positive vs windows beating the benchmark](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/consistency.png)

Making money and beating buy-and-hold are different achievements. Most strategies manage the first far more often than the second.

<sub>`assets/v2/consistency.png` · numbers in `consistency.csv`</sub>


### Pooled Sharpe with 95% confidence intervals

![Pooled Sharpe with 95% confidence intervals](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/sharpe_forest.png)

**The most important chart here.** Almost every interval crosses zero — including MaxSharpe's, at [−0.29, 1.68]. Point estimates without these bars would overstate every row in the table.

<sub>`assets/v2/sharpe_forest.png` · numbers in `sharpe_forest.csv`</sub>


### Risk against return

![Risk against return](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/risk_return.png)

Annualised return versus volatility. The classical allocators cluster; PPO sits with the fully-invested baselines.

<sub>`assets/v2/risk_return.png` · numbers in `risk_return.csv`</sub>


### PPO across three seeds

![PPO across three seeds](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/ppo_seed_dispersion.png)

Three independent training runs per window. They agree closely — median within-window spread 0.51% — which is the evidence that PPO's result is what the reward actually asks for, not noise. The fix is a different question, not more training.

<sub>`assets/v2/ppo_seed_dispersion.png` · numbers in `ppo_seed_dispersion.csv`</sub>


### HRP allocation through time

![HRP allocation through time](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/allocation_hrp.png)

Realised weights, not requested ones — what the portfolio actually held.

<sub>`assets/v2/allocation_hrp.png` · numbers in `allocation_hrp.csv`</sub>


### Detected regimes over time

![Detected regimes over time](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/regime_timeline.png)

Labels assigned using only past data. Compare against what you remember of these years: the 2020 crash and the 2022 drawdown should be visible.

<sub>`assets/v2/regime_timeline.png` · numbers in `regime_timeline.csv`</sub>


### Are the regime models usable?

![Are the regime models usable?](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/regime_validation.png)

Persistence (mean run 20.6 days), detection lag, and refit stability (κ = 0.70). Run before anything is conditioned on regimes — a model that flips every three days is untradeable after costs however well it fits.

<sub>`assets/v2/regime_validation.png` · numbers in `regime_validation.csv`</sub>


### Do the five detectors agree?

![Do the five detectors agree?](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/regime_agreement.png)

Pairwise agreement. Independent methods converging on the same structure is evidence it is real rather than fitted.

<sub>`assets/v2/regime_agreement.png` · numbers in `regime_agreement.csv`</sub>


### Transition probabilities

![Transition probabilities](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/regime_transitions.png)

How likely each regime is to persist or switch. The diagonal sets expected duration.

<sub>`assets/v2/regime_transitions.png` · numbers in `regime_transitions.csv`</sub>


### Strategy performance conditioned on regime

![Strategy performance conditioned on regime](https://raw.githubusercontent.com/tridibjena/nifty50-rl-portfolio-optimization/main/assets/v2/performance_by_regime.png)

Where the regime layer earns its place — as diagnosis. As an exposure signal it did not pay for itself: HRP +48.1% pooled against HRP+Regime +37.8%.

<sub>`assets/v2/performance_by_regime.png` · numbers in `performance_by_regime.csv`</sub>
